In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
from google.colab import drive

# 1. Google Driveのマウント
drive.mount('/content/drive')

# 2. パスの設定
INPUT_ROOT = Path('/content/drive/MyDrive/石河研/spectrum/tsukuba')
OUTPUT_DIR = Path('/content/drive/MyDrive/石河研/spectrum')

# 【修正1】Google Drive上であることの確認
if not str(INPUT_ROOT).startswith('/content/drive/MyDrive/'):
    raise ValueError(f"【処理停止】INPUT_ROOTが /content/drive/MyDrive/ 配下にありません: {INPUT_ROOT}")
if not str(OUTPUT_DIR).startswith('/content/drive/MyDrive/'):
    raise ValueError(f"【処理停止】OUTPUT_DIRが /content/drive/MyDrive/ 配下にありません: {OUTPUT_DIR}")

# 【修正2】チェックポイント関連パスを最初から定義
CHECKPOINT_ROOT = OUTPUT_DIR / "ape_HSR_r12_350_1050_checkpoints"
RUN_MANIFEST_PATH = CHECKPOINT_ROOT / "run_manifest.json"
RESUME_STATE_PATH = CHECKPOINT_ROOT / "resume_state.json"
PROCESSING_LOCK_PATH = CHECKPOINT_ROOT / "processing_lock.json"
BATCH_DIR = CHECKPOINT_ROOT / "batches"
ERROR_DIR = CHECKPOINT_ROOT / "errors"
TEMP_DIR = CHECKPOINT_ROOT / "temporary"

FINAL_PICKLE_PATH = OUTPUT_DIR / "target_ape_tsukuba_HSR_r12_350_1050.pkl"
FINAL_JSON_PATH = OUTPUT_DIR / "target_ape_tsukuba_HSR_r12_350_1050_quality_report.json"

# 【修正3】既存成果物の確認
paths_to_check = {
    "FINAL_PICKLE_PATH": FINAL_PICKLE_PATH,
    "FINAL_JSON_PATH": FINAL_JSON_PATH,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "RUN_MANIFEST_PATH": RUN_MANIFEST_PATH,
    "RESUME_STATE_PATH": RESUME_STATE_PATH,
    "PROCESSING_LOCK_PATH": PROCESSING_LOCK_PATH,
    "BATCH_DIR": BATCH_DIR,
    "ERROR_DIR": ERROR_DIR,
    "TEMP_DIR": TEMP_DIR,
}

print("--- パスの存在確認 ---")
path_status = {}
for name, p in paths_to_check.items():
    exists = p.exists()
    path_status[name] = exists
    print(f"{name}: {'存在する' if exists else '存在しない'} ({p})")

if path_status["FINAL_PICKLE_PATH"] or path_status["FINAL_JSON_PATH"]:
    raise FileExistsError("【処理停止】最終成果物（pickleまたはJSON）が既に存在します。上書き防止のため停止します。")

# 【修正4】新規／再開候補の判定
if not path_status["CHECKPOINT_ROOT"]:
    mode = "新規"
    print("\n実行候補モード：新規")
else:
    mode = "再開"
    print("\n実行候補モード：再開\n既存チェックポイントの検証が必要です")

if path_status["PROCESSING_LOCK_PATH"]:
    print("処理ロックあり：次の検証セルでheartbeatを確認する必要があります")

# バッチファイルのカウント
batch_parquet_count = len(list(BATCH_DIR.glob("batch_*.parquet"))) if path_status["BATCH_DIR"] else 0
batch_json_count = len(list(BATCH_DIR.glob("batch_*_quality.json"))) if path_status["BATCH_DIR"] else 0

# 【修正5】表示内容
print("\n--- 確認結果サマリー ---")
print(f"INPUT_ROOT: {INPUT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CHECKPOINT_ROOT: {CHECKPOINT_ROOT}")
print(f"最終pickleの存在: {path_status['FINAL_PICKLE_PATH']}")
print(f"最終品質JSONの存在: {path_status['FINAL_JSON_PATH']}")
print(f"run_manifestの存在: {path_status['RUN_MANIFEST_PATH']}")
print(f"resume_stateの存在: {path_status['RESUME_STATE_PATH']}")
print(f"processing_lockの存在: {path_status['PROCESSING_LOCK_PATH']}")
print(f"発見した既存batch parquet数: {batch_parquet_count}")
print(f"発見した既存batch quality JSON数: {batch_json_count}")
print(f"実行候補モード: {mode}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- パスの存在確認 ---
FINAL_PICKLE_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050.pkl)
FINAL_JSON_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050_quality_report.json)
CHECKPOINT_ROOT: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints)
RUN_MANIFEST_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/run_manifest.json)
RESUME_STATE_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/resume_state.json)
PROCESSING_LOCK_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/processing_lock.json)
BATCH_DIR: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/batches)
ERROR_DIR: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/errors)
TE

In [3]:
import re
from datetime import datetime
from collections import defaultdict

print("--- HSRファイルの検索開始 ---")
print(f"検索ルート: {INPUT_ROOT}")

# 正規表現パターン
pattern_hsr = re.compile(r"^10HSR\d{6}_\d{3}\.csv$")
pattern_tsr = re.compile(r"^10TSR.*\.csv$")
pattern_met = re.compile(r"^10MET.*\.csv$")

all_csv_count = 0
hsr_files = []
excluded_tsr_count = 0
excluded_met_count = 0
other_excluded_count = 0

# 【修正1・2】ファイル検索 (is_fileとfullmatch)
for path in INPUT_ROOT.rglob("*.csv"):
    if not path.is_file():
        continue

    all_csv_count += 1
    filename = path.name

    if pattern_hsr.fullmatch(filename):
        hsr_files.append(path)
    elif pattern_tsr.match(filename):
        excluded_tsr_count += 1
    elif pattern_met.match(filename):
        excluded_met_count += 1
    else:
        other_excluded_count += 1

# 【修正3】HSRファイルのソート (相対パスで決定論的に)
hsr_files = sorted(hsr_files, key=lambda p: p.relative_to(INPUT_ROOT).as_posix())
hsr_count = len(hsr_files)

# 【修正4】相対パス一覧の保持
hsr_relative_paths = [
    p.relative_to(INPUT_ROOT).as_posix()
    for p in hsr_files
]

# 【修正5】同一ファイル名の重複確認
name_to_paths = defaultdict(list)
for rel_p in hsr_relative_paths:
    filename = rel_p.split('/')[-1]
    name_to_paths[filename].append(rel_p)

duplicate_names = {name: paths for name, paths in name_to_paths.items() if len(paths) > 1}
if duplicate_names:
    print("\n【処理停止】同一ファイル名が複数の場所に存在します。重複行の原因となるため停止します。")
    for name, paths in duplicate_names.items():
        print(f"重複ファイル名: {name}")
        for p in paths:
            print(f"  - 相対パス: {p}")
    print(f"重複ファイル名数: {len(duplicate_names)}")
    raise ValueError("重複ファイル名が検出されました。")

# 【修正6】見込み日付の抽出とエラーハンドリング
dates = []
invalid_date_files = []

for p, rel_p in zip(hsr_files, hsr_relative_paths):
    # 10HSRyymmdd_ppp.csv -> yymmdd部分を抽出
    date_str = p.name[5:11]
    try:
        # 2000年代と仮定
        date_obj = datetime.strptime("20" + date_str, "%Y%m%d").date()
        dates.append(date_obj)
    except ValueError:
        invalid_date_files.append((p.name, rel_p, date_str))

if invalid_date_files:
    print("\n【処理停止】日付変換に失敗したファイルがあります。")
    for name, rel_p, d_str in invalid_date_files:
        print(f"ファイル名: {name}")
        print(f"相対パス: {rel_p}")
        print(f"抽出した日付文字列: {d_str}")
    raise ValueError("日付変換に失敗したファイルが存在します。")

# 【修正7】件数の一致確認
if len(dates) != hsr_count:
    raise RuntimeError(f"HSRファイル数({hsr_count})と日付抽出成功数({len(dates)})が一致しません。")

min_date = min(dates) if dates else None
max_date = max(dates) if dates else None

print("\n--- 検索結果 ---")
print(f"発見した全CSV数: {all_csv_count}")
print(f"HSRとして採用するファイル数: {hsr_count}")
print(f"除外されたTSR数: {excluded_tsr_count}")
print(f"除外されたMET数: {excluded_met_count}")
print(f"その他除外数 (出力ファイルや条件不一致): {other_excluded_count}")

if hsr_count == 0:
    raise ValueError("【処理停止】対象となるHSRファイルが0件です。")

# 【修正8】先頭・末尾5件の相対パス表示
print("\n--- 採用ファイル先頭5件 (相対パス) ---")
for rp in hsr_relative_paths[:5]:
    print(rp)

print("\n--- 採用ファイル末尾5件 (相対パス) ---")
for rp in hsr_relative_paths[-5:]:
    print(rp)

print("\n--- 見込み日付範囲 (ファイル名より) ---")
print(f"最小日付: {min_date}")
print(f"最大日付: {max_date}")

# 【修正9】追加の確認事項表示
print("\n--- 品質・整合性チェック結果 ---")
print(f"重複ファイル名数: {len(duplicate_names)}")
print(f"日付変換失敗数: {len(invalid_date_files)}")
print(f"hsr_relative_pathsの件数: {len(hsr_relative_paths)}")
print(f"HSRファイル数と日付抽出成功数の一致: {len(dates) == hsr_count}")


--- HSRファイルの検索開始 ---
検索ルート: /content/drive/MyDrive/石河研/spectrum/tsukuba

--- 検索結果 ---
発見した全CSV数: 3680
HSRとして採用するファイル数: 1824
除外されたTSR数: 1796
除外されたMET数: 60
その他除外数 (出力ファイルや条件不一致): 0

--- 採用ファイル先頭5件 (相対パス) ---
201101/10HSR110101_301.csv
201101/10HSR110102_301.csv
201101/10HSR110103_301.csv
201101/10HSR110104_301.csv
201101/10HSR110105_301.csv

--- 採用ファイル末尾5件 (相対パス) ---
201512/10HSR151227_301.csv
201512/10HSR151228_301.csv
201512/10HSR151229_301.csv
201512/10HSR151230_301.csv
201512/10HSR151231_301.csv

--- 見込み日付範囲 (ファイル名より) ---
最小日付: 2011-01-01
最大日付: 2015-12-31

--- 品質・整合性チェック結果 ---
重複ファイル名数: 0
日付変換失敗数: 0
hsr_relative_pathsの件数: 1824
HSRファイル数と日付抽出成功数の一致: True


In [4]:
import json
import hashlib
import math
from datetime import date, timedelta, datetime, timezone
import collections

print("--- マニフェスト事前検証 (読み取り専用) ---")

# 【1. HSRの日付完全性確認】
start_date = date(2011, 1, 1)
end_date = date(2015, 12, 31)
total_calendar_days = (end_date - start_date).days + 1

date_to_files = collections.defaultdict(list)
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    date_str = p.name[5:11]
    d = datetime.strptime("20" + date_str, "%Y%m%d").date()
    date_to_files[d].append(rel_p)

present_days = len(date_to_files)
missing_dates = []
for i in range(total_calendar_days):
    d = start_date + timedelta(days=i)
    if d not in date_to_files:
        missing_dates.append(d.isoformat())

multiple_files_dates = {k.isoformat(): v for k, v in date_to_files.items() if len(v) > 1}

# 【2. 地点番号の確認】
site_counts = collections.defaultdict(int)
invalid_sites = []
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    site = p.name[-7:-4] # 例: 10HSR151231_301.csv -> 301
    site_counts[site] += 1
    if site != "301":
        invalid_sites.append(rel_p)

if invalid_sites:
    print("\n【処理停止】地点番号301以外のファイルが検出されました。")
    for inv_p in invalid_sites:
        print(f"  - {inv_p}")
    raise ValueError("不正な地点番号が存在します。")

# 【3. CONFIGとconfig_hash】
CONFIG = {
    "pipeline_version": "1.0.0",
    "dataset_name": "experiment_B_HSR",
    "plane": "HSR",
    "w_min": 350,
    "w_max": 1050,
    "remarks_used": [1, 2],
    "duplicate_policy": "error",
    "ape_valid_range_eV": [1.2, 2.2],
    "source_timezone": "Asia/Tokyo",
    "datetime_storage": "timezone-naive local time",
    "source_file_pattern": "^10HSR\\d{6}_\\d{3}\\.csv$",
    "formula": "1239.84193 * sum(G) / sum(G * wavelength)",
    "batch_size": 25,
    "input_root": str(INPUT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "expected_site_numbers": [301]
}

config_json_str = json.dumps(
    CONFIG,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":")
)
config_hash = hashlib.sha256(config_json_str.encode('utf-8')).hexdigest()

# 【4. ファイルメタデータ】
file_metadata = {}
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    st = p.stat()
    file_metadata[rel_p] = {
        "relative_path": rel_p,
        "filename": p.name,
        "size": st.st_size,
        "mtime_ns": st.st_mtime_ns
    }
if len(file_metadata) != hsr_count:
    raise RuntimeError("ファイルメタデータ件数がHSRファイル数と一致しません。")

# 【5. 決定論的バッチ構成】
total_files = len(hsr_relative_paths)
batch_size = CONFIG["batch_size"]
total_batches = math.ceil(total_files / batch_size)

batches_config = {}
assigned_files = set()
duplicates_in_batches = 0

for i in range(total_batches):
    b_id = f"batch_{i:05d}"
    start_idx = i * batch_size
    end_idx = min(start_idx + batch_size, total_files)
    batch_files = hsr_relative_paths[start_idx:end_idx]
    batches_config[b_id] = batch_files

    for f in batch_files:
        if f in assigned_files:
            duplicates_in_batches += 1
        assigned_files.add(f)

unassigned_count = total_files - len(assigned_files)
if len(assigned_files) != total_files or duplicates_in_batches > 0 or unassigned_count > 0:
    raise RuntimeError("バッチ構成に重複または未割り当てが存在します。")

first_batch_id = list(batches_config.keys())[0]
last_batch_id = list(batches_config.keys())[-1]
first_batch_size = len(batches_config[first_batch_id])
last_batch_size = len(batches_config[last_batch_id])

# 【6. manifest_preview】
manifest_preview = {
    **CONFIG,
    "config_hash": config_hash,
    "total_hsr_files": total_files,
    "total_batches": total_batches,
    "sorted_hsr_files": hsr_relative_paths,
    "file_metadata": file_metadata,
    "batches": batches_config,
    "missing_dates": missing_dates,
    "station_file_counts": dict(site_counts),
    "manifest_created_at": datetime.now(timezone.utc).isoformat()
}

# 【7. 実行直前状態の再確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

if mode == "新規" and CHECKPOINT_ROOT.exists():
    raise FileExistsError("【処理停止】modeは新規ですが、CHECKPOINT_ROOTが既に存在します。別セッションの干渉の可能性があります。")

if mode == "再開":
    if not RUN_MANIFEST_PATH.exists():
        raise FileNotFoundError("【処理停止】再開モードですが run_manifest.json が見つかりません。")

    with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
        existing_manifest = json.load(f)

    if existing_manifest.get("config_hash") != config_hash:
        raise ValueError("【処理停止】抽出条件や設定内容 (config_hash) が前回と異なります。")
    if existing_manifest.get("total_hsr_files") != total_files:
        raise ValueError("【処理停止】ファイル数が一致しません。")
    if existing_manifest.get("total_batches") != total_batches:
        raise ValueError("【処理停止】バッチ数が一致しません。")
    if existing_manifest.get("sorted_hsr_files") != hsr_relative_paths:
        raise ValueError("【処理停止】HSRファイルの順序が一致しません。")
    if existing_manifest.get("batches") != batches_config:
        raise ValueError("【処理停止】バッチ構成が一致しません。")

    old_metadata = existing_manifest.get("file_metadata", {})
    for rel_p, meta in file_metadata.items():
        if rel_p not in old_metadata:
            raise ValueError(f"【処理停止】新しいファイルが見つかりました: {rel_p}")
        old_meta = old_metadata[rel_p]
        if old_meta["size"] != meta["size"] or old_meta["mtime_ns"] != meta["mtime_ns"]:
            raise ValueError(f"【処理停止】ファイルサイズまたは更新日時(mtime_ns)が変化しています: {rel_p}")

# 【8. 最終表示】
print("\n--- 事前検証結果 ---")
print(f"実行候補モード: {mode}")
print(f"config_hash: {config_hash}")
print(f"地点番号別ファイル数: {dict(site_counts)}")
print(f"総暦日数: {total_calendar_days} 日")
print(f"HSR存在日数: {present_days} 日")
print(f"HSR欠測日数: {len(missing_dates)} 日")
if missing_dates:
    print(f"欠測日一覧: {missing_dates}")
if multiple_files_dates:
    print(f"複数ファイルが存在する日: {multiple_files_dates}")
print(f"総ファイル数: {total_files}")
print(f"総バッチ数: {total_batches}")
print(f"最初のバッチ ({first_batch_id}) サイズ: {first_batch_size}")
print(f"最後のバッチ ({last_batch_id}) サイズ: {last_batch_size}")
print(f"ファイルメタデータ件数: {len(file_metadata)}")
print(f"重複割り当て数: {duplicates_in_batches}")
print(f"未割り当て数: {unassigned_count}")
print("\n✅ マニフェスト事前検証：成功")
print("※このセルではGoogle Driveへの書き込み（ファイル・ディレクトリ作成等）は一切行っていません。")


--- マニフェスト事前検証 (読み取り専用) ---

--- 事前検証結果 ---
実行候補モード: 新規
config_hash: f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9
地点番号別ファイル数: {'301': 1824}
総暦日数: 1826 日
HSR存在日数: 1824 日
HSR欠測日数: 2 日
欠測日一覧: ['2011-04-29', '2011-04-30']
総ファイル数: 1824
総バッチ数: 73
最初のバッチ (batch_00000) サイズ: 25
最後のバッチ (batch_00072) サイズ: 24
ファイルメタデータ件数: 1824
重複割り当て数: 0
未割り当て数: 0

✅ マニフェスト事前検証：成功
※このセルではGoogle Driveへの書き込み（ファイル・ディレクトリ作成等）は一切行っていません。


In [6]:
import json
import uuid
import os
from datetime import datetime, timezone

print("--- 新規モード限定：安全な初期化とロック取得 ---")

# 【1. 新規モード限定】
if mode != "新規":
    print("再開モードでは専用のロック検証・取得セルを使用してください。")
    raise RuntimeError("このセルは新規モード専用です。")

# 【2. 実行直前の再確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists() or CHECKPOINT_ROOT.exists() or \
   RUN_MANIFEST_PATH.exists() or RESUME_STATE_PATH.exists() or PROCESSING_LOCK_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物またはチェックポイントが既に存在します。mode判定後に状態が変化した可能性があります。")

# 【4. 孤立したステージングの検出】
staging_pattern = ".ape_HSR_r12_350_1050_initializing_*"
existing_stagings = list(OUTPUT_DIR.glob(staging_pattern))
if existing_stagings:
    print("\n【処理停止】過去の初期化中断と思われるステージングディレクトリが存在します。")
    for s_dir in existing_stagings:
        print(f"ディレクトリ名: {s_dir.name}")
        st = s_dir.stat()
        mtime = datetime.fromtimestamp(st.st_mtime).isoformat()
        print(f"更新日時: {mtime}")
        files = [f.name for f in s_dir.iterdir()]
        print(f"内部ファイル一覧: {files}")
    raise FileExistsError("孤立したステージングディレクトリが検出されました。安全のため自動削除・上書きは行わず停止します。")

# セッションと日時の生成
session_id = str(uuid.uuid4())
current_time = datetime.now(timezone.utc)

# 【3. ステージングディレクトリ】
STAGING_ROOT = OUTPUT_DIR / f".ape_HSR_r12_350_1050_initializing_{session_id}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)

STAGING_BATCH_DIR = STAGING_ROOT / "batches"
STAGING_ERROR_DIR = STAGING_ROOT / "errors"
STAGING_TEMP_DIR = STAGING_ROOT / "temporary"

STAGING_BATCH_DIR.mkdir(exist_ok=False)
STAGING_ERROR_DIR.mkdir(exist_ok=False)
STAGING_TEMP_DIR.mkdir(exist_ok=False)

staging_manifest_path = STAGING_ROOT / "run_manifest.json"
staging_resume_path = STAGING_ROOT / "resume_state.json"
staging_lock_path = STAGING_ROOT / "processing_lock.json"

# 【5. JSONの安全な保存関数】
def atomic_write_json(data_dict, target_path, session_id):
    temp_name = f"{target_path.stem}_{session_id}.tmp.json"
    temp_path = target_path.parent / temp_name

    # 1. 保存先と同じディレクトリに一時JSONを作る
    with open(temp_path, "x", encoding="utf-8") as f:
        # 2, 3, 4. json.dump, flush, fsync
        json.dump(data_dict, f, indent=4, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())

    # 5. 一時JSONを読み直す
    with open(temp_path, "r", encoding="utf-8") as f:
        read_back = json.load(f)

    # 6. 元のPython辞書と一致確認
    if read_back != data_dict:
        raise ValueError(f"【処理停止】一時JSONの内容が元の辞書と一致しません: {temp_path}")

    # 7. JSONとして再度読み込めることを確認
    try:
        with open(temp_path, "r", encoding="utf-8") as f:
            json.load(f)
    except Exception as e:
        raise ValueError(f"【処理停止】一時JSONのパースに失敗しました: {e}")

    # 8. os.replaceで正式なJSON名へ変更
    os.replace(temp_path, target_path)

    # 9. 正式JSONを再度読み直して一致確認
    with open(target_path, "r", encoding="utf-8") as f:
        final_read = json.load(f)
    if final_read != data_dict:
        raise ValueError(f"【処理停止】正式JSONの内容が元の辞書と一致しません: {target_path}")

# 【6. ステージングへ保存する内容】
initial_resume_state = {
    "pipeline_version": CONFIG["pipeline_version"],
    "config_hash": config_hash,
    "status": "initialized",
    "completed_batch_ids": [],
    "completed_source_files": [],
    "pending_batch_ids": list(batches_config.keys()),
    "failed_batch_ids": [],
    "total_batches": total_batches,
    "completed_batches": 0,
    "completed_files": 0,
    "total_files": total_files,
    "last_completed_batch": None,
    "last_completed_source_file": None,
    "last_update": current_time.isoformat(),
    "current_session_id": session_id
}

lock_data = {
    "session_id": session_id,
    "config_hash": config_hash,
    "start_time": current_time.isoformat(),
    "heartbeat": current_time.isoformat(),
    "status": "running",
    "lock_schema_version": "1.0"
}

# 保存実行
atomic_write_json(manifest_preview, staging_manifest_path, session_id)
atomic_write_json(initial_resume_state, staging_resume_path, session_id)
atomic_write_json(lock_data, staging_lock_path, session_id)

# 【7. ステージング全体の検証】
if not staging_manifest_path.exists() or not staging_resume_path.exists() or not staging_lock_path.exists():
    raise RuntimeError(f"【処理停止】ステージング内に必要なJSONファイルが揃っていません。\nSTAGING_ROOT: {STAGING_ROOT}")

with open(staging_manifest_path, "r", encoding="utf-8") as f:
    v_manifest = json.load(f)
with open(staging_resume_path, "r", encoding="utf-8") as f:
    v_resume = json.load(f)
with open(staging_lock_path, "r", encoding="utf-8") as f:
    v_lock = json.load(f)

if v_manifest["config_hash"] != config_hash:
    raise ValueError("manifestのconfig_hashが現在値と一致しません。")
if v_resume["config_hash"] != config_hash:
    raise ValueError("resume_stateのconfig_hashが現在値と一致しません。")
if v_lock["config_hash"] != config_hash:
    raise ValueError("lockのconfig_hashが現在値と一致しません。")
if v_resume["current_session_id"] != session_id or v_lock["session_id"] != session_id:
    raise ValueError("session_idが一致しません。")
if v_manifest["total_hsr_files"] != 1824:
    raise ValueError("manifestの総ファイル数が1,824ではありません。")
if v_manifest["total_batches"] != 73:
    raise ValueError("manifestの総バッチ数が73ではありません。")
if len(v_resume["pending_batch_ids"]) != 73:
    raise ValueError("pending_batch_idsが73件ではありません。")
if len(v_resume["completed_batch_ids"]) != 0:
    raise ValueError("completed_batch_idsが空ではありません。")
if len(v_resume["completed_source_files"]) != 0:
    raise ValueError("completed_source_filesが空ではありません。")

if not STAGING_BATCH_DIR.exists() or not STAGING_ERROR_DIR.exists() or not STAGING_TEMP_DIR.exists():
    raise RuntimeError("ステージング内の必須ディレクトリが存在しません。")
if any(STAGING_BATCH_DIR.iterdir()):
    raise RuntimeError("batchesディレクトリが空ではありません。")
if any(STAGING_ERROR_DIR.iterdir()):
    raise RuntimeError("errorsディレクトリが空ではありません。")

# temporaryディレクトリ及びSTAGING_ROOT直下に一時ファイルが残っていないか
if any(STAGING_ROOT.glob("*.tmp.json")):
    raise RuntimeError("保存先ディレクトリにatomic_write_jsonの一時ファイルが残っています。")
if any(STAGING_TEMP_DIR.iterdir()):
    raise RuntimeError("temporaryディレクトリが空ではありません。")

# 【8. 正式名称への切り替え直前確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists() or CHECKPOINT_ROOT.exists():
    raise FileExistsError("【処理停止】正式名称への切り替え直前に競合ファイルまたはディレクトリが検出されました。")

# 【9. 正式CHECKPOINT_ROOTへの切り替え】
# ディレクトリのrenameによるアトミックな作成
try:
    STAGING_ROOT.rename(CHECKPOINT_ROOT)
except Exception as e:
    raise RuntimeError(f"【処理停止】正式CHECKPOINT_ROOTへのrenameに失敗しました: {e}")

# 【10. 正式化後の再検証】
if not CHECKPOINT_ROOT.exists():
    raise RuntimeError("CHECKPOINT_ROOTが存在しません。renameに失敗した可能性があります。")
if STAGING_ROOT.exists():
    raise RuntimeError("元のSTAGING_ROOTが存在しています。renameが正常に完了していません。")

with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    f_manifest = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    f_resume = json.load(f)
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    f_lock = json.load(f)

if f_manifest != v_manifest or f_resume != v_resume or f_lock != v_lock:
    raise RuntimeError("正式化後にJSONの内容が変化しています。")

if f_lock["session_id"] != session_id:
    raise ValueError("正式化後のsession_idが一致しません。")
if f_lock["config_hash"] != config_hash:
    raise ValueError("正式化後のconfig_hashが一致しません。")
if f_lock["status"] != "running":
    raise ValueError("正式化後のstatusが正しくありません。")

# 【12. 最終表示】
print("\n--- 初期化完了 ---")
print(f"session_id: {session_id}")
print(f"config_hash: {config_hash}")
print(f"正式CHECKPOINT_ROOT: {CHECKPOINT_ROOT}")
print("manifest検証結果: 成功")
print("resume_state検証結果: 成功")
print("processing_lock検証結果: 成功")
print(f"総ファイル数: {f_manifest['total_hsr_files']}")
print(f"総バッチ数: {f_manifest['total_batches']}")
print(f"pending batch数: {len(f_resume['pending_batch_ids'])}")
print(f"completed batch数: {len(f_resume['completed_batch_ids'])}")
print(f"初期化ステータス: {f_resume['status']}")
print("\n✅ 初期化とロック取得が完全に成功しました。次にCSV処理へ進めます。")


--- 新規モード限定：安全な初期化とロック取得 ---


FileExistsError: 【処理停止】最終成果物またはチェックポイントが既に存在します。mode判定後に状態が変化した可能性があります。

In [7]:
import json
import hashlib
import os
from datetime import datetime, timezone

print("--- 既存チェックポイント読み取り専用監査 ---")

# 【1. 最終成果物の確認】
print("\n【1. 最終成果物の確認】")
for name, path in [("FINAL_PICKLE_PATH", FINAL_PICKLE_PATH), ("FINAL_JSON_PATH", FINAL_JSON_PATH)]:
    if path.exists():
        st = path.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
        print(f"{name}: 存在する - {path} (Size: {st.st_size}, Mtime: {mtime})")
    else:
        print(f"{name}: 存在しない")

# 【2. CHECKPOINT_ROOTの全内容】
print("\n【2. CHECKPOINT_ROOTの全内容】")
expected_items = {
    "run_manifest.json", "resume_state.json", "processing_lock.json",
    "batches", "errors", "temporary"
}
unexpected_items = []
if CHECKPOINT_ROOT.exists():
    for p in CHECKPOINT_ROOT.rglob("*"):
        rel_p = p.relative_to(CHECKPOINT_ROOT).as_posix()
        st = p.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
        ftype = "Dir " if p.is_dir() else "File"
        print(f"- {rel_p} ({ftype}, Size: {st.st_size}, Mtime: {mtime})")

        # 期待するファイル構造からの逸脱を記録
        if p.parent == CHECKPOINT_ROOT and p.name not in expected_items:
            unexpected_items.append(rel_p)
        elif p.parent != CHECKPOINT_ROOT:
            # 今回の段階ではサブディレクトリ配下には一切ファイルが存在しないことが期待される
            unexpected_items.append(rel_p)
else:
    print("CHECKPOINT_ROOTが存在しません。")

# 【3. ディレクトリ内部の確認】
print("\n【3. ディレクトリ内部の確認】")
batch_parquet_count = len(list(BATCH_DIR.glob("batch_*.parquet"))) if BATCH_DIR.exists() else 0
batch_json_count = len(list(BATCH_DIR.glob("batch_*_quality.json"))) if BATCH_DIR.exists() else 0
batches_files = len(list(BATCH_DIR.iterdir())) if BATCH_DIR.exists() else 0
errors_files = len(list(ERROR_DIR.iterdir())) if ERROR_DIR.exists() else 0
temp_files = len(list(TEMP_DIR.iterdir())) if TEMP_DIR.exists() else 0
tmp_files = len(list(CHECKPOINT_ROOT.rglob("*.tmp"))) + len(list(CHECKPOINT_ROOT.rglob("*.tmp.json")))
staging_dirs = len(list(OUTPUT_DIR.glob(".ape_HSR_r12_350_1050_initializing_*")))

print(f"batches内の総ファイル数: {batches_files}")
print(f"batch parquet数: {batch_parquet_count}")
print(f"batch quality JSON数: {batch_json_count}")
print(f"errors内の総ファイル数: {errors_files}")
print(f"temporary内の総ファイル数: {temp_files}")
print(f"一時ファイル (*.tmp, *.tmp.json) 数: {tmp_files}")
print(f"initialization用ステージングディレクトリ数: {staging_dirs}")

# 【4. 3つのJSONの読み取り】
print("\n【4. 3つのJSONの読み取り】")
json_data = {}
json_status = {}

for name, path in [("run_manifest", RUN_MANIFEST_PATH), ("resume_state", RESUME_STATE_PATH), ("processing_lock", PROCESSING_LOCK_PATH)]:
    status = {"exists": path.exists(), "valid_json": False, "is_dict": False, "keys": [], "size": 0, "sha256": "", "error": None}
    if status["exists"]:
        try:
            st = path.stat()
            status["size"] = st.st_size
            with open(path, "rb") as f:
                raw_bytes = f.read()
                status["sha256"] = hashlib.sha256(raw_bytes).hexdigest()

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                status["valid_json"] = True
                if isinstance(data, dict):
                    status["is_dict"] = True
                    status["keys"] = list(data.keys())
                    json_data[name] = data
        except Exception as e:
            status["error"] = f"{type(e).__name__}: {e}"
    json_status[name] = status
    print(f"- {name}.json:")
    print(f"  存在: {status['exists']}, 正常読込: {status['valid_json']}, dict型: {status['is_dict']}")
    print(f"  サイズ: {status['size']} bytes, SHA-256: {status['sha256']}")
    if status["is_dict"]:
        print(f"  キー: {status['keys']}")
    if status["error"]:
        print(f"  エラー: {status['error']}")

# 【5. run_manifestの検証】
print("\n【5. run_manifestの検証】")
manifest_valid = False
manifest_diff_keys = []
if "run_manifest" in json_data:
    m = json_data["run_manifest"]
    v_results = {
        "config_hash_match_current": m.get("config_hash") == config_hash,
        "config_hash_match_preview": m.get("config_hash") == manifest_preview.get("config_hash"),
        "total_hsr_files": m.get("total_hsr_files") == 1824,
        "total_batches": m.get("total_batches") == 73,
        "sorted_hsr_files": m.get("sorted_hsr_files") == hsr_relative_paths,
        "file_metadata": m.get("file_metadata") == manifest_preview.get("file_metadata"),
        "batches": m.get("batches") == batches_config,
        "missing_dates": m.get("missing_dates") == ["2011-04-29", "2011-04-30"],
        "expected_site_numbers": m.get("expected_site_numbers") == [301],
        "station_file_counts": m.get("station_file_counts") == {"301": 1824},
        "pipeline_version": m.get("pipeline_version") == "1.0.0"
    }
    for k, v in v_results.items():
        print(f"  {k}: {v}")

    manifest_valid = all(v_results.values())

    for k in set(m.keys()).union(set(manifest_preview.keys())):
        if m.get(k) != manifest_preview.get(k):
            manifest_diff_keys.append(k)
    print(f"  manifest_previewとの相違キー: {manifest_diff_keys}")
else:
    print("  検証スキップ (データなし)")

# 【6. resume_stateの検証】
print("\n【6. resume_stateの検証】")
resume_valid = False
if "resume_state" in json_data:
    r = json_data["resume_state"]
    v_results = {
        "config_hash_match": r.get("config_hash") == config_hash,
        "status": r.get("status") == "initialized",
        "total_batches": r.get("total_batches") == 73,
        "total_files": r.get("total_files") == 1824,
        "completed_batches": r.get("completed_batches") == 0,
        "completed_files": r.get("completed_files") == 0,
        "completed_batch_ids_empty": r.get("completed_batch_ids") == [],
        "completed_source_files_empty": r.get("completed_source_files") == [],
        "failed_batch_ids_empty": r.get("failed_batch_ids") == [],
        "pending_batch_ids_count": len(r.get("pending_batch_ids", [])) == 73,
        "pending_batch_ids_match": r.get("pending_batch_ids") == list(batches_config.keys()),
        "last_completed_batch": r.get("last_completed_batch") is None,
        "last_completed_source_file": r.get("last_completed_source_file") is None,
        "current_session_id_not_empty": bool(r.get("current_session_id"))
    }
    for k, v in v_results.items():
        print(f"  {k}: {v}")
    resume_valid = all(v_results.values())
else:
    print("  検証スキップ (データなし)")

# 【7. processing_lockの検証】
print("\n【7. processing_lockの検証】")
lock_missing_keys = []
heartbeat_elapsed_mins = -1.0
lock_session_match = False

if "processing_lock" in json_data:
    l = json_data["processing_lock"]
    print(f"  session_id: {l.get('session_id')}")
    print(f"  start_time: {l.get('start_time')}")
    print(f"  heartbeat: {l.get('heartbeat')}")
    print(f"  status: {l.get('status')}")

    has_config_hash = "config_hash" in l
    has_schema_version = "lock_schema_version" in l
    print(f"  config_hashが存在するか: {has_config_hash}")
    print(f"  lock_schema_versionが存在するか: {has_schema_version}")

    if not has_config_hash: lock_missing_keys.append("config_hash")
    if not has_schema_version: lock_missing_keys.append("lock_schema_version")
    if lock_missing_keys:
        print("  ※移行が必要: 一部キーが不足しています。")

    rs_session_id = json_data.get("resume_state", {}).get("current_session_id")
    lock_session_match = (rs_session_id == l.get("session_id")) and bool(rs_session_id)
    print(f"  resume_state.current_session_idとlock.session_idが一致するか: {lock_session_match}")

    try:
        st = datetime.fromisoformat(l.get("start_time", ""))
        hb = datetime.fromisoformat(l.get("heartbeat", ""))
        tz_aware = (st.tzinfo is not None) and (hb.tzinfo is not None)
        print(f"  start_timeとheartbeatがtimezone-awareか: {tz_aware}")

        now_utc = datetime.now(timezone.utc)
        if hb.tzinfo is None:
            hb = hb.replace(tzinfo=timezone.utc)
        heartbeat_elapsed_mins = (now_utc - hb).total_seconds() / 60.0
        print(f"  heartbeatから現在UTC時刻までの経過分数: {heartbeat_elapsed_mins:.2f} 分")
    except Exception as e:
        print(f"  日時のパースエラー: {e}")
else:
    print("  検証スキップ (データなし)")

# 【8. 監査判定】
print("\n【8. 監査判定】")
decision = "RESET_REQUIRES_EXPLICIT_CONFIRMATION"
reason = []

if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    reason.append("最終成果物が存在します")
if not manifest_valid:
    reason.append("manifestが不完全または不一致")
if not resume_valid:
    reason.append("resume_stateが不完全または不一致")
if not lock_session_match:
    reason.append("session_id不一致または存在しない")
if batches_files > 0 or errors_files > 0 or temp_files > 0:
    reason.append("バッチ/エラー/一時ディレクトリ内にファイルが存在します")
if tmp_files > 0:
    reason.append("一時ファイル(*.tmp)が存在します")
if unexpected_items:
    reason.append("予期せぬファイル/ディレクトリが存在します")

if not reason:
    decision = "ADOPTABLE_WITH_MIGRATION"
else:
    decision = "RESET_REQUIRES_EXPLICIT_CONFIRMATION"

print(f"判定結果: {decision}")
if reason:
    print("理由: " + ", ".join(reason))

# 【9. 最終表示】
print("\n--- 監査結果サマリー ---")
print(f"最終pickleの存在: {FINAL_PICKLE_PATH.exists()}")
print(f"最終品質JSONの存在: {FINAL_JSON_PATH.exists()}")
print("JSON 3件の読み込み結果:")
for k, v in json_status.items():
    print(f"  {k}: 読込={'成功' if v['valid_json'] else '失敗'}")
print(f"manifest検証結果: {'成功' if manifest_valid else '失敗'}")
print(f"resume_state検証結果: {'成功' if resume_valid else '失敗'}")
print(f"lock検証結果: セッション一致={lock_session_match}")
print(f"lockに不足するキー: {lock_missing_keys}")
print(f"heartbeat経過分数: {heartbeat_elapsed_mins:.2f} 分")
print(f"batch parquet数: {batch_parquet_count}")
print(f"batch quality JSON数: {batch_json_count}")
print(f"temporaryファイル数: {temp_files}")
print(f"unexpected_items: {unexpected_items}")
print(f"\n総合監査判定: {decision}")
print("※このセルではGoogle Driveへの書き込みを一切行っていません。")

--- 既存チェックポイント読み取り専用監査 ---

【1. 最終成果物の確認】
FINAL_PICKLE_PATH: 存在しない
FINAL_JSON_PATH: 存在しない

【2. CHECKPOINT_ROOTの全内容】
- batches (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- errors (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- temporary (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- run_manifest.json (File, Size: 567394, Mtime: 2026-07-30T15:09:27+00:00)
- resume_state.json (File, Size: 2255, Mtime: 2026-07-30T15:09:27+00:00)
- processing_lock.json (File, Size: 192, Mtime: 2026-07-30T15:09:27+00:00)

【3. ディレクトリ内部の確認】
batches内の総ファイル数: 0
batch parquet数: 0
batch quality JSON数: 0
errors内の総ファイル数: 0
temporary内の総ファイル数: 0
一時ファイル (*.tmp, *.tmp.json) 数: 0
initialization用ステージングディレクトリ数: 0

【4. 3つのJSONの読み取り】
- run_manifest.json:
  存在: True, 正常読込: True, dict型: True
  サイズ: 567394 bytes, SHA-256: 9d4c7fb99a481789e530c6318cbeb91082d51bd92af9d42e27c095aa05fe7fda
  キー: ['pipeline_version', 'dataset_name', 'plane', 'w_min', 'w_max', 'remarks_used', 'duplicate_policy', 'ape_valid_ra

In [10]:
import json
import hashlib
import os
from datetime import datetime, timezone

print("--- processing_lock.json の安全なスキーマ移行 ---")

# 【1. 現在のランタイムがロック所有者であることの確認】
print("\n【1. ランタイムのロック所有権確認】")
runtime_session_id = globals().get("session_id")
if not runtime_session_id:
    raise RuntimeError("【処理停止】メモリ上に session_id が存在しません。現在のランタイムがロック所有者であることを証明できません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    old_lock = json.load(f)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

if old_lock.get("status") != "running":
    raise ValueError(f"【処理停止】既存ロックの status が 'running' ではありません: {old_lock.get('status')}")
if not old_lock.get("session_id"):
    raise ValueError("【処理停止】既存ロックに session_id が存在しません。")
if old_lock.get("session_id") != current_resume.get("current_session_id"):
    raise ValueError("【処理停止】既存ロックの session_id が resume_state と一致しません。")

try:
    st_dt = datetime.fromisoformat(old_lock.get("start_time", ""))
    hb_dt = datetime.fromisoformat(old_lock.get("heartbeat", ""))
    if st_dt.tzinfo is None or hb_dt.tzinfo is None:
        raise ValueError("timezone-awareではありません")
except Exception as e:
    raise ValueError(f"【処理停止】既存ロックの日時形式が不正です: {e}")

if runtime_session_id != old_lock["session_id"]:
    raise RuntimeError(f"【処理停止】メモリ上の session_id ({runtime_session_id}) が既存ロックの session_id ({old_lock['session_id']}) と一致しません。\n自動引継ぎは行いません。")

print("✅ メモリ上のsession_idと既存ロックが一致しました。正当な所有者として移行を続行します。")

# 【2. 変更前のハッシュ】
def get_file_sha256(filepath):
    with open(filepath, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

manifest_sha_before = get_file_sha256(RUN_MANIFEST_PATH)
resume_sha_before = get_file_sha256(RESUME_STATE_PATH)
lock_sha_before = get_file_sha256(PROCESSING_LOCK_PATH)

# 【3. 新しいlock内容】
current_time = datetime.now(timezone.utc)
new_lock = dict(old_lock) # 未知のキーを保持

new_lock["config_hash"] = config_hash
new_lock["heartbeat"] = current_time.isoformat()
new_lock["status"] = "running"
new_lock["lock_schema_version"] = "1.0"
new_lock["migration"] = {
    "migrated_at": current_time.isoformat(),
    "previous_lock_sha256": lock_sha_before,
    "reason": "add config_hash and lock_schema_version to legacy initialization lock",
    "previous_keys": list(old_lock.keys())
}

# 【4. 既存JSONを更新する原子的関数】
print("\n【4. 原子的更新の実行】")
temp_lock_path = CHECKPOINT_ROOT / f".processing_lock_{runtime_session_id}.migration.tmp.json"

if temp_lock_path.exists():
    raise FileExistsError(f"【処理停止】一時ファイルが既に存在します: {temp_lock_path}")

with open(temp_lock_path, "x", encoding="utf-8") as f:
    json.dump(new_lock, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())

with open(temp_lock_path, "r", encoding="utf-8") as f:
    read_temp_lock = json.load(f)

if read_temp_lock != new_lock:
    raise RuntimeError("【処理停止】一時ファイルの内容がnew_lockと一致しません。")

# 直前ハッシュ確認
if get_file_sha256(PROCESSING_LOCK_PATH) != lock_sha_before:
    raise RuntimeError("【処理停止】処理中に既存ロックが変化しました。競合の可能性があります。")
if get_file_sha256(RUN_MANIFEST_PATH) != manifest_sha_before:
    raise RuntimeError("【処理停止】処理中にmanifestが変化しました。")
if get_file_sha256(RESUME_STATE_PATH) != resume_sha_before:
    raise RuntimeError("【処理停止】処理中にresume_stateが変化しました。")

os.replace(temp_lock_path, PROCESSING_LOCK_PATH)

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    final_read_lock = json.load(f)

if final_read_lock != new_lock:
    raise RuntimeError("【処理停止】置換後のロック内容がnew_lockと一致しません。")
if temp_lock_path.exists():
    raise RuntimeError("【処理停止】一時ファイルが削除されずに残っています。")

# 【5. 移行後の検証】
print("\n【5. 移行後の検証】")
lock_sha_after = get_file_sha256(PROCESSING_LOCK_PATH)
manifest_sha_after = get_file_sha256(RUN_MANIFEST_PATH)
resume_sha_after = get_file_sha256(RESUME_STATE_PATH)

validation_results = {
    "session_id_match": final_read_lock["session_id"] == old_lock["session_id"],
    "session_id_resume_match": final_read_lock["session_id"] == current_resume["current_session_id"],
    "config_hash_match": final_read_lock["config_hash"] == config_hash,
    "schema_version_match": final_read_lock["lock_schema_version"] == "1.0",
    "start_time_match": final_read_lock["start_time"] == old_lock["start_time"],
    "heartbeat_tz_aware": datetime.fromisoformat(final_read_lock["heartbeat"]).tzinfo is not None,
    "status_running": final_read_lock["status"] == "running",
    "migration_sha_match": final_read_lock["migration"]["previous_lock_sha256"] == lock_sha_before,
    "manifest_sha_unchanged": manifest_sha_after == manifest_sha_before,
    "resume_sha_unchanged": resume_sha_after == resume_sha_before,
    "lock_sha_updated": lock_sha_after != lock_sha_before,
    "batches_empty": len(list(BATCH_DIR.iterdir())) == 0,
    "errors_empty": len(list(ERROR_DIR.iterdir())) == 0,
    "temporary_empty": len(list(TEMP_DIR.iterdir())) == 0,
    "finals_not_exist": not FINAL_PICKLE_PATH.exists() and not FINAL_JSON_PATH.exists()
}

all_valid = all(validation_results.values())
for k, v in validation_results.items():
    if not v:
        print(f"  [失敗] {k}")

if not all_valid:
    raise RuntimeError("【処理停止】移行後の検証に失敗しました。")

# 【7. 最終表示】
added_keys = set(final_read_lock.keys()) - set(old_lock.keys())

print("\n--- 移行完了サマリー ---")
print(f"runtime_session_id: {runtime_session_id}")
print(f"lock.session_id: {final_read_lock['session_id']}")
print(f"config_hash: {final_read_lock['config_hash']}")
print(f"manifest SHA-256: {manifest_sha_before} -> {manifest_sha_after} (変更なし)")
print(f"resume_state SHA-256: {resume_sha_before} -> {resume_sha_after} (変更なし)")
print(f"processing_lock SHA-256: {lock_sha_before} -> {lock_sha_after} (更新)")
print(f"追加されたキー: {list(added_keys)}")
print(f"start_time: {final_read_lock['start_time']}")
print(f"更新後heartbeat: {final_read_lock['heartbeat']}")
print(f"lock_schema_version: {final_read_lock['lock_schema_version']}")
print(f"migration情報: {final_read_lock['migration']}")
print(f"バッチファイル数: batches={len(list(BATCH_DIR.iterdir()))}, errors={len(list(ERROR_DIR.iterdir()))}, temporary={len(list(TEMP_DIR.iterdir()))}")
print(f"\n移行判定: {'成功' if all_valid else '失敗'}")
if all_valid:
    print("\n✅ processing_lock.json のスキーマ移行が安全に完了しました。次に進むことができます。")

--- processing_lock.json の安全なスキーマ移行 ---

【1. ランタイムのロック所有権確認】


RuntimeError: 【処理停止】メモリ上の session_id (06be6212-53f4-4872-b7ac-ec56ad7d9b3b) が既存ロックの session_id (15a525f6-b3ce-4736-a8c2-3ea9db6cae49) と一致しません。
自動引継ぎは行いません。

In [11]:
import json
import uuid
from datetime import datetime, timezone

print("--- チェックポイント初期化とロック取得 ---")

# 過去のロックが残っている場合に強制解除するためのフラグ
# （ハートビートから30分以上経過している場合のみ有効です）
FORCE_CLEAR_LOCK = False

session_id = str(uuid.uuid4())
current_time = datetime.now(timezone.utc)

# 【1. ロックの確認】
if PROCESSING_LOCK_PATH.exists():
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        try:
            existing_lock = json.load(f)
            last_heartbeat = datetime.fromisoformat(existing_lock["heartbeat"])
            time_diff = current_time - last_heartbeat

            if time_diff.total_seconds() < 30 * 60:
                raise RuntimeError(f"【処理停止】他のセッションが実行中の可能性があります。\n最後のheartbeat: {last_heartbeat}\n経過時間: {time_diff.total_seconds() / 60:.1f}分")
            else:
                if not FORCE_CLEAR_LOCK:
                    print(f"【確認】過去のランタイム切断等によりロックが残っている可能性があります。\n前回のheartbeat: {last_heartbeat}\n経過時間: {time_diff.total_seconds() / 60:.1f}分")
                    print("自動的に乗っ取ることはしません。安全を確認の上、再開する場合はこのセルの FORCE_CLEAR_LOCK = True に変更して再実行してください。")
                    raise RuntimeError("古い処理ロックが残っています。")
                else:
                    print("FORCE_CLEAR_LOCKがTrueに設定されているため、古いロックを解除して処理を引き継ぎます。")
        except json.JSONDecodeError:
            print("ロックファイルが破損しているため上書きします。")

# 【2. ディレクトリの作成】
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(exist_ok=True)
ERROR_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)

# 【3. マニフェストと進捗状態の保存（新規のみ）】
if mode == "新規":
    with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_preview, f, indent=4, ensure_ascii=False)

    initial_resume_state = {
        "pipeline_version": CONFIG["pipeline_version"],
        "config_hash": config_hash,
        "status": "initialized",
        "completed_batch_ids": [],
        "completed_source_files": [],
        "pending_batch_ids": list(batches_config.keys()),
        "failed_batch_ids": [],
        "total_batches": total_batches,
        "completed_batches": 0,
        "completed_files": 0,
        "total_files": total_files,
        "last_completed_batch": None,
        "last_completed_source_file": None,
        "last_update": current_time.isoformat(),
        "current_session_id": session_id
    }
    with open(RESUME_STATE_PATH, "w", encoding="utf-8") as f:
        json.dump(initial_resume_state, f, indent=4, ensure_ascii=False)

    print(f"新規チェックポイント環境を構築しました。")
    print(f"  - {RUN_MANIFEST_PATH.name}")
    print(f"  - {RESUME_STATE_PATH.name}")
else:
    print("既存のチェックポイント環境を引き継ぎます。")

# 【4. ロックの取得（上書き）】
lock_data = {
    "session_id": session_id,
    "start_time": current_time.isoformat(),
    "heartbeat": current_time.isoformat(),
    "status": "running"
}
with open(PROCESSING_LOCK_PATH, "w", encoding="utf-8") as f:
    json.dump(lock_data, f, indent=4, ensure_ascii=False)

print("\n✅ 処理ロックを取得しました。")
print(f"  session_id: {session_id}")


--- チェックポイント初期化とロック取得 ---


RuntimeError: 【処理停止】他のセッションが実行中の可能性があります。
最後のheartbeat: 2026-07-30 15:09:27.030972+00:00
経過時間: 29.6分

In [12]:
import json
import hashlib
from datetime import datetime, timezone

print("--- 再開・ロック引継ぎ事前検証 (読み取り専用) ---")

# 【1. FINAL_PICKLE_PATHとFINAL_JSON_PATHが存在しないこと】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

# 【2. 必須パスが存在すること】
required_paths = {
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "RUN_MANIFEST_PATH": RUN_MANIFEST_PATH,
    "RESUME_STATE_PATH": RESUME_STATE_PATH,
    "PROCESSING_LOCK_PATH": PROCESSING_LOCK_PATH
}
for name, p in required_paths.items():
    if not p.exists():
        raise FileNotFoundError(f"【処理停止】必須パスが存在しません: {name} ({p})")
print("✅ 必須パスの存在を確認しました。")

# 【3. run_manifest.jsonとresume_state.jsonを再読込して検証】
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    resume_data = json.load(f)

# 【4. config_hashが現在のCONFIGと一致すること】
if manifest_data.get("config_hash") != config_hash:
    raise ValueError("【処理停止】manifestのconfig_hashが現在のCONFIGと一致しません。")
if resume_data.get("config_hash") != config_hash:
    raise ValueError("【処理停止】resume_stateのconfig_hashが現在のCONFIGと一致しません。")
print("✅ config_hashの一致を確認しました。")

# 【5. manifest上の73バッチと現在の入力ファイル一覧が一致すること】
manifest_batches = manifest_data.get("batches", {})
if len(manifest_batches) != 73:
    raise ValueError(f"【処理停止】manifestのバッチ数が73ではありません: {len(manifest_batches)}")
if list(manifest_batches.keys()) != list(batches_config.keys()):
    raise ValueError("【処理停止】manifestのバッチ構成と現在の入力ファイル構成が一致しません。")
print("✅ マニフェスト上のバッチ構成(73バッチ)と入力ファイルの一致を確認しました。")

# 【6. 完了バッチ、batch parquet、batch quality JSONの数を表示】
completed_batches = resume_data.get("completed_batch_ids", [])
batch_parquet_count = len(list(BATCH_DIR.glob("batch_*.parquet"))) if BATCH_DIR.exists() else 0
batch_json_count = len(list(BATCH_DIR.glob("batch_*_quality.json"))) if BATCH_DIR.exists() else 0
print("\n--- 進捗状況 ---")
print(f"完了バッチ数 (resume_state): {len(completed_batches)}")
print(f"発見した既存batch parquet数: {batch_parquet_count}")
print(f"発見した既存batch quality JSON数: {batch_json_count}")

# 【7. processing_lock.jsonのsession_id、heartbeat_at、作成日時を表示】
# 【11. 既存ロックJSON全体のSHA-256を計算】
with open(PROCESSING_LOCK_PATH, "rb") as f:
    lock_bytes = f.read()
lock_sha256 = hashlib.sha256(lock_bytes).hexdigest()
lock_data = json.loads(lock_bytes.decode("utf-8"))

existing_session_id = lock_data.get("session_id")
start_time_str = lock_data.get("start_time")
heartbeat_str = lock_data.get("heartbeat")

print("\n--- ロック情報 ---")
print(f"session_id: {existing_session_id}")
print(f"start_time: {start_time_str}")
print(f"heartbeat: {heartbeat_str}")
print(f"ロックファイルSHA-256: {lock_sha256}")

# 【8. 現在UTC時刻からheartbeat経過分数を再計算】
current_utc = datetime.now(timezone.utc)
try:
    heartbeat_dt = datetime.fromisoformat(heartbeat_str)
    if heartbeat_dt.tzinfo is None:
        heartbeat_dt = heartbeat_dt.replace(tzinfo=timezone.utc)
    elapsed_mins = (current_utc - heartbeat_dt).total_seconds() / 60.0
except Exception as e:
    raise ValueError(f"【処理停止】heartbeat日時のパースに失敗しました: {e}")

print(f"heartbeat経過分数: {elapsed_mins:.2f} 分")

# 【9 & 10. heartbeatの判定】
if elapsed_mins < 30.0:
    raise RuntimeError(f"【処理停止】ロック有効 (経過時間が30分未満です: {elapsed_mins:.2f}分)。安全のため処理を停止します。")
else:
    print("判定: 明示的なロック引継ぎ候補 (30分以上経過)")

# 【12. 確認トークンの表示】
lock_sha_prefix = lock_sha256[:12]
config_hash_prefix = config_hash[:12]
token = f"TAKEOVER:{existing_session_id}:{lock_sha_prefix}:{config_hash_prefix}"

print("\n--- 後続セル用 確認トークン ---")
print(token)
print("\n※このセルではGoogle Driveへの書き込み操作、ロックの変更、session_idの発行等を一切行っていません。")

--- 再開・ロック引継ぎ事前検証 (読み取り専用) ---
✅ 必須パスの存在を確認しました。
✅ config_hashの一致を確認しました。
✅ マニフェスト上のバッチ構成(73バッチ)と入力ファイルの一致を確認しました。

--- 進捗状況 ---
完了バッチ数 (resume_state): 0
発見した既存batch parquet数: 0
発見した既存batch quality JSON数: 0

--- ロック情報 ---
session_id: 15a525f6-b3ce-4736-a8c2-3ea9db6cae49
start_time: 2026-07-30T15:09:27.030972+00:00
heartbeat: 2026-07-30T15:09:27.030972+00:00
ロックファイルSHA-256: 1bc1ff84597aba31b10074960f5620c80b917ab62ae7b241710e80d6c823b6f5
heartbeat経過分数: 64.19 分
判定: 明示的なロック引継ぎ候補 (30分以上経過)

--- 後続セル用 確認トークン ---
TAKEOVER:15a525f6-b3ce-4736-a8c2-3ea9db6cae49:1bc1ff84597a:f483833250e9

※このセルではGoogle Driveへの書き込み操作、ロックの変更、session_idの発行等を一切行っていません。


In [13]:
import json
import hashlib
import os
import uuid
from datetime import datetime, timezone

print("--- 古いロックの明示的引継ぎ・スキーマ移行 ---")

# 【1. 実行承認】
TAKEOVER_CONFIRMATION = "TAKEOVER:15a525f6-b3ce-4736-a8c2-3ea9db6cae49:1bc1ff84597a:f483833250e9"
EXPECTED_OLD_SESSION_ID = "15a525f6-b3ce-4736-a8c2-3ea9db6cae49"
EXPECTED_LOCK_SHA256 = "1bc1ff84597aba31b10074960f5620c80b917ab62ae7b241710e80d6c823b6f5"
EXPECTED_CONFIG_HASH = "f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9"

reconstructed_token = f"TAKEOVER:{EXPECTED_OLD_SESSION_ID}:{EXPECTED_LOCK_SHA256[:12]}:{EXPECTED_CONFIG_HASH[:12]}"
if TAKEOVER_CONFIRMATION != reconstructed_token:
    raise ValueError("【処理停止】確認トークンが再構成された文字列と一致しません。")
print("✅ 実行承認トークンの一致を確認しました。")

# 【2. 書き込み直前の再検証】
print("\n--- 書き込み直前の再検証 ---")

if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    resume_data = json.load(f)
with open(PROCESSING_LOCK_PATH, "rb") as f:
    lock_bytes = f.read()
lock_data = json.loads(lock_bytes.decode("utf-8"))

current_config_hash = config_hash # 前セル等で計算済みの値
if current_config_hash != EXPECTED_CONFIG_HASH:
    raise ValueError("【処理停止】現在のconfig_hashがEXPECTED_CONFIG_HASHと一致しません。")

if manifest_data != manifest_preview:
    raise ValueError("【処理停止】run_manifest.jsonがmanifest_previewと完全一致しません。")

manifest_batches = manifest_data.get("batches", {})
all_files_in_batches = []
for b_files in manifest_batches.values():
    all_files_in_batches.extend(b_files)
if len(all_files_in_batches) != 1824:
    raise ValueError("【処理停止】manifest内のファイル総数が1824ではありません。")
if len(set(all_files_in_batches)) != 1824:
    raise ValueError("【処理停止】manifest内のファイルに重複があります。")

if resume_data.get("current_session_id") != EXPECTED_OLD_SESSION_ID:
    raise ValueError("【処理停止】resume_stateのcurrent_session_idがEXPECTED_OLD_SESSION_IDと一致しません。")

if lock_data.get("session_id") != EXPECTED_OLD_SESSION_ID:
    raise ValueError("【処理停止】processing_lockのsession_idがEXPECTED_OLD_SESSION_IDと一致しません。")

actual_lock_sha = hashlib.sha256(lock_bytes).hexdigest()
if actual_lock_sha != EXPECTED_LOCK_SHA256:
    raise ValueError("【処理停止】processing_lock.jsonのSHA-256がEXPECTED_LOCK_SHA256と完全一致しません。")

now_utc = datetime.now(timezone.utc)
old_heartbeat_str = lock_data.get("heartbeat")
old_heartbeat = datetime.fromisoformat(old_heartbeat_str)
if old_heartbeat.tzinfo is None:
    old_heartbeat = old_heartbeat.replace(tzinfo=timezone.utc)
elapsed_mins = (now_utc - old_heartbeat).total_seconds() / 60.0
if elapsed_mins < 30.0:
    raise ValueError(f"【処理停止】heartbeatから30分経過していません ({elapsed_mins:.2f}分)。")

if len(resume_data.get("completed_batch_ids", [])) != 0 or \
   len(resume_data.get("completed_source_files", [])) != 0 or \
   len(resume_data.get("failed_batch_ids", [])) != 0:
    raise ValueError("【処理停止】完了・失敗済みバッチ/ファイルが存在します。進捗0/73を期待しています。")

pending_ids = resume_data.get("pending_batch_ids", [])
if len(pending_ids) != 73 or set(pending_ids) != set(manifest_batches.keys()):
    raise ValueError("【処理停止】pending_batch_idsが73件でないか、manifestと一致しません。")

if resume_data.get("completed_batches", -1) != 0 or resume_data.get("completed_files", -1) != 0:
    raise ValueError("【処理停止】completed_batchesまたはcompleted_filesが0ではありません。")

if any(BATCH_DIR.iterdir()) or any(ERROR_DIR.iterdir()) or any(TEMP_DIR.iterdir()):
    raise ValueError("【処理停止】batches, errors, temporaryディレクトリが空ではありません。")

tmp_files = list(CHECKPOINT_ROOT.rglob("*.tmp")) + list(CHECKPOINT_ROOT.rglob("*.tmp.json"))
if tmp_files:
    raise ValueError("【処理停止】一時ファイルが存在します。")

print("✅ 再検証をすべて通過しました。")

# 【3. 新セッション】
new_session_id = str(uuid.uuid4())
print(f"\n新セッションIDを生成しました: {new_session_id}")

# 【4. 更新内容】
new_lock = dict(lock_data)
new_lock.update({
    "session_id": new_session_id,
    "config_hash": EXPECTED_CONFIG_HASH,
    "start_time": now_utc.isoformat(),
    "heartbeat": now_utc.isoformat(),
    "status": "running",
    "lock_schema_version": "1.0",
    "taken_over_from_session_id": EXPECTED_OLD_SESSION_ID,
    "previous_lock_sha256": EXPECTED_LOCK_SHA256,
    "previous_heartbeat": old_heartbeat_str,
    "takeover_at": now_utc.isoformat(),
    "takeover_reason": "stale heartbeat after Colab runtime interruption"
})

new_resume = dict(resume_data)
new_resume.update({
    "current_session_id": new_session_id,
    "status": "running",
    "last_update": now_utc.isoformat(),
    "previous_session_id": EXPECTED_OLD_SESSION_ID,
    "takeover_at": now_utc.isoformat(),
    "previous_lock_sha256": EXPECTED_LOCK_SHA256
})

# 【5. 安全な保存】
def atomic_write_json(data_dict, target_path, sess_id):
    temp_path = target_path.parent / f"{target_path.stem}_{sess_id}.tmp.json"
    if temp_path.exists():
        raise FileExistsError(f"一時ファイルが既に存在します: {temp_path}")

    try:
        with open(temp_path, "x", encoding="utf-8") as f:
            json.dump(data_dict, f, indent=4, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())

        with open(temp_path, "r", encoding="utf-8") as f:
            read_back = json.load(f)
        if read_back != data_dict:
            raise ValueError(f"一時ファイルの内容が不一致です: {temp_path}")

        os.replace(temp_path, target_path)
    finally:
        if temp_path.exists():
            os.unlink(temp_path)

print("\n--- 更新処理実行 ---")
atomic_write_json(new_lock, PROCESSING_LOCK_PATH, new_session_id)
atomic_write_json(new_resume, RESUME_STATE_PATH, new_session_id)

# 正式ファイルの再読込・検証
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    final_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    final_resume = json.load(f)

final_checks = {
    "session_id_match": final_lock.get("session_id") == new_session_id and final_resume.get("current_session_id") == new_session_id,
    "config_hash_match": final_lock.get("config_hash") == EXPECTED_CONFIG_HASH,
    "schema_version": final_lock.get("lock_schema_version") == "1.0",
    "status_running": final_lock.get("status") == "running" and final_resume.get("status") == "running",
    "legacy_info_retained": final_lock.get("taken_over_from_session_id") == EXPECTED_OLD_SESSION_ID and final_resume.get("takeover_at") is not None,
    "progress_retained": len(final_resume.get("completed_batch_ids")) == 0 and len(final_resume.get("pending_batch_ids")) == 73,
    "no_tmp_files": len(list(CHECKPOINT_ROOT.rglob("*.tmp.json"))) == 0
}

all_passed = all(final_checks.values())
if not all_passed:
    print(f"【警告】一部の最終検証に失敗しました: {final_checks}")
    raise RuntimeError("最終検証に失敗しました。")

# 【6. 最終表示】
print("\n--- 最終結果サマリー ---")
print(f"旧session_id: {EXPECTED_OLD_SESSION_ID}")
print(f"新session_id: {new_session_id}")
print(f"旧ロックSHA-256: {EXPECTED_LOCK_SHA256}")
print(f"config_hash: {EXPECTED_CONFIG_HASH}")
print(f"引継ぎ前heartbeat経過分数: {elapsed_mins:.2f} 分")
print(f"lockスキーマ移行結果: 成功")
print(f"resume_state更新結果: 成功")
print(f"完了バッチ数: 0")
print(f"pendingバッチ数: 73")
print(f"batch parquet数: 0")
print(f"batch quality JSON数: 0")
print("\n最終判定:")
print("ロック引継ぎ・スキーマ移行成功。CSV処理開始前の状態です")


--- 古いロックの明示的引継ぎ・スキーマ移行 ---
✅ 実行承認トークンの一致を確認しました。

--- 書き込み直前の再検証 ---
✅ 再検証をすべて通過しました。

新セッションIDを生成しました: 152d121e-2e5f-445f-88e6-e644e5829f2d

--- 更新処理実行 ---

--- 最終結果サマリー ---
旧session_id: 15a525f6-b3ce-4736-a8c2-3ea9db6cae49
新session_id: 152d121e-2e5f-445f-88e6-e644e5829f2d
旧ロックSHA-256: 1bc1ff84597aba31b10074960f5620c80b917ab62ae7b241710e80d6c823b6f5
config_hash: f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9
引継ぎ前heartbeat経過分数: 72.09 分
lockスキーマ移行結果: 成功
resume_state更新結果: 成功
完了バッチ数: 0
pendingバッチ数: 73
batch parquet数: 0
batch quality JSON数: 0

最終判定:
ロック引継ぎ・スキーマ移行成功。CSV処理開始前の状態です


In [14]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import collections

print("--- 現在セッションの所有権確認・heartbeat更新・HSR CSV少数標本監査 ---")

# 【1. セッション所有権の確認】
print("\n【1. セッション所有権の確認】")
EXPECTED_NEW_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"

if 'new_session_id' not in globals():
    raise RuntimeError("【処理停止】メモリ上に new_session_id 変数が存在しません。ランタイムが断絶した可能性があります。所有者を偽装せず停止します。")

if new_session_id != EXPECTED_NEW_SESSION_ID:
    raise RuntimeError(f"【処理停止】メモリ上の new_session_id ({new_session_id}) が期待される値 ({EXPECTED_NEW_SESSION_ID}) と一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

lock_session = current_lock.get("session_id")
resume_session = current_resume.get("current_session_id")

if not (lock_session == resume_session == new_session_id):
    raise RuntimeError(f"【処理停止】セッションIDが不一致です。\nlock: {lock_session}\nresume: {resume_session}\nmemory: {new_session_id}")

session_id = new_session_id
print(f"✅ セッション所有権の完全一致を確認しました。このランタイムの処理用session_idを {session_id} に同期しました。")

# 【2. heartbeat更新】
print("\n【2. heartbeat更新】")
now_utc = datetime.now(timezone.utc).isoformat()
updated_lock = dict(current_lock)
updated_lock["heartbeat"] = now_utc
updated_lock["status"] = "inspecting_source_schema"

def atomic_update_lock(data_dict, target_path, sess_id):
    temp_path = target_path.parent / f"{target_path.stem}_{sess_id}_hb.tmp.json"
    if temp_path.exists():
        raise FileExistsError(f"一時ファイルが既に存在します: {temp_path}")
    try:
        with open(temp_path, "x", encoding="utf-8") as f:
            json.dump(data_dict, f, indent=4, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        with open(temp_path, "r", encoding="utf-8") as f:
            read_back = json.load(f)
        if read_back != data_dict:
            raise ValueError("一時ファイルの内容が不一致です。")
        os.replace(temp_path, target_path)
    finally:
        if temp_path.exists():
            os.unlink(temp_path)

atomic_update_lock(updated_lock, PROCESSING_LOCK_PATH, session_id)

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    verified_lock = json.load(f)

if verified_lock.get("session_id") != lock_session or verified_lock.get("config_hash") != current_lock.get("config_hash"):
    raise RuntimeError("【処理停止】heartbeat更新時にsession_idまたはconfig_hashが予期せず変更されました。")
print(f"✅ heartbeatを更新し、statusを inspecting_source_schema に変更しました。")

# 【3. 調査対象ファイルの決定】
print("\n【3. 調査対象ファイルの決定】")
# hsr_relative_paths は事前検証セル等で既にメモリ上にある想定 (sorted)
sampled_files = []

def find_file(year, target_month=None, position="first"):
    candidates = [f for f in hsr_relative_paths if f.startswith(f"{year}")]
    if target_month:
        candidates = [f for f in candidates if f[4:6] == f"{target_month:02d}"]
    if not candidates:
        return None
    if position == "first": return candidates[0]
    if position == "last": return candidates[-1]
    if position == "middle": return candidates[len(candidates)//2]

samples_req = {
    "2011年初回": find_file(2011, position="first"),
    "2012年中央": find_file(2012, position="middle"),
    "2013年中央": find_file(2013, position="middle"),
    "2014年中央": find_file(2014, position="middle"),
    "2015年最後": find_file(2015, position="last"),
    "春(4月)代表": find_file(2013, 4, "middle"),
    "夏(8月)代表": find_file(2013, 8, "middle"),
    "秋(10月)代表": find_file(2013, 10, "middle"),
    "冬(1月)代表": find_file(2014, 1, "middle")
}

for reason, path_str in samples_req.items():
    if path_str and path_str not in [s["path"] for s in sampled_files]:
        sampled_files.append({"reason": reason, "path": path_str})

print("以下のファイルを決定論的に抽出しました:")
for s in sampled_files:
    print(f" - {s['reason']}: {s['path']}")

# 【4. CSV構造の読み取り専用監査】
print("\n【4. CSV構造の読み取り専用監査】")
import chardet

audit_results = []
for s in sampled_files:
    full_path = INPUT_ROOT / s['path']
    st = full_path.stat()

    # 最初の数キロバイトを読んでエンコーディング推測
    with open(full_path, "rb") as f:
        raw_head = f.read(10000)
    enc_guess = chardet.detect(raw_head)
    encoding = enc_guess['encoding'] or 'utf-8'

    # 生ファイルの先頭30行
    with open(full_path, "r", encoding=encoding, errors="replace") as f:
        head_lines = [f.readline().rstrip('\n') for _ in range(30)]

    # 構造推測 (NEDOファイルは通常、数行のメタデータの後にヘッダー行がある)
    # カンマ区切りの数が多い行をヘッダーと見なす
    delimiter = ','
    skiprows = 0
    max_cols = 0
    for i, line in enumerate(head_lines):
        cols = len(line.split(delimiter))
        if cols > max_cols:
            max_cols = cols
            skiprows = i

    try:
        df = pd.read_csv(full_path, encoding=encoding, sep=delimiter, skiprows=skiprows)

        # 必要な情報の抽出
        columns = list(df.columns)
        shape = df.shape
        dtypes = df.dtypes.astype(str).to_dict()

        datetime_cols = [c for c in columns if "date" in c.lower() or "time" in c.lower() or "日時" in c]
        remark_cols = [c for c in columns if "remark" in c.lower() or "備考" in c]
        # 波長列（数値としてパース可能な列名）
        wave_cols = [c for c in columns if c.replace('.','',1).isdigit()]

        nans = int(df.isna().sum().sum())
        # 数値列のみinf, 負値をチェック
        num_df = df.select_dtypes(include=[np.number])
        infs = int(np.isinf(num_df).sum().sum()) if not num_df.empty else 0
        negatives = int((num_df < 0).sum().sum()) if not num_df.empty else 0

        remark_counts = {}
        if remark_cols:
            r_col = remark_cols[0]
            remark_counts = df[r_col].value_counts().to_dict()

        times_count = len(df)
        time_dups = 0
        if datetime_cols:
            # 単純に全時刻列の組み合わせで重複確認
            time_dups = int(df.duplicated(subset=datetime_cols).sum())

        waves = sorted([float(w) for w in wave_cols])
        wave_min = min(waves) if waves else None
        wave_max = max(waves) if waves else None
        wave_monotonic = all(waves[i] <= waves[i+1] for i in range(len(waves)-1)) if waves else False
        wave_dups = len(waves) - len(set(waves))
        has_350 = 350.0 in waves
        has_1050 = 1050.0 in waves
        points_in_range = len([w for w in waves if 350.0 <= w <= 1050.0])

        res = {
            "path": s['path'],
            "size": st.st_size,
            "encoding": encoding,
            "skiprows": skiprows,
            "delimiter": delimiter,
            "shape": shape,
            "datetime_cols": datetime_cols,
            "remark_cols": remark_cols,
            "num_wave_cols": len(wave_cols),
            "nans": nans,
            "infs": infs,
            "negatives": negatives,
            "remark_counts": remark_counts,
            "times_count": times_count,
            "time_dups": time_dups,
            "wave_min": wave_min,
            "wave_max": wave_max,
            "wave_monotonic": wave_monotonic,
            "wave_dups": wave_dups,
            "has_350": has_350,
            "has_1050": has_1050,
            "points_in_range": points_in_range,
            "head_preview": head_lines[:5] # 長すぎるので5行
        }
        audit_results.append(res)
    except Exception as e:
        audit_results.append({"path": s['path'], "error": str(e)})

for res in audit_results:
    print(f"\n-- ファイル: {res['path']} --")
    if "error" in res:
        print(f"解析エラー: {res['error']}")
        continue
    print(f"サイズ: {res['size']} bytes, エンコーディング: {res['encoding']}")
    print(f"推測区切り文字: '{res['delimiter']}', 推測skiprows: {res['skiprows']}")
    print(f"形状: {res['shape']} (行, 列)")
    print(f"日時列候補: {res['datetime_cols']}, 備考列候補: {res['remark_cols']}")
    print(f"波長列数: {res['num_wave_cols']} (最小: {res['wave_min']}, 最大: {res['wave_max']})")
    print(f"波長は単調増加か: {res['wave_monotonic']}, 波長の重複: {res['wave_dups']}")
    print(f"350nm存在: {res['has_350']}, 1050nm存在: {res['has_1050']}, 350-1050nm内点数: {res['points_in_range']}")
    print(f"観測時刻数(行数): {res['times_count']}, 同一ファイル内時刻重複: {res['time_dups']}")
    print(f"欠損値: {res['nans']}, 無限値: {res['infs']}, 負値: {res['negatives']}")
    print(f"Remark値の分布: {res['remark_counts']}")

# 【5. ファイル間の一貫性】
print("\n【5. ファイル間の一貫性】")
consistency = {
    "skiprows_match": len(set(r.get("skiprows") for r in audit_results if "skiprows" in r)) <= 1,
    "shape_cols_match": len(set(r.get("shape")[1] for r in audit_results if "shape" in r)) <= 1,
    "wave_min_match": len(set(r.get("wave_min") for r in audit_results if "wave_min" in r)) <= 1,
    "wave_max_match": len(set(r.get("wave_max") for r in audit_results if "wave_max" in r)) <= 1,
}
for k, v in consistency.items():
    print(f"{k}: {v}")

# 【6. Experiment B条件への適合性】
print("\n【6. Experiment B条件への適合性】")
print("- APE = HSRのみ: ファイル名・ディレクトリで絞り込み済み。")
print("- Remark = 1または2: 各ファイルの remark 列 (多くは 'Remark') でフィルタ可能。")
print("- 波長範囲 = 350～1050 nm: ヘッダー名から波長を特定し、範囲外をdrop可能。")
print("- 同一時刻平均 = 行わない: ")
any_time_dups = any(r.get("time_dups", 0) > 0 for r in audit_results)
print(f"  実際の同一時刻重複の存在: {'あり' if any_time_dups else '標本内にはなし'}")
print("  【重要】重複が存在する、または複数ファイルを結合する際のインデックスについて:")
print("  timestamp単独では一意性が保証できないため、[timestamp, source_file, source_row] などの複合キーを保持する設計が安全です。")

# 【8. 最終表示】
print("\n--- 最終表示 ---")
print("1. セッション所有権確認結果: 成功")
print("2. heartbeat更新結果: 成功 (status: inspecting_source_schema)")
print(f"3. 選択標本ファイル数: {len(sampled_files)} (年・季節を網羅)")
print(f"4. 共通CSV構造: {'一貫性あり' if all(consistency.values()) else '差分あり'}")
if any_time_dups:
    print("5. 同一時刻重複の有無: 【あり】※一意な識別子(source_row等)の付与が必須です。")
else:
    print("5. 同一時刻重複の有無: 標本内では確認されず (ただし全体では存在する可能性あり)")
print("6. Experiment Bの各条件を実装可能か: 実装可能")
print("7. 実装前に未確定の事項: ")
print("   - 抽出した波長ヘッダー(文字列)を具体的なfloatにキャストしてカラムを維持する方法")
print("   - 日時列(Date, Time)の厳密なパース文字列")
print("8. 次段階(APE計算・バッチ作成)へ進めるか: 監査結果の確認後、進めることが可能です。")


--- 現在セッションの所有権確認・heartbeat更新・HSR CSV少数標本監査 ---

【1. セッション所有権の確認】
✅ セッション所有権の完全一致を確認しました。このランタイムの処理用session_idを 152d121e-2e5f-445f-88e6-e644e5829f2d に同期しました。

【2. heartbeat更新】
✅ heartbeatを更新し、statusを inspecting_source_schema に変更しました。

【3. 調査対象ファイルの決定】
以下のファイルを決定論的に抽出しました:
 - 2011年初回: 201101/10HSR110101_301.csv
 - 2012年中央: 201207/10HSR120702_301.csv
 - 2013年中央: 201307/10HSR130702_301.csv
 - 2014年中央: 201407/10HSR140702_301.csv
 - 2015年最後: 201512/10HSR151231_301.csv
 - 春(4月)代表: 201304/10HSR130416_301.csv
 - 夏(8月)代表: 201308/10HSR130816_301.csv
 - 秋(10月)代表: 201310/10HSR131016_301.csv
 - 冬(1月)代表: 201401/10HSR140116_301.csv

【4. CSV構造の読み取り専用監査】

-- ファイル: 201101/10HSR110101_301.csv --
サイズ: 1087115 bytes, エンコーディング: SHIFT_JIS
推測区切り文字: ',', 推測skiprows: 0
形状: (97, 1356) (行, 列)
日時列候補: [], 備考列候補: []
波長列数: 0 (最小: None, 最大: None)
波長は単調増加か: False, 波長の重複: 0
350nm存在: False, 1050nm存在: False, 350-1050nm内点数: 0
観測時刻数(行数): 97, 同一ファイル内時刻重複: 0
欠損値: 0, 無限値: 0, 負値: 131047
Remark値の分布: {}

-- ファイル: 201207/10HSR1207

In [16]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import collections
import re
from pathlib import Path
import traceback
import sys

print("--- 日本語CSV構造に基づく厳密な再監査 ---")

# 【1. セッション確認とheartbeat更新】
print("\n【1. セッション確認とheartbeat更新】")
EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"

if 'session_id' not in globals():
    raise RuntimeError("【処理停止】メモリ上に session_id が存在しません。")
if session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"【処理停止】メモリ上の session_id ({session_id}) が期待値と一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock.json の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state.json の current_session_id が一致しません。")

now_utc = datetime.now(timezone.utc).isoformat()
updated_lock = dict(current_lock)
updated_lock["heartbeat"] = now_utc
updated_lock["status"] = "inspecting_japanese_csv_schema"

def atomic_update_lock(data_dict, target_path, sess_id):
    temp_path = target_path.parent / f"{target_path.stem}_{sess_id}_hb.tmp.json"
    if temp_path.exists():
        raise FileExistsError(f"一時ファイルが既に存在します: {temp_path}")
    try:
        with open(temp_path, "x", encoding="utf-8") as f:
            json.dump(data_dict, f, indent=4, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        with open(temp_path, "r", encoding="utf-8") as f:
            read_back = json.load(f)
        if read_back != data_dict:
            raise ValueError("一時ファイルの内容が不一致です。")
        os.replace(temp_path, target_path)
    finally:
        if temp_path.exists():
            os.unlink(temp_path)

atomic_update_lock(updated_lock, PROCESSING_LOCK_PATH, session_id)
print("✅ セッション確認成功。heartbeatを更新し、statusを 'inspecting_japanese_csv_schema' にしました。")

# 【2. 調査対象ファイルの準備】
sampled_files = [
    {'reason': '2011年初回', 'path': '201101/10HSR110101_301.csv'},
    {'reason': '2012年中央', 'path': '201207/10HSR120702_301.csv'},
    {'reason': '2013年中央', 'path': '201307/10HSR130702_301.csv'},
    {'reason': '2014年中央', 'path': '201407/10HSR140702_301.csv'},
    {'reason': '2015年最後', 'path': '201512/10HSR151231_301.csv'},
    {'reason': '春(4月)代表', 'path': '201304/10HSR130416_301.csv'},
    {'reason': '夏(8月)代表', 'path': '201308/10HSR130816_301.csv'},
    {'reason': '秋(10月)代表', 'path': '201310/10HSR131016_301.csv'},
    {'reason': '冬(1月)代表', 'path': '201401/10HSR140116_301.csv'}
]

audit_results = []
EXPECTED_HEADERS_TOP5 = ["地点番号", "地点名", "年月日", "時分", "リマーク"]
WAVE_PATTERN = re.compile(r"^日射強度\((\d+)nm\)")

all_timestamps = []
processing_failed = False

try:
    for s in sampled_files:
        print(f"\n=========================================")
        print(f"対象ファイル: {s['path']} ({s['reason']})")
        full_path = INPUT_ROOT / s['path']

        # 【修正2】ファイル名の日付抽出
        filename = Path(s["path"]).name
        m = re.fullmatch(r"10HSR(\d{6})_(\d{3})\.csv", filename)
        if not m:
            raise ValueError(f"ファイル名が正規表現に一致しません: {filename}")
        file_yymmdd = m.group(1)
        file_site = m.group(2)

        # 読込
        df = pd.read_csv(
            full_path,
            encoding="shift_jis",
            header=0,
            low_memory=False
        )

        cols = list(df.columns)
        if cols[:5] != EXPECTED_HEADERS_TOP5:
            raise ValueError(f"先頭5列が期待と異なります。実際の先頭: {cols[:5]}")

        # 波長列の抽出
        wave_cols_info = []
        units = set()
        for c in cols[5:]:
            m_wave = WAVE_PATTERN.match(c)
            if m_wave:
                wave_val = int(m_wave.group(1))
                unit = c[m_wave.end():] if len(c) > m_wave.end() else ""
                wave_cols_info.append({"col_name": c, "wave": wave_val, "unit": unit})
                units.add(unit)

        waves = [info["wave"] for info in wave_cols_info]
        expected_waves = list(range(350, 1701))

        # 【修正3】波長最小・最大判定とbool型保証
        wave_check = {
            "count_1351": bool(len(waves) == 1351),
            "min_350": bool(waves) and bool(min(waves) == 350),
            "max_1700": bool(waves) and bool(max(waves) == 1700),
            "order_match": bool(waves == expected_waves),
            "dups_0": bool(len(waves) == len(set(waves))),
            "missing_0": bool(len(set(expected_waves) - set(waves)) == 0),
            "extra_0": bool(len(set(waves) - set(expected_waves)) == 0),
            "count_350_1050": bool(len([w for w in waves if 350 <= w <= 1050]) == 701),
            "only_one_350": bool(waves.count(350) == 1),
            "only_one_1050": bool(waves.count(1050) == 1),
            "uniform_unit": bool(len(units) == 1)
        }

        print("--- 波長列検証 ---")
        for k, v in wave_check.items():
            print(f"  {k}: {v} (type: {type(v).__name__})")
        print(f"  単位表記: {list(units)[0] if units else 'N/A'}")

        # メタデータの確認
        df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')

        # 【修正8】source_rowの定義
        df['source_row_index'] = df.index
        df['source_line_number'] = df['source_row_index'] + 2
        df['source_file'] = s['path']

        # 日時の解析
        parsed_times = []
        parse_fails = 0
        count_2400 = 0
        date_mismatch = 0
        site_mismatch = 0

        for idx, row in df.iterrows():
            ymd = str(row['年月日']).strip()
            hm = str(row['時分']).strip()
            site_val = str(row['地点番号']).strip()

            if site_val != file_site:
                site_mismatch += 1

            try:
                base_date = datetime.strptime(ymd, "%Y/%m/%d")

                # 【修正2】年月日の比較はbase_dateで行う
                if base_date.strftime("%y%m%d") != file_yymmdd:
                    date_mismatch += 1

                if hm == "24:00":
                    count_2400 += 1
                    dt = base_date + timedelta(days=1)
                else:
                    h, m_str = hm.split(':')
                    dt = base_date.replace(hour=int(h), minute=int(m_str))
                parsed_times.append(dt)
            except Exception:
                parse_fails += 1
                parsed_times.append(pd.NaT)

        df['parsed_dt'] = parsed_times

        # 【修正1】Timedeltaの分変換
        print("\n--- 日時解析 ---")
        print(f"  24:00の件数: {count_2400}")
        print(f"  パース失敗数: {parse_fails}")
        print(f"  ファイル名日付と年月日の不一致数: {date_mismatch}")
        print(f"  ファイル名地点番号とCSV地点番号の不一致数: {site_mismatch}")

        valid_dts = df['parsed_dt'].dropna()
        if not valid_dts.empty:
            diff_minutes = df["parsed_dt"].diff().dt.total_seconds().div(60.0).dropna()
            print(f"  時刻間隔の値別件数: {diff_minutes.value_counts().to_dict()}")
            print(f"  10分間隔の件数: {(diff_minutes == 10.0).sum()}")
            print(f"  10分以外の件数: {(diff_minutes != 10.0).sum()}")
            print(f"  0分以下の件数: {(diff_minutes <= 0).sum()}")
            is_monotonic = (diff_minutes > 0).all()
            print(f"  時刻順序が単調増加か: {is_monotonic}")

        # 【修正7】有効行判定のベクトル化
        wave_cols_1050 = [info['col_name'] for info in wave_cols_info if 350 <= info['wave'] <= 1050]
        wave_cols_1700 = [info['col_name'] for info in wave_cols_info if 350 <= info['wave'] <= 1700]

        def get_valid_mask(block_cols):
            if not block_cols:
                return pd.Series(False, index=df.index)
            arr = df[block_cols].to_numpy(dtype=np.float64)
            return pd.Series(np.isfinite(arr).all(axis=1) & (arr != -9999).all(axis=1), index=df.index)

        valid_1050_mask = get_valid_mask(wave_cols_1050)
        valid_1700_mask = get_valid_mask(wave_cols_1700)
        rem_mask = (df['remark_num'] == 1) | (df['remark_num'] == 2)

        # -9999やその他の負値の集計
        arr_1700 = df[wave_cols_1700].to_numpy(dtype=np.float64) if wave_cols_1700 else np.array([])
        m9999_count = int((arr_1700 == -9999).sum()) if arr_1700.size > 0 else 0
        other_neg_count = int(((arr_1700 < 0) & (arr_1700 != -9999)).sum()) if arr_1700.size > 0 else 0

        # 【修正6】Remark 3の扱い
        rem3_mask = (df['remark_num'] == 3)
        rem3_count = rem3_mask.sum()
        rem3_df = df[rem3_mask]

        rem3_1050_nan = 0
        rem3_m9999 = 0
        rem3_other_neg = 0
        if rem3_count > 0 and wave_cols_1700:
            arr_rem3_1050 = rem3_df[wave_cols_1050].to_numpy(dtype=np.float64)
            rem3_1050_nan = int(np.isnan(arr_rem3_1050).any(axis=1).sum())

            arr_rem3_1700 = rem3_df[wave_cols_1700].to_numpy(dtype=np.float64)
            rem3_m9999 = int((arr_rem3_1700 == -9999).any(axis=1).sum())
            rem3_other_neg = int(((arr_rem3_1700 < 0) & (arr_rem3_1700 != -9999)).any(axis=1).sum())

        print("\n--- Remark 3 の詳細 ---")
        print(f"  Remark 3の行数: {rem3_count}")
        print(f"  うち350-1050nmにNaNがある行数: {rem3_1050_nan}")
        print(f"  うち-9999がある行数: {rem3_m9999}")
        print(f"  うち-9999以外の負値がある行数: {rem3_other_neg}")
        print(f"  Experiment BではRemark条件により除外される行数: {rem3_count}")

        # 結合用
        all_timestamps.append(df)

        # 成功時のみ追加
        audit_results.append({
            "path": s['path'],
            "cols": cols,
            "wave_cols": wave_cols_1700,
            "unit": list(units)[0] if units else None,
            "total_rows": len(df),
            "rem_counts": df['remark_num'].value_counts().to_dict(),
            "rem_1_2": rem_mask.sum(),
            "valid_1050": valid_1050_mask.sum(),
            "valid_1700": valid_1700_mask.sum(),
            "m9999_count": m9999_count,
            "other_neg_count": other_neg_count,
            "count_2400": count_2400,
            "parse_fails": parse_fails,
            "date_mismatch": date_mismatch,
            "site_mismatch": site_mismatch,
            "wave_check": wave_check
        })

except Exception as e:
    # 【修正10】例外時の判定
    processing_failed = True
    print("\n【処理停止】監査中に例外が発生しました。")
    print(f"例外ファイル名: {s['path']}")
    print(f"例外型: {type(e).__name__}")
    print(f"メッセージ: {e}")
    traceback.print_exc(file=sys.stdout)

# 【修正5】全9ファイルの検証 ＆ 最終表示
if not processing_failed:
    print("\n=========================================")
    print("【最終検証と結果】")

    all_valid = True
    if len(audit_results) != 9:
        print(f"[失敗] audit_resultsの件数が9ではありません: {len(audit_results)}")
        all_valid = False

    ref_cols = audit_results[0]['cols']
    ref_wave_cols = audit_results[0]['wave_cols']
    ref_unit = audit_results[0]['unit']

    total_rows = 0
    total_rem_1_2 = 0
    total_valid_1050 = 0
    total_valid_1700 = 0
    total_m9999 = 0
    total_other_neg = 0
    total_2400 = 0
    total_parse_fails = 0
    total_date_mismatch = 0
    total_site_mismatch = 0

    for res in audit_results:
        if not all(res['wave_check'].values()):
            print(f"[失敗] 波長チェック失敗項目あり: {res['path']}")
            all_valid = False
        if res['parse_fails'] > 0:
            print(f"[失敗] 日時パース失敗あり: {res['path']}")
            all_valid = False
        if res['date_mismatch'] > 0:
            print(f"[失敗] ファイル名日付不一致あり: {res['path']}")
            all_valid = False
        if res['site_mismatch'] > 0:
            print(f"[失敗] ファイル名地点番号不一致あり: {res['path']}")
            all_valid = False
        if res['cols'] != ref_cols:
            print(f"[失敗] 列名不一致あり: {res['path']}")
            all_valid = False
        if res['wave_cols'] != ref_wave_cols:
            print(f"[失敗] 波長列順不一致あり: {res['path']}")
            all_valid = False
        if res['unit'] != ref_unit:
            print(f"[失敗] 単位不一致あり: {res['path']}")
            all_valid = False

        total_rows += res['total_rows']
        total_rem_1_2 += res['rem_1_2']
        total_valid_1050 += res['valid_1050']
        total_valid_1700 += res['valid_1700']
        total_m9999 += res['m9999_count']
        total_other_neg += res['other_neg_count']
        total_2400 += res['count_2400']
        total_parse_fails += res['parse_fails']
        total_date_mismatch += res['date_mismatch']
        total_site_mismatch += res['site_mismatch']

    # 【修正4】pd.concat(all_timestamps, ignore_index=True)
    combined_df = pd.concat(all_timestamps, ignore_index=True)

    # 【修正9】重複時刻
    dups_all = combined_df[combined_df.duplicated(subset=['parsed_dt'], keep=False)]

    # フィルタ後の重複
    filter_mask = ((combined_df['remark_num'] == 1) | (combined_df['remark_num'] == 2)) & \
                  combined_df[wave_cols_1050].notna().all(axis=1) & \
                  ~np.isinf(combined_df[wave_cols_1050].fillna(0)).any(axis=1) & \
                  (combined_df[wave_cols_1050] != -9999).all(axis=1)
    filtered_df = combined_df[filter_mask]
    dups_filtered = filtered_df[filtered_df.duplicated(subset=['parsed_dt'], keep=False)]

    print("\n--- 重複時刻の確認 ---")
    print(f"  フィルタ前の重複行数: {len(dups_all)}")
    print(f"  Remark1,2かつ350-1050nm有効行の重複行数: {len(dups_filtered)}")
    if len(dups_all) > 0:
        print("\n  [重複詳細 (最大20件)]")
        disp_cols = ['parsed_dt', 'source_file', 'source_row_index', 'source_line_number', 'リマーク']
        print(dups_all.sort_values('parsed_dt')[disp_cols].head(20).to_string())

    print("\n--- 全9ファイルサマリー ---")
    for res in audit_results:
        print(f"  {res['path']}: 行数={res['total_rows']}, Remark分布={res['rem_counts']}")

    print(f"\n  全9ファイル合計行数: {total_rows}")
    print(f"  Remark 1・2採用候補数: {total_rem_1_2}")
    print(f"  350-1050nm有効数: {total_valid_1050}")
    print(f"  350-1700nm有効数: {total_valid_1700}")
    print(f"  両判定の差分: {total_valid_1050 - total_valid_1700}")
    print(f"  -9999件数(セル数): {total_m9999}")
    print(f"  -9999以外の負値件数(セル数): {total_other_neg}")
    print(f"  24:00件数: {total_2400}")
    print(f"  日時パース失敗数: {total_parse_fails}")
    print(f"  日付・地点不一致数: {total_date_mismatch + total_site_mismatch}")
    print(f"  重複timestamp数(フィルタ前): {len(dups_all)}")
    print(f"  CSV構造完全一致結果: {all_valid}")

    if all_valid:
        print("\n日本語CSV構造の特定成功。APE試算前監査へ進める")


--- 日本語CSV構造に基づく厳密な再監査 ---

【1. セッション確認とheartbeat更新】
✅ セッション確認成功。heartbeatを更新し、statusを 'inspecting_japanese_csv_schema' にしました。

対象ファイル: 201101/10HSR110101_301.csv (2011年初回)
--- 波長列検証 ---
  count_1351: True (type: bool)
  min_350: True (type: bool)
  max_1700: True (type: bool)
  order_match: True (type: bool)
  dups_0: True (type: bool)
  missing_0: True (type: bool)
  extra_0: True (type: bool)
  count_350_1050: True (type: bool)
  only_one_350: True (type: bool)
  only_one_1050: True (type: bool)
  uniform_unit: True (type: bool)
  単位表記: [W/m2/μm]

--- 日時解析 ---
  24:00の件数: 0
  パース失敗数: 0
  ファイル名日付と年月日の不一致数: 0
  ファイル名地点番号とCSV地点番号の不一致数: 0
  時刻間隔の値別件数: {10.0: 96}
  10分間隔の件数: 96
  10分以外の件数: 0
  0分以下の件数: 0
  時刻順序が単調増加か: True

--- Remark 3 の詳細 ---
  Remark 3の行数: 97
  うち350-1050nmにNaNがある行数: 0
  うち-9999がある行数: 97
  うち-9999以外の負値がある行数: 0
  Experiment BではRemark条件により除外される行数: 97

対象ファイル: 201207/10HSR120702_301.csv (2012年中央)
--- 波長列検証 ---
  count_1351: True (type: bool)
  min_350: True (type: bool)


In [17]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import traceback
import sys
import re
from pathlib import Path

print("--- 9標本限定・メモリ上のみのAPE試算 ---")

# 【1. セッション・設定確認】
print("\n【1. セッション・設定確認】")
EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"

if 'session_id' not in globals() or session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"【処理停止】メモリ上の session_id が {EXPECTED_SESSION_ID} と一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state の current_session_id が一致しません。")

EXPECTED_CONFIG_HASH = "f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9"
if CONFIG["w_min"] != 350 or CONFIG["w_max"] != 1050:
    raise ValueError("CONFIG の波長範囲が 350-1050 ではありません。")
if set(CONFIG["remarks_used"]) != {1, 2}:
    raise ValueError("CONFIG の remarks_used が [1, 2] ではありません。")
if CONFIG["ape_valid_range_eV"] != [1.2, 2.2]:
    raise ValueError("CONFIG の ape_valid_range_eV が [1.2, 2.2] ではありません。")
if CONFIG["formula"] != "1239.84193 * sum(G) / sum(G * wavelength)":
    raise ValueError("CONFIG の formula が一致しません。")
if config_hash != EXPECTED_CONFIG_HASH or manifest_data.get("config_hash") != EXPECTED_CONFIG_HASH:
    raise ValueError("config_hash が一致しません。")

now_utc = datetime.now(timezone.utc).isoformat()
updated_lock = dict(current_lock)
updated_lock["heartbeat"] = now_utc
updated_lock["status"] = "testing_ape_on_samples"

def atomic_update_lock(data_dict, target_path, sess_id):
    temp_path = target_path.parent / f"{target_path.stem}_{sess_id}_hb.tmp.json"
    if temp_path.exists():
        raise FileExistsError(f"一時ファイルが既に存在します: {temp_path}")
    try:
        with open(temp_path, "x", encoding="utf-8") as f:
            json.dump(data_dict, f, indent=4, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        with open(temp_path, "r", encoding="utf-8") as f:
            read_back = json.load(f)
        if read_back != data_dict:
            raise ValueError("一時ファイルの内容が不一致です。")
        os.replace(temp_path, target_path)
    finally:
        if temp_path.exists():
            os.unlink(temp_path)

atomic_update_lock(updated_lock, PROCESSING_LOCK_PATH, session_id)
print("✅ セッション・設定確認成功。heartbeatを更新し、statusを 'testing_ape_on_samples' にしました。")

# 【2 & 3 & 4. 対象ファイル読込と正確な採用数】
print("\n【2 & 3 & 4. 対象ファイル読込と正確な採用数】")
EXPECTED_HEADERS_TOP5 = ["地点番号", "地点名", "年月日", "時分", "リマーク"]
WAVE_PATTERN = re.compile(r"^日射強度\((\d+)nm\)")

total_rows_all = 0
total_rem_1_2_all = 0
total_valid_1050_all = 0
total_adopted_all = 0
total_rem_valid_wave_invalid_all = 0
total_wave_valid_rem_invalid_all = 0
total_adopted_rem1 = 0
total_adopted_rem2 = 0

adopted_dfs = []
all_other_negatives = []

processing_failed = False

for s in sampled_files:
    filename = Path(s["path"]).name
    m_file = re.fullmatch(r"10HSR(\d{6})_(\d{3})\.csv", filename)
    if not m_file:
        processing_failed = True
        print(f"[エラー] ファイル名が10HSRではありません: {filename}")
        continue

    full_path = INPUT_ROOT / s['path']
    df = pd.read_csv(full_path, encoding="shift_jis", header=0, low_memory=False)

    cols = list(df.columns)
    if cols[:5] != EXPECTED_HEADERS_TOP5:
        processing_failed = True
        print(f"[エラー] 先頭5列構造不一致: {filename}")
        continue

    wave_cols = [c for c in cols[5:] if WAVE_PATTERN.match(c)]
    if len(wave_cols) != 1351:
        processing_failed = True
        print(f"[エラー] 波長列が1351列ではありません: {filename} ({len(wave_cols)}列)")
        continue

    wave_cols_1050 = wave_cols[:701] # 350-1050nm

    df['source_row_index'] = df.index
    df['source_line_number'] = df.index + 2
    df['source_file'] = s['path']

    df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')

    parsed_times = []
    for idx, row in df.iterrows():
        try:
            ymd = str(row['年月日']).strip()
            hm = str(row['時分']).strip()
            base_date = datetime.strptime(ymd, "%Y/%m/%d")
            if hm == "24:00":
                dt = base_date + timedelta(days=1)
            else:
                h, m_str = hm.split(':')
                dt = base_date.replace(hour=int(h), minute=int(m_str))
            parsed_times.append(dt)
        except Exception:
            parsed_times.append(pd.NaT)
    df['Datetime'] = parsed_times

    # 条件マスク
    mask_site = df['地点番号'] == 301
    mask_rem = (df['remark_num'] == 1) | (df['remark_num'] == 2)
    mask_dt = df['Datetime'].notna()

    arr_1050 = df[wave_cols_1050].to_numpy(dtype=np.float64)
    mask_valid_1050 = np.isfinite(arr_1050).all(axis=1) & (arr_1050 != -9999).all(axis=1)

    # 負値チェック
    neg_mask = (arr_1050 < 0) & (arr_1050 != -9999)
    if neg_mask.any():
        neg_rows = df[neg_mask.any(axis=1)]
        for idx, row in neg_rows.iterrows():
            row_arr = arr_1050[idx]
            min_val = row_arr[(row_arr < 0) & (row_arr != -9999)].min()
            all_other_negatives.append({
                'source_file': row['source_file'],
                'source_line_number': row['source_line_number'],
                'min_neg_val': min_val
            })

    mask_adopted = mask_site & mask_rem & mask_dt & mask_valid_1050

    t_rows = len(df)
    t_rem_1_2 = mask_rem.sum()
    t_valid_1050 = mask_valid_1050.sum()
    t_adopted = mask_adopted.sum()
    t_rem_valid_wave_invalid = (mask_rem & ~mask_valid_1050).sum()
    t_wave_valid_rem_invalid = (~mask_rem & mask_valid_1050).sum()
    t_rem1 = (mask_adopted & (df['remark_num'] == 1)).sum()
    t_rem2 = (mask_adopted & (df['remark_num'] == 2)).sum()

    print(f"  {s['path']}:")
    print(f"    全観測行数: {t_rows}")
    print(f"    Remark 1・2行数: {t_rem_1_2}")
    print(f"    350-1050nm有効行数: {t_valid_1050}")
    print(f"    Remark 1・2かつ波長有効(採用数): {t_adopted}")
    print(f"    Remark条件合致だが波長無効: {t_rem_valid_wave_invalid}")
    print(f"    波長有効だがRemark条件外: {t_wave_valid_rem_invalid}")
    print(f"    (採用内訳) Remark 1: {t_rem1}, Remark 2: {t_rem2}")

    total_rows_all += t_rows
    total_rem_1_2_all += t_rem_1_2
    total_valid_1050_all += t_valid_1050
    total_adopted_all += t_adopted
    total_rem_valid_wave_invalid_all += t_rem_valid_wave_invalid
    total_wave_valid_rem_invalid_all += t_wave_valid_rem_invalid
    total_adopted_rem1 += t_rem1
    total_adopted_rem2 += t_rem2

    if t_adopted > 0:
        adopted_dfs.append(df[mask_adopted].copy())

print("\n--- 全9ファイル合計 ---")
print(f"全観測行数: {total_rows_all}")
print(f"Remark 1・2行数: {total_rem_1_2_all}")
print(f"350-1050nm有効行数: {total_valid_1050_all}")
print(f"Remark 1・2かつ波長有効(総採用数): {total_adopted_all}")
print(f"Remark条件合致だが波長無効: {total_rem_valid_wave_invalid_all}")
print(f"波長有効だがRemark条件外: {total_wave_valid_rem_invalid_all}")
print(f"(採用内訳) Remark 1: {total_adopted_rem1}, Remark 2: {total_adopted_rem2}")

if all_other_negatives:
    print("\n--- -9999以外の負値が存在する行 (採用有無問わず表示) ---")
    for n in all_other_negatives[:20]:
        print(f"  {n['source_file']} (Line {n['source_line_number']}): 最小負値 = {n['min_neg_val']}")
    if len(all_other_negatives) > 20:
        print(f"  ...他 {len(all_other_negatives) - 20} 件")

if not adopted_dfs:
    raise RuntimeError("【処理停止】採用可能なデータが0件です。")

# 結合
combined_adopted = pd.concat(adopted_dfs, ignore_index=True)

# 【5. APE計算】
print("\n【5. APE計算】")
wavelength_nm = np.arange(350, 1051, dtype=np.float64)
G_array = combined_adopted[wave_cols_1050].to_numpy(dtype=np.float64)

if G_array.shape[1] != 701:
    raise ValueError(f"Gの列数が701ではありません: {G_array.shape[1]}")
if len(wavelength_nm) != 701:
    raise ValueError(f"wavelength_nmの要素数が701ではありません: {len(wavelength_nm)}")

numerator = np.sum(G_array, axis=1)
denominator = np.sum(G_array * wavelength_nm[None, :], axis=1)

combined_adopted['numerator'] = numerator
combined_adopted['denominator'] = denominator

bad_num_den = combined_adopted[~np.isfinite(numerator) | ~np.isfinite(denominator) | (numerator <= 0) | (denominator <= 0)]
if len(bad_num_den) > 0:
    print("\n[警告] 分子・分母が0以下または非有限の行が存在します。")
    disp_cols = ['source_file', 'source_line_number', 'remark_num', 'numerator', 'denominator']
    print(bad_num_den[disp_cols].head(20).to_string())
    processing_failed = True

APE = 1239.84193 * numerator / denominator
combined_adopted['APE'] = APE
print("APE計算完了 (float64)。")

# 【6. 独立な式による照合】
print("\n【6. 独立な式による照合】")
weighted_mean_wavelength_nm = denominator / numerator
APE_check = 1239.84193 / weighted_mean_wavelength_nm
combined_adopted['APE_check'] = APE_check

abs_diff = np.abs(APE - APE_check)
max_diff = abs_diff.max()
print(f"APEとAPE_checkの最大絶対差: {max_diff:.3e} eV")
if max_diff > 1e-12:
    raise ValueError("【処理停止】独立計算の差が1e-12 eVを超えました。")

print("\n--- 先頭5採用行の計算詳細 ---")
disp_calc_cols = ['Datetime', 'remark_num', 'source_file', 'source_row_index', 'source_line_number',
                  'numerator', 'denominator', 'APE_check'] # APE_check column represents the secondary test but let's just print both APE and APE_check
for i in range(min(5, len(combined_adopted))):
    r = combined_adopted.iloc[i]
    print(f"Row {i}:")
    print(f"  Datetime: {r['Datetime']}, Remark: {r['remark_num']}")
    print(f"  Source: {r['source_file']} (RowIndex: {r['source_row_index']}, Line: {r['source_line_number']})")
    print(f"  numerator: {r['numerator']:.4f}")
    print(f"  denominator: {r['denominator']:.4f}")
    print(f"  weighted_mean_wavelength: {r['denominator'] / r['numerator']:.4f}")
    print(f"  APE: {r['APE']:.8f}, APE_check: {r['APE_check']:.8f}, Diff: {abs_diff[i]:.3e}")

# 【7. APE範囲の検証】
print("\n【7. APE範囲の検証】")
apes = combined_adopted['APE']

cnt_total = len(apes)
cnt_nan = apes.isna().sum()
cnt_pinf = (apes == np.inf).sum()
cnt_minf = (apes == -np.inf).sum()

valid_apes = apes[np.isfinite(apes)]
cnt_under = (valid_apes < 1.2).sum()
cnt_in_range = ((valid_apes >= 1.2) & (valid_apes <= 2.2)).sum()
cnt_over = (valid_apes > 2.2).sum()

print(f"  計算可能行数(全体): {cnt_total}")
print(f"  APE < 1.2 eV: {cnt_under}")
print(f"  1.2 <= APE <= 2.2 eV: {cnt_in_range}")
print(f"  APE > 2.2 eV: {cnt_over}")
print(f"  NaN: {cnt_nan}")
print(f"  +inf: {cnt_pinf}")
print(f"  -inf: {cnt_minf}")

out_of_range = combined_adopted[~( (combined_adopted['APE'] >= 1.2) & (combined_adopted['APE'] <= 2.2) )]
if len(out_of_range) > 0:
    print("\n  [範囲外・非有限行 詳細 (最大20件)]")
    disp_out = ['Datetime', 'APE', 'remark_num', 'source_file', 'source_line_number', 'numerator', 'denominator']
    print(out_of_range[disp_out].head(20).to_string())

# 【8. APE統計】
print("\n【8. APE統計】")
def print_stats(series, title):
    print(f"--- {title} ---")
    if len(series) == 0:
        print("  データなし")
        return
    s_desc = series.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
    for k, v in s_desc.items():
        print(f"  {k}: {v:.6f}")

print_stats(valid_apes, "範囲フィルタ前のAPE統計 (有限値のみ)")
apes_in_range = valid_apes[(valid_apes >= 1.2) & (valid_apes <= 2.2)]
print_stats(apes_in_range, "1.2～2.2 eV範囲内のAPE統計")

print("\n--- ファイル別統計 (1.2～2.2 eV範囲内) ---")
in_range_df = combined_adopted[(combined_adopted['APE'] >= 1.2) & (combined_adopted['APE'] <= 2.2)]
if not in_range_df.empty:
    file_stats = in_range_df.groupby('source_file')['APE'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(file_stats.to_string())

print("\n--- Remark別統計 (1.2～2.2 eV範囲内) ---")
if not in_range_df.empty:
    rem_stats = in_range_df.groupby('remark_num')['APE'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(rem_stats.to_string())

if total_adopted_rem2 == 0:
    print("\n※ Remark 2は標本内0件であり、全データでの動作は未検証")

# 【9. 出力候補DataFrame】
print("\n【9. 出力候補DataFrame作成】")
output_cols = ['Datetime', '地点番号', '地点名', 'remark_num', 'APE', 'source_file', 'source_row_index', 'source_line_number']
final_df = combined_adopted[output_cols].rename(columns={'地点番号': 'SiteNum', '地点名': 'SiteName', 'remark_num': 'Remark'})

print(f"  元の観測採用行数: {total_adopted_all}")
print(f"  出力DataFrame行数: {len(final_df)}")
if total_adopted_all != len(final_df):
    processing_failed = True
    print("  [エラー] 行数が1対1ではありません。")

dups_keys = final_df[final_df.duplicated(subset=['source_file', 'source_line_number'], keep=False)]
if len(dups_keys) > 0:
    processing_failed = True
    print("  [エラー] source_fileとsource_line_numberの複合キーが重複しています。")
else:
    print("  source_fileとsource_line_numberの複合キーが一意であることを確認。")

time_dups = final_df[final_df.duplicated(subset=['Datetime'], keep=False)]
if len(time_dups) > 0:
    print(f"  timestamp重複の有無: あり ({len(time_dups)}行)")
    print("  ※timestampが重複しても平均・結合等の処理は行っていません。")
else:
    print("  timestamp重複の有無: なし (標本内)")

# 【10. 既存Notebookとの対応】
print("\n【10. 既存Notebookとの対応と相違点】")
print("既存Notebookの式 `1239.84193 * sum(G) / sum(G * wavelength)` と完全に同じ式を使用しています。")
print("■ 既存Notebookとの違い:")
print("  - HSRファイルだけを使用")
print("  - Remark 1・2だけを使用")
print("  - 有効性判定をAPE対象の350～1050 nmだけで実施")
print("  - source_fileとsource_line_numberを保持")
print("  - 同一時刻平均を行わない")
print("  - 範囲外APEを削除する前に件数を記録")

# 【12. 最終判定】
print("\n=========================================")
print("【最終判定】")
if not processing_failed:
    print("9標本APE試算成功。全件メタデータ・品質走査へ進める")
else:
    print("試算中にエラーや異常値が検出されました。ログを確認してください。")


--- 9標本限定・メモリ上のみのAPE試算 ---

【1. セッション・設定確認】
✅ セッション・設定確認成功。heartbeatを更新し、statusを 'testing_ape_on_samples' にしました。

【2 & 3 & 4. 対象ファイル読込と正確な採用数】
  201101/10HSR110101_301.csv:
    全観測行数: 97
    Remark 1・2行数: 0
    350-1050nm有効行数: 0
    Remark 1・2かつ波長有効(採用数): 0
    Remark条件合致だが波長無効: 0
    波長有効だがRemark条件外: 0
    (採用内訳) Remark 1: 0, Remark 2: 0
  201207/10HSR120702_301.csv:
    全観測行数: 97
    Remark 1・2行数: 52
    350-1050nm有効行数: 74
    Remark 1・2かつ波長有効(採用数): 52
    Remark条件合致だが波長無効: 0
    波長有効だがRemark条件外: 22
    (採用内訳) Remark 1: 52, Remark 2: 0
  201307/10HSR130702_301.csv:
    全観測行数: 97
    Remark 1・2行数: 68
    350-1050nm有効行数: 86
    Remark 1・2かつ波長有効(採用数): 68
    Remark条件合致だが波長無効: 0
    波長有効だがRemark条件外: 18
    (採用内訳) Remark 1: 68, Remark 2: 0
  201407/10HSR140702_301.csv:
    全観測行数: 97
    Remark 1・2行数: 70
    350-1050nm有効行数: 86
    Remark 1・2かつ波長有効(採用数): 70
    Remark条件合致だが波長無効: 0
    波長有効だがRemark条件外: 16
    (採用内訳) Remark 1: 70, Remark 2: 0
  201512/10HSR151231_301.csv:
    全観測行数: 97
    Re

In [18]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import traceback
import sys
import re
from pathlib import Path
import collections

print("--- batch_00021 dry-run 処理 ---")

# 【1. セッション確認】
print("\n【1. セッション確認】")
EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"
TARGET_BATCH = "batch_00021"
TARGET_SAMPLE_FILE = "201207/10HSR120702_301.csv"

if 'session_id' not in globals() or session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"【処理停止】メモリ上の session_id が {EXPECTED_SESSION_ID} と一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state の current_session_id が一致しません。")

EXPECTED_CONFIG_HASH = "f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9"
if config_hash != EXPECTED_CONFIG_HASH or manifest_data.get("config_hash") != EXPECTED_CONFIG_HASH:
    raise ValueError("config_hash が一致しません。")

def atomic_update_lock(status_str):
    now_utc = datetime.now(timezone.utc).isoformat()
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        lock = json.load(f)
    if lock.get("session_id") != EXPECTED_SESSION_ID:
        raise RuntimeError("所有権が失われました。")
    lock["heartbeat"] = now_utc
    lock["status"] = status_str
    temp_path = PROCESSING_LOCK_PATH.parent / f"{PROCESSING_LOCK_PATH.stem}_{EXPECTED_SESSION_ID}_hb.tmp.json"
    if temp_path.exists():
        os.unlink(temp_path)
    with open(temp_path, "x", encoding="utf-8") as f:
        json.dump(lock, f, indent=4, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    with open(temp_path, "r", encoding="utf-8") as f:
        read_back = json.load(f)
    if read_back != lock:
        raise ValueError("一時ファイルの内容が不一致です。")
    os.replace(temp_path, PROCESSING_LOCK_PATH)

atomic_update_lock(f"dry_running_{TARGET_BATCH}")
print(f"✅ セッション・設定確認成功。statusを 'dry_running_{TARGET_BATCH}' にしました。")

# 【2. バッチ確認】
print("\n【2. バッチ確認】")
batches = manifest_data.get("batches", {})
if TARGET_BATCH not in batches:
    raise ValueError(f"{TARGET_BATCH} がmanifestに存在しません。")
batch_files = batches[TARGET_BATCH]

if len(batch_files) != 25:
    raise ValueError(f"{TARGET_BATCH}のファイル数が25ではありません: {len(batch_files)}")

sorted_hsr = manifest_data.get("sorted_hsr_files", [])
if not all(f in sorted_hsr for f in batch_files):
    raise ValueError("一部のファイルがsorted_hsr_filesに存在しません。")

file_metadata = manifest_data.get("file_metadata", {})
for rel_p in batch_files:
    if rel_p not in file_metadata:
        raise ValueError(f"メタデータが見つかりません: {rel_p}")
    full_path = INPUT_ROOT / rel_p
    st = full_path.stat()
    meta = file_metadata[rel_p]
    if st.st_size != meta["size"] or st.st_mtime_ns != meta["mtime_ns"]:
        raise ValueError(f"ファイルのサイズまたはmtimeが変更されています: {rel_p}")

if TARGET_SAMPLE_FILE not in batch_files:
    raise ValueError(f"{TARGET_BATCH}に {TARGET_SAMPLE_FILE} が含まれていません。")

if TARGET_BATCH in current_resume.get("completed_batch_ids", []):
    raise ValueError(f"{TARGET_BATCH}は既に完了済みです。")

batch_parquet_path = BATCH_DIR / f"{TARGET_BATCH}.parquet"
batch_json_path = BATCH_DIR / f"{TARGET_BATCH}_quality.json"
if batch_parquet_path.exists() or batch_json_path.exists():
    raise ValueError(f"{TARGET_BATCH}の生成物が既に存在します。")

print(f"✅ バッチ事前確認に成功しました。")

# 【3. 本番用関数の定義】
print("\n【3. 本番用関数の定義】")
EXPECTED_HEADERS_TOP5 = ["地点番号", "地点名", "年月日", "時分", "リマーク"]
WAVE_PATTERN = re.compile(r"^日射強度\((\d+)nm\)")

def parse_hsr_header(filepath):
    # ファイルヘッダーを読み込んで構造を検証し、波長列情報を返す
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False, nrows=0)
    cols = list(df.columns)
    if cols[:5] != EXPECTED_HEADERS_TOP5:
        raise ValueError(f"先頭5列構造不一致: 実際の先頭={cols[:5]}")
    if len(cols) != 1356:
        raise ValueError(f"全列数が1356ではありません: {len(cols)}")

    wave_cols = []
    for c in cols[5:]:
        m = WAVE_PATTERN.match(c)
        if not m:
            raise ValueError(f"波長列名が想定外です: {c}")
        wave = int(m.group(1))
        wave_cols.append({'col_name': c, 'wave': wave})

    waves = [info['wave'] for info in wave_cols]
    if waves != list(range(350, 1701)):
        raise ValueError("波長列が350-1700nmの1nm刻みではありません。")

    unit = cols[5].split(']')[0].split('[')[-1]
    if unit != "W/m2/μm":
        raise ValueError(f"単位が異なります: {unit}")

    target_wave_cols = [info['col_name'] for info in wave_cols if 350 <= info['wave'] <= 1050]
    full_wave_cols = [info['col_name'] for info in wave_cols if 350 <= info['wave'] <= 1700]

    if len(target_wave_cols) != 701:
         raise ValueError(f"APE対象波長列数が701ではありません: {len(target_wave_cols)}")

    return target_wave_cols, full_wave_cols

def parse_datetime_columns(df, source_file):
    # 24:00処理やパースエラー件数をカウント
    parsed_times = []
    failures = 0
    count_2400 = 0
    for idx, row in df.iterrows():
        try:
            ymd = str(row['年月日']).strip()
            hm = str(row['時分']).strip()
            base_date = datetime.strptime(ymd, "%Y/%m/%d")
            if hm == "24:00":
                dt = base_date + timedelta(days=1)
                count_2400 += 1
            else:
                h, m_str = hm.split(':')
                dt = base_date.replace(hour=int(h), minute=int(m_str))
            parsed_times.append(dt)
        except Exception:
            failures += 1
            parsed_times.append(pd.NaT)
    df['Datetime'] = parsed_times
    return df, failures, count_2400

def calculate_ape_for_file(filepath, relative_path, config):
    target_wave_cols, full_wave_cols = parse_hsr_header(filepath)
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False)

    m_file = re.fullmatch(r"10HSR(\d{6})_(\d{3})\.csv", Path(filepath).name)
    if not m_file:
         raise ValueError(f"ファイル名不一致: {Path(filepath).name}")
    file_yymmdd = m_file.group(1)
    file_site = m_file.group(2)

    raw_rows = len(df)
    df['source_row_index'] = df.index
    df['source_line_number'] = df.index + 2
    df['source_file'] = relative_path
    df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')

    df, failures, count_2400 = parse_datetime_columns(df, relative_path)

    # 地点・日付不一致
    site_mismatches = (df['地点番号'].astype(str) != file_site).sum()
    date_mismatches = 0
    for d in df['Datetime'].dropna():
        # 24:00処理されたものは翌日になるため、元データか変換前のbase_dateで判定すべきだが、
        # 既存ロジックに合わせてDatetimeで判定するとズレる場合がある。
        # 今回は簡略化のため一旦0とするか、厳密にはパース時に判定する。ここではパース時の日付で判定する。
        pass
    # パース時に判定すべきだが、今回は独立させて確認
    def check_date(row):
         try:
            base_date = datetime.strptime(str(row['年月日']).strip(), "%Y/%m/%d")
            if base_date.strftime("%y%m%d") != file_yymmdd: return 1
         except: pass
         return 0
    date_mismatches = df.apply(check_date, axis=1).sum()

    # 重複時刻
    timestamp_duplicate_rows = df.duplicated(subset=['Datetime'], keep=False).sum()

    # Remark
    rem_counts = df['remark_num'].value_counts().to_dict()
    mask_rem = (df['remark_num'] == 1) | (df['remark_num'] == 2)
    mask_dt = df['Datetime'].notna()

    # 波長有効性 (target)
    arr_1050 = df[target_wave_cols].to_numpy(dtype=np.float64)
    mask_valid_1050 = np.isfinite(arr_1050).all(axis=1) & (arr_1050 != -9999).all(axis=1)
    m9999_cells_target = (arr_1050 == -9999).sum()
    other_neg_cells_target = ((arr_1050 < 0) & (arr_1050 != -9999)).sum()
    nan_cells_target = np.isnan(arr_1050).sum()

    # 波長有効性 (full)
    arr_1700 = df[full_wave_cols].to_numpy(dtype=np.float64)
    mask_valid_1700 = np.isfinite(arr_1700).all(axis=1) & (arr_1700 != -9999).all(axis=1)
    m9999_cells_full = (arr_1700 == -9999).sum()

    mask_adopted = mask_rem & mask_dt & mask_valid_1050

    adopted_df = df[mask_adopted].copy()

    # APE計算
    ape_below_1_2 = 0
    ape_in_range = 0
    ape_above_2_2 = 0
    num_den_invalid = 0

    if not adopted_df.empty:
        wavelength_nm = np.arange(350, 1051, dtype=np.float64)
        G_adopted = adopted_df[target_wave_cols].to_numpy(dtype=np.float64)

        numerator = np.sum(G_adopted, axis=1)
        denominator = np.sum(G_adopted * wavelength_nm[None, :], axis=1)

        valid_num_den = np.isfinite(numerator) & np.isfinite(denominator) & (numerator > 0) & (denominator > 0)
        num_den_invalid = (~valid_num_den).sum()

        APE = 1239.84193 * numerator / denominator
        adopted_df['APE'] = APE

        # 独立照合
        APE_check = 1239.84193 / (denominator / numerator)
        abs_diff = np.abs(APE - APE_check)
        if abs_diff.max() > 1e-12:
            raise ValueError(f"APE_check不一致 (最大差: {abs_diff.max():.3e})")

        # 範囲
        ape_below_1_2 = (APE < 1.2).sum()
        ape_in_range = ((APE >= 1.2) & (APE <= 2.2)).sum()
        ape_above_2_2 = (APE > 2.2).sum()

        adopted_df = adopted_df[(adopted_df['APE'] >= 1.2) & (adopted_df['APE'] <= 2.2)]

    stats = {
        'raw_rows': raw_rows,
        'remark_counts': rem_counts,
        'remark_1_count': rem_counts.get(1, 0),
        'remark_2_count': rem_counts.get(2, 0),
        'target_350_1050_valid_rows': int(mask_valid_1050.sum()),
        'full_350_1700_valid_rows': int(mask_valid_1700.sum()),
        'target_valid_but_full_invalid_rows': int((mask_valid_1050 & ~mask_valid_1700).sum()),
        'remark_valid_but_target_invalid_rows': int((mask_rem & ~mask_valid_1050).sum()),
        'numerator_or_denominator_invalid_rows': int(num_den_invalid),
        'ape_below_1_2': int(ape_below_1_2),
        'ape_in_range': int(ape_in_range),
        'ape_above_2_2': int(ape_above_2_2),
        'accepted_rows': len(adopted_df),
        'minus_9999_cells_target': int(m9999_cells_target),
        'minus_9999_cells_full': int(m9999_cells_full),
        'other_negative_cells_target': int(other_neg_cells_target),
        'NaN_cells_target': int(nan_cells_target),
        '24_00_rows': count_2400,
        'datetime_parse_failures': failures,
        'timestamp_duplicate_rows': int(timestamp_duplicate_rows),
        'date_or_site_mismatches': int(date_mismatches + site_mismatches)
    }

    # その他の負値詳細を記録（除外はしない）
    if other_neg_cells_target > 0:
         neg_rows = df[(arr_1050 < 0).any(axis=1) & (arr_1050 != -9999).any(axis=1)]
         # 簡単のため今回は集計のみ

    if adopted_df.empty:
        output_df = pd.DataFrame(columns=['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number'])
    else:
        output_df = adopted_df[['Datetime', '地点番号', '地点名', 'remark_num', 'APE', 'source_file', 'source_row_index', 'source_line_number']].rename(columns={'地点番号': 'SiteNum', '地点名': 'SiteName', 'remark_num': 'Remark'})

    return output_df, stats

def process_batch(batch_id, relative_paths, dry_run=True):
    all_dfs = []
    all_stats = {}
    errors = 0

    for i, rel_p in enumerate(relative_paths):
        if i > 0 and i % 5 == 0:
             atomic_update_lock(f"dry_running_{batch_id}_file_{i}")

        try:
            filepath = INPUT_ROOT / rel_p
            df, stats = calculate_ape_for_file(filepath, rel_p, CONFIG)
            df['batch_id'] = batch_id
            all_dfs.append(df)
            all_stats[rel_p] = stats
        except Exception as e:
            print(f"[エラー] {rel_p}: {e}")
            traceback.print_exc(file=sys.stdout)
            errors += 1

    if all_dfs:
        final_batch_df = pd.concat(all_dfs, ignore_index=True)
        final_batch_df = final_batch_df.sort_values(by=['Datetime', 'source_file', 'source_line_number'], kind='stable', ignore_index=True)
    else:
        final_batch_df = pd.DataFrame()

    return final_batch_df, all_stats, errors

print("✅ 関数定義完了。")

# 【4-9. ファイル処理と集計検証】
print("\n【4-7 & 9. ファイル処理・統計・バッチ全体検証】")
batch_df, batch_stats, errors = process_batch(TARGET_BATCH, batch_files, dry_run=True)

total_raw = sum(s['raw_rows'] for s in batch_stats.values())
total_rem1 = sum(s['remark_1_count'] for s in batch_stats.values())
total_rem2 = sum(s['remark_2_count'] for s in batch_stats.values())
total_val1050 = sum(s['target_350_1050_valid_rows'] for s in batch_stats.values())
total_val1700 = sum(s['full_350_1700_valid_rows'] for s in batch_stats.values())
total_accepted = sum(s['accepted_rows'] for s in batch_stats.values())
total_under = sum(s['ape_below_1_2'] for s in batch_stats.values())
total_over = sum(s['ape_above_2_2'] for s in batch_stats.values())
total_in_range = sum(s['ape_in_range'] for s in batch_stats.values())

print(f"--- {TARGET_BATCH} 処理結果 ---")
print(f"入力ファイル数: {len(batch_files)}")
print(f"raw総行数: {total_raw}")
print(f"Remark 1候補数: {total_rem1}")
print(f"Remark 2件数: {total_rem2}")
print(f"350-1050nm有効数: {total_val1050}")
print(f"350-1700nm有効数: {total_val1700}")
print(f"両者の差: {total_val1050 - total_val1700}")
print(f"APE < 1.2: {total_under}")
print(f"1.2 <= APE <= 2.2: {total_in_range}")
print(f"APE > 2.2: {total_over}")
print(f"最終採用数: {total_accepted}")

# 複合キーとtimestampの重複確認
if not batch_df.empty:
    dups_keys = batch_df.duplicated(subset=['source_file', 'source_line_number']).sum()
    dups_time = batch_df.duplicated(subset=['Datetime'], keep=False).sum()
    print(f"複合由来キー重複数: {dups_keys}")
    print(f"timestamp重複行数(出力DF上): {dups_time}")

    print("\n--- APE統計 ---")
    print(batch_df['APE'].describe().to_string())
else:
    dups_keys = 0
    print("バッチ全体で採用行なし。")

print(f"ファイル処理エラー数: {errors}")
print(f"推定parquet行数: {len(batch_df)}")
print(f"推定品質JSONの要素数(ファイル単位統計): {len(batch_stats)}")

# 【8. 既存9標本試算との照合】
print("\n【8. 既存9標本試算との照合】")
if 'final_df' not in globals():
    raise RuntimeError("【処理停止】メモリ上に final_df (9標本試算結果) が存在しません。ランタイム断絶の疑いがあります。")

# dry-run結果から対象ファイルを抽出
dry_run_target = batch_df[batch_df['source_file'] == TARGET_SAMPLE_FILE].copy()

# 9標本結果から対象ファイルを抽出
ref_target = final_df[final_df['source_file'] == TARGET_SAMPLE_FILE].copy()

print(f"dry-run抽出行数: {len(dry_run_target)}")
print(f"9標本抽出行数: {len(ref_target)}")

if len(dry_run_target) != 52:
     raise ValueError(f"採用行数が52ではありません: {len(dry_run_target)}")

# 照合
merged = pd.merge(
    dry_run_target, ref_target,
    on=['source_file', 'source_line_number'],
    suffixes=('_dry', '_ref')
)

if len(merged) != 52:
    raise ValueError("source_fileとsource_line_numberのキー集合が完全一致しません。")

dt_match = (merged['Datetime_dry'] == merged['Datetime_ref']).all()
rem_match = (merged['Remark_dry'] == merged['Remark_ref']).all()
ape_diff = np.abs(merged['APE_dry'] - merged['APE_ref']).max()

print(f"Datetime完全一致: {dt_match}")
print(f"Remark完全一致: {rem_match}")
print(f"APE最大絶対差: {ape_diff:.3e} eV")

if not dt_match or not rem_match or ape_diff > 1e-12:
    raise ValueError("9標本試算結果と一致しません。")

print("✅ 既存9標本試算と完全一致しました。")

# 【10. 書き込み禁止】
print("\n【10. 書き込み禁止チェック】")
if list(BATCH_DIR.iterdir()):
    raise RuntimeError("BATCH_DIR内にファイルが作成されています。")
if list(ERROR_DIR.iterdir()):
    raise RuntimeError("ERROR_DIR内にファイルが作成されています。")
if list(TEMP_DIR.iterdir()):
    raise RuntimeError("TEMP_DIR内にファイルが作成されています。")

# 【11. 最終判定】
print("\n=========================================")
if errors == 0 and dups_keys == 0 and dt_match and rem_match and ape_diff <= 1e-12:
    print(f"{TARGET_BATCH} dry-run成功。本番用バッチ保存処理の実装へ進める")
else:
    print("失敗した条件があります。ログを確認してください。")


--- batch_00021 dry-run 処理 ---

【1. セッション確認】
✅ セッション・設定確認成功。statusを 'dry_running_batch_00021' にしました。

【2. バッチ確認】
✅ バッチ事前確認に成功しました。

【3. 本番用関数の定義】
✅ 関数定義完了。

【4-7 & 9. ファイル処理・統計・バッチ全体検証】
--- batch_00021 処理結果 ---
入力ファイル数: 25
raw総行数: 2425
Remark 1候補数: 1391
Remark 2件数: 0
350-1050nm有効数: 1917
350-1700nm有効数: 1917
両者の差: 0
APE < 1.2: 0
1.2 <= APE <= 2.2: 1391
APE > 2.2: 0
最終採用数: 1391
複合由来キー重複数: 0
timestamp重複行数(出力DF上): 0

--- APE統計 ---
count    1391.000000
mean        1.921714
std         0.019227
min         1.854402
25%         1.910092
50%         1.918436
75%         1.929657
max         2.032143
ファイル処理エラー数: 0
推定parquet行数: 1391
推定品質JSONの要素数(ファイル単位統計): 25

【8. 既存9標本試算との照合】
dry-run抽出行数: 52
9標本抽出行数: 52
Datetime完全一致: True
Remark完全一致: True
APE最大絶対差: 0.000e+00 eV
✅ 既存9標本試算と完全一致しました。

【10. 書き込み禁止チェック】

batch_00021 dry-run成功。本番用バッチ保存処理の実装へ進める


/tmp/ipykernel_1213/3518584110.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_batch_df = pd.concat(all_dfs, ignore_index=True)


In [19]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import traceback
import sys
import re
import shutil
from pathlib import Path
import collections

print("--- batch_00021 本番保存・コミット・resume_state更新処理 ---")

EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"
TARGET_BATCH = "batch_00021"
start_utc = datetime.now(timezone.utc)

# 【1. ロック更新関数】（一時ファイル自動削除禁止）
def atomic_update_lock(status_str):
    now_utc = datetime.now(timezone.utc).isoformat()
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        lock = json.load(f)
    if lock.get("session_id") != EXPECTED_SESSION_ID:
        raise RuntimeError(f"所有権が失われました。現在のsession_id: {lock.get('session_id')}")
    lock["heartbeat"] = now_utc
    lock["status"] = status_str
    temp_path = PROCESSING_LOCK_PATH.parent / f"{PROCESSING_LOCK_PATH.stem}_{EXPECTED_SESSION_ID}_hb.tmp.json"

    if temp_path.exists():
        st = temp_path.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
        raise FileExistsError(f"【処理停止】一時ファイルが既に存在します。削除・上書きを禁止しています。\nパス: {temp_path}\nサイズ: {st.st_size}\n更新日時: {mtime}")

    with open(temp_path, "x", encoding="utf-8") as f:
        json.dump(lock, f, indent=4, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    with open(temp_path, "r", encoding="utf-8") as f:
        read_back = json.load(f)
    if read_back != lock:
        raise ValueError("一時ファイルの内容が不一致です。")
    os.replace(temp_path, PROCESSING_LOCK_PATH)

# 【2. 実行前確認】
print("\n【2. 実行前確認】")
if 'session_id' not in globals() or session_id != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】メモリ上の session_id が一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state の current_session_id が一致しません。")

EXPECTED_CONFIG_HASH = "f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9"
if config_hash != EXPECTED_CONFIG_HASH or manifest_data.get("config_hash") != EXPECTED_CONFIG_HASH:
    raise ValueError("【処理停止】config_hash が一致しません。")

if TARGET_BATCH not in current_resume.get("pending_batch_ids", []):
    raise ValueError(f"【処理停止】{TARGET_BATCH} が pending_batch_ids に存在しません。")
if TARGET_BATCH in current_resume.get("completed_batch_ids", []):
    raise ValueError(f"【処理停止】{TARGET_BATCH} は completed_batch_ids に既に存在します。")

batches = manifest_data.get("batches", {})
batch_files = batches.get(TARGET_BATCH)
if not batch_files or len(batch_files) != 25:
    raise ValueError("【処理停止】manifestのファイル構成が不正です。")

file_metadata = manifest_data.get("file_metadata", {})
for rel_p in batch_files:
    full_path = INPUT_ROOT / rel_p
    st = full_path.stat()
    meta = file_metadata[rel_p]
    if st.st_size != meta["size"] or st.st_mtime_ns != meta["mtime_ns"]:
        raise ValueError(f"【処理停止】ファイル変更検出: {rel_p}")

batch_parquet_path = BATCH_DIR / f"{TARGET_BATCH}.parquet"
batch_json_path = BATCH_DIR / f"{TARGET_BATCH}_quality.json"
batch_commit_path = BATCH_DIR / f"{TARGET_BATCH}_commit.json"

if batch_parquet_path.exists(): raise FileExistsError(f"【処理停止】{batch_parquet_path.name} が存在します。")
if batch_json_path.exists(): raise FileExistsError(f"【処理停止】{batch_json_path.name} が存在します。")
if batch_commit_path.exists(): raise FileExistsError(f"【処理停止】{batch_commit_path.name} が存在します。")
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

STAGING_BATCH_ROOT = TEMP_DIR / f"{TARGET_BATCH}_{session_id}.staging"
if STAGING_BATCH_ROOT.exists():
    raise FileExistsError(f"【処理停止】ステージングディレクトリが既に存在します: {STAGING_BATCH_ROOT}")

print("✅ 実行前確認クリア")

# 【3. 本番対象ロック更新】
atomic_update_lock(f"processing_{TARGET_BATCH}")
print(f"✅ ロック更新: status = processing_{TARGET_BATCH}")

# 【4. ステージング作成】
STAGING_BATCH_ROOT.mkdir(parents=True, exist_ok=False)
staging_parquet = STAGING_BATCH_ROOT / f"{TARGET_BATCH}.parquet"
staging_json = STAGING_BATCH_ROOT / f"{TARGET_BATCH}_quality.json"
print(f"✅ ステージングディレクトリ作成: {STAGING_BATCH_ROOT}")

# 【関数定義】（修正事項適用）
EXPECTED_HEADERS_TOP5 = ["地点番号", "地点名", "年月日", "時分", "リマーク"]
WAVE_PATTERN = re.compile(r"^日射強度\((\d+)nm\)")

def parse_hsr_header(filepath):
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False, nrows=0)
    cols = list(df.columns)
    if cols[:5] != EXPECTED_HEADERS_TOP5:
        raise ValueError(f"先頭5列構造不一致: {cols[:5]}")
    if len(cols) != 1356:
        raise ValueError(f"全列数が1356ではありません: {len(cols)}")

    wave_cols = []
    for c in cols[5:]:
        m = WAVE_PATTERN.match(c)
        if not m: raise ValueError(f"波長列名異常: {c}")
        wave_cols.append({'col_name': c, 'wave': int(m.group(1))})

    waves = [info['wave'] for info in wave_cols]
    if waves != list(range(350, 1701)): raise ValueError("波長列不一致")

    target_wave_cols = [info['col_name'] for info in wave_cols if 350 <= info['wave'] <= 1050]
    full_wave_cols = [info['col_name'] for info in wave_cols if 350 <= info['wave'] <= 1700]
    return target_wave_cols, full_wave_cols

def parse_datetime_columns(df, source_file):
    parsed_times = []
    failures = 0
    count_2400 = 0
    for idx, row in df.iterrows():
        try:
            ymd = str(row['年月日']).strip()
            hm = str(row['時分']).strip()
            base_date = datetime.strptime(ymd, "%Y/%m/%d")
            if hm == "24:00":
                dt = base_date + timedelta(days=1)
                count_2400 += 1
            else:
                h, m_str = hm.split(':')
                dt = base_date.replace(hour=int(h), minute=int(m_str))
            parsed_times.append(dt)
        except Exception:
            failures += 1
            parsed_times.append(pd.NaT)
    df['Datetime'] = parsed_times
    return df, failures, count_2400

def calculate_ape_for_file(filepath, relative_path, config):
    target_wave_cols, full_wave_cols = parse_hsr_header(filepath)
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False)

    m_file = re.fullmatch(r"10HSR(\d{6})_(\d{3})\.csv", Path(filepath).name)
    if not m_file: raise ValueError(f"ファイル名不一致")
    file_yymmdd = m_file.group(1)
    file_site = m_file.group(2)

    raw_rows = len(df)
    df['source_row_index'] = df.index
    df['source_line_number'] = df.index + 2
    df['source_file'] = relative_path
    df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')

    df, failures, count_2400 = parse_datetime_columns(df, relative_path)

    site_mismatches = (df['地点番号'].astype(str) != file_site).sum()
    def check_date(row):
         try:
            base_date = datetime.strptime(str(row['年月日']).strip(), "%Y/%m/%d")
            if base_date.strftime("%y%m%d") != file_yymmdd: return 1
         except: pass
         return 0
    date_mismatches = df.apply(check_date, axis=1).sum()

    # Datetime有効行のみで重複確認
    valid_dt_mask = df['Datetime'].notna()
    timestamp_duplicate_rows = df[valid_dt_mask].duplicated(subset=['Datetime'], keep=False).sum()

    rem_counts = df['remark_num'].value_counts().to_dict()
    mask_rem = (df['remark_num'] == 1) | (df['remark_num'] == 2)

    arr_1050 = df[target_wave_cols].to_numpy(dtype=np.float64)
    mask_valid_1050 = np.isfinite(arr_1050).all(axis=1) & (arr_1050 != -9999).all(axis=1)
    m9999_cells_target = (arr_1050 == -9999).sum()

    # 修正3: 負値抽出
    other_negative_mask = ((arr_1050 < 0) & (arr_1050 != -9999))
    other_negative_cells_target = other_negative_mask.sum()
    nan_cells_target = np.isnan(arr_1050).sum()

    arr_1700 = df[full_wave_cols].to_numpy(dtype=np.float64)
    mask_valid_1700 = np.isfinite(arr_1700).all(axis=1) & (arr_1700 != -9999).all(axis=1)
    m9999_cells_full = (arr_1700 == -9999).sum()

    mask_adopted = mask_rem & valid_dt_mask & mask_valid_1050
    adopted_df = df[mask_adopted].copy()

    ape_below_1_2, ape_in_range, ape_above_2_2 = 0, 0, 0
    num_den_invalid = 0
    invalid_num_den_records = []

    if not adopted_df.empty:
        wavelength_nm = np.arange(350, 1051, dtype=np.float64)
        G_adopted = adopted_df[target_wave_cols].to_numpy(dtype=np.float64)

        numerator = np.sum(G_adopted, axis=1)
        denominator = np.sum(G_adopted * wavelength_nm[None, :], axis=1)

        valid_num_den = np.isfinite(numerator) & np.isfinite(denominator) & (numerator > 0) & (denominator > 0)
        num_den_invalid = (~valid_num_den).sum()

        # 修正4: 異常行除外と記録
        if num_den_invalid > 0:
            bad_df = adopted_df[~valid_num_den].copy()
            bad_df['num'] = numerator[~valid_num_den]
            bad_df['den'] = denominator[~valid_num_den]
            for _, r in bad_df.iterrows():
                invalid_num_den_records.append({
                    'source_file': r['source_file'],
                    'source_line_number': int(r['source_line_number']),
                    'Remark': int(r['remark_num']),
                    'numerator': float(r['num']),
                    'denominator': float(r['den']),
                    '異常理由': 'numerator or denominator is <= 0 or not finite'
                })
            adopted_df = adopted_df[valid_num_den].copy()
            numerator = numerator[valid_num_den]
            denominator = denominator[valid_num_den]

        if not adopted_df.empty:
            APE = 1239.84193 * numerator / denominator
            adopted_df['APE'] = APE

            APE_check = 1239.84193 / (denominator / numerator)
            abs_diff = np.abs(APE - APE_check)
            if abs_diff.max() > 1e-12:
                raise ValueError(f"APE独立式差が1e-12 eV超過: {abs_diff.max():.3e}")

            ape_below_1_2 = (APE < 1.2).sum()
            ape_in_range = ((APE >= 1.2) & (APE <= 2.2)).sum()
            ape_above_2_2 = (APE > 2.2).sum()
            adopted_df = adopted_df[(adopted_df['APE'] >= 1.2) & (adopted_df['APE'] <= 2.2)]

    stats = {
        'raw_rows': raw_rows,
        'remark_counts': {int(k): int(v) for k, v in rem_counts.items()},
        'remark_1_count': int(rem_counts.get(1, 0)),
        'remark_2_count': int(rem_counts.get(2, 0)),
        'target_350_1050_valid_rows': int(mask_valid_1050.sum()),
        'full_350_1700_valid_rows': int(mask_valid_1700.sum()),
        'target_valid_but_full_invalid_rows': int((mask_valid_1050 & ~mask_valid_1700).sum()),
        'remark_valid_but_target_invalid_rows': int((mask_rem & ~mask_valid_1050).sum()),
        'numerator_or_denominator_invalid_rows': int(num_den_invalid),
        'invalid_num_den_records': invalid_num_den_records,
        'ape_below_1_2': int(ape_below_1_2),
        'ape_in_range': int(ape_in_range),
        'ape_above_2_2': int(ape_above_2_2),
        'accepted_rows': len(adopted_df),
        'minus_9999_cells_target': int(m9999_cells_target),
        'minus_9999_cells_full': int(m9999_cells_full),
        'other_negative_cells_target': int(other_negative_cells_target),
        'NaN_cells_target': int(nan_cells_target),
        '24_00_rows': count_2400,
        'datetime_parse_failures': failures,
        'timestamp_duplicate_rows': int(timestamp_duplicate_rows),
        'date_or_site_mismatches': int(date_mismatches + site_mismatches)
    }

    # 修正5: 1つでもあればエラー
    if failures > 0 or date_mismatches > 0 or site_mismatches > 0 or df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
         raise ValueError(f"必須条件を満たさない行が存在します (failures={failures}, mismatches={date_mismatches+site_mismatches}, dup_keys={df.duplicated(subset=['source_file', 'source_line_number']).sum()})")

    if adopted_df.empty:
        output_df = pd.DataFrame(columns=['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number'])
    else:
        output_df = adopted_df[['Datetime', '地点番号', '地点名', 'remark_num', 'APE', 'source_file', 'source_row_index', 'source_line_number']].rename(columns={'地点番号': 'SiteNum', '地点名': 'SiteName', 'remark_num': 'Remark'})

    return output_df, stats

def process_batch(batch_id, relative_paths):
    all_dfs = []
    all_stats = {}
    errors = 0

    for i, rel_p in enumerate(relative_paths):
        if i > 0 and i % 5 == 0:
             atomic_update_lock(f"processing_{batch_id}_file_{i}")

        try:
            filepath = INPUT_ROOT / rel_p
            df, stats = calculate_ape_for_file(filepath, rel_p, CONFIG)
            df['batch_id'] = batch_id
            # 修正2: FutureWarning対応、空でも追加しないが統計は残す（既に空DFを考慮したconcat対応を行う）
            all_dfs.append(df)
            all_stats[rel_p] = stats
        except Exception as e:
            print(f"[エラー] {rel_p}: {e}")
            traceback.print_exc(file=sys.stdout)
            errors += 1

    # 修正2: 空DF対応
    valid_dfs = [d for d in all_dfs if not d.empty]
    if valid_dfs:
        final_batch_df = pd.concat(valid_dfs, ignore_index=True)
        final_batch_df = final_batch_df.sort_values(by=['Datetime', 'source_file', 'source_line_number'], kind='stable', ignore_index=True)
    else:
        final_batch_df = pd.DataFrame(columns=['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id'])

    return final_batch_df, all_stats, errors

# 【バッチ処理実行】
print("\n【バッチ処理実行】")
batch_df, batch_stats, errors = process_batch(TARGET_BATCH, batch_files)
if errors > 0:
    raise RuntimeError(f"【処理停止】ファイル処理失敗が {errors} 件あります。バッチ成果物を作成しません。")

# 【5. Parquet出力】
print("\n【5. Parquet出力】")
parquet_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']
if not batch_df.empty:
    batch_df = batch_df[parquet_cols]
    batch_df['Datetime'] = pd.to_datetime(batch_df['Datetime'])
    batch_df['SiteNum'] = batch_df['SiteNum'].astype(int)
    batch_df['SiteName'] = batch_df['SiteName'].astype(str)
    batch_df['Remark'] = batch_df['Remark'].astype(int)
    batch_df['APE'] = batch_df['APE'].astype('float64')
    batch_df['source_file'] = batch_df['source_file'].astype(str)
    batch_df['source_row_index'] = batch_df['source_row_index'].astype(int)
    batch_df['source_line_number'] = batch_df['source_line_number'].astype(int)
    batch_df['batch_id'] = batch_df['batch_id'].astype(str)

    batch_df.to_parquet(staging_parquet, index=False)

    with open(staging_parquet, "rb") as f:
         f.flush()
         os.fsync(f.fileno())
else:
     pd.DataFrame(columns=parquet_cols).to_parquet(staging_parquet, index=False)

# 再読込・検証
loaded_df = pd.read_parquet(staging_parquet)
if len(loaded_df) != 1391:
    raise ValueError(f"Parquet行数不一致: {len(loaded_df)}")
if list(loaded_df.columns) != parquet_cols:
    raise ValueError("Parquet列不一致")
if loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
    raise ValueError("Parquet内で複合由来キー重複あり")
if not (loaded_df['batch_id'] == TARGET_BATCH).all():
    raise ValueError("batch_idが不一致")
ape_diff = np.abs(loaded_df['APE'] - batch_df['APE']).max()
if ape_diff > 0:
    raise ValueError(f"APE最大絶対差異常: {ape_diff}")
if loaded_df['APE'].isna().sum() > 0 or np.isinf(loaded_df['APE']).sum() > 0:
    raise ValueError("NaN/inf APEが存在")
if (loaded_df['APE'] < 1.2).sum() > 0 or (loaded_df['APE'] > 2.2).sum() > 0:
    raise ValueError("APE範囲外が存在")

with open(staging_parquet, "rb") as f:
    pq_sha256 = hashlib.sha256(f.read()).hexdigest()

# 【6. Quality JSON】
print("\n【6. Quality JSON出力】")
end_utc = datetime.now(timezone.utc).isoformat()

def get_stats_sum(key):
    return sum(s.get(key, 0) for s in batch_stats.values())

ape_desc = loaded_df['APE'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict() if not loaded_df.empty else {}

quality_data = {
    "quality_schema_version": "1.0",
    "pipeline_version": CONFIG["pipeline_version"],
    "config_hash": config_hash,
    "session_id": session_id,
    "batch_id": TARGET_BATCH,
    "started_at": start_utc.isoformat(),
    "completed_at": end_utc,
    "input_file_count": len(batch_files),
    "input_files": batch_files,
    "file_metadata": {p: {"size": file_metadata[p]["size"], "mtime_ns": file_metadata[p]["mtime_ns"]} for p in batch_files},
    "raw_rows": get_stats_sum('raw_rows'),
    "remark_counts": {str(k): v for k, v in sum((collections.Counter(s.get('remark_counts', {})) for s in batch_stats.values()), collections.Counter()).items()},
    "remark_1_count": get_stats_sum('remark_1_count'),
    "remark_2_count": get_stats_sum('remark_2_count'),
    "target_350_1050_valid_rows": get_stats_sum('target_350_1050_valid_rows'),
    "full_350_1700_valid_rows": get_stats_sum('full_350_1700_valid_rows'),
    "target_valid_but_full_invalid_rows": get_stats_sum('target_valid_but_full_invalid_rows'),
    "remark_valid_but_target_invalid_rows": get_stats_sum('remark_valid_but_target_invalid_rows'),
    "numerator_or_denominator_invalid_rows": get_stats_sum('numerator_or_denominator_invalid_rows'),
    "ape_below_1_2": get_stats_sum('ape_below_1_2'),
    "ape_in_range": get_stats_sum('ape_in_range'),
    "ape_above_2_2": get_stats_sum('ape_above_2_2'),
    "accepted_rows": len(loaded_df),
    "minus_9999_cells_target": get_stats_sum('minus_9999_cells_target'),
    "minus_9999_cells_full": get_stats_sum('minus_9999_cells_full'),
    "other_negative_cells_target": get_stats_sum('other_negative_cells_target'),
    "NaN_cells_target": get_stats_sum('NaN_cells_target'),
    "24_00_rows": get_stats_sum('24_00_rows'),
    "datetime_parse_failures": get_stats_sum('datetime_parse_failures'),
    "timestamp_duplicate_rows": get_stats_sum('timestamp_duplicate_rows'),
    "source_key_duplicate_rows": int(loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum()),
    "date_or_site_mismatches": get_stats_sum('date_or_site_mismatches'),
    "processing_error_count": errors,
    "file_statistics": batch_stats,
    "ape_statistics": ape_desc,
    "parquet_schema": {col: str(dtype) for col, dtype in loaded_df.dtypes.items()},
    "parquet_sha256": pq_sha256
}

temp_q_json = STAGING_BATCH_ROOT / "temp_q.json"
with open(temp_q_json, "x", encoding="utf-8") as f:
    json.dump(quality_data, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())
with open(temp_q_json, "r", encoding="utf-8") as f:
    read_q = json.load(f)
if read_q != quality_data:
    raise ValueError("Quality JSONの保存に失敗")
os.replace(temp_q_json, staging_json)

# 【7. ステージング全体の検証】
print("\n【7. ステージング全体の検証】")
with open(staging_json, "r", encoding="utf-8") as f:
    v_q = json.load(f)
v_df = pd.read_parquet(staging_parquet)

if v_q["processing_error_count"] != 0: raise ValueError("processing_error_count != 0")
if v_q["input_file_count"] != 25: raise ValueError("input_file_count != 25")
if v_q["accepted_rows"] != 1391: raise ValueError("accepted_rows != 1391")
if len(v_df) != 1391: raise ValueError("Parquet行数が一致しない")
if v_q["parquet_sha256"] != pq_sha256: raise ValueError("SHA256不一致")
if v_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0: raise ValueError("キー重複")
unexpected = [f.name for f in STAGING_BATCH_ROOT.iterdir() if f.name not in [staging_parquet.name, staging_json.name]]
if unexpected: raise ValueError(f"予定外ファイル存在: {unexpected}")

# dry-run比較
merged_dry = pd.merge(v_df[v_df['source_file'] == TARGET_SAMPLE_FILE], dry_run_target, on=['source_file', 'source_line_number'], suffixes=('_st', '_dry'))
if len(merged_dry) != 52: raise ValueError("dry-runキー集合不一致")
if np.abs(merged_dry['APE_st'] - merged_dry['APE_dry']).max() > 1e-12: raise ValueError("dry-run APE不一致")
print("✅ ステージング検証通過")

# 【8. 正式公開】
print("\n【8. 正式公開】")
if batch_parquet_path.exists() or batch_json_path.exists():
    raise FileExistsError("公開直前に同名ファイル出現")

os.replace(staging_parquet, batch_parquet_path)
os.replace(staging_json, batch_json_path)

v_df_final = pd.read_parquet(batch_parquet_path)
with open(batch_json_path, "r", encoding="utf-8") as f:
    v_q_final = json.load(f)
with open(batch_parquet_path, "rb") as f:
    final_pq_sha = hashlib.sha256(f.read()).hexdigest()
if final_pq_sha != pq_sha256 or v_q_final != quality_data:
    raise ValueError("正式公開後の検証失敗")
print("✅ 正式公開成功")

# 【9. コミットマーカー】
print("\n【9. コミットマーカー】")
commit_data = {
    "commit_schema_version": "1.0",
    "batch_id": TARGET_BATCH,
    "config_hash": config_hash,
    "session_id": session_id,
    "committed_at": datetime.now(timezone.utc).isoformat(),
    "parquet_filename": batch_parquet_path.name,
    "parquet_sha256": pq_sha256,
    "quality_filename": batch_json_path.name,
    "quality_sha256": hashlib.sha256(json.dumps(quality_data, ensure_ascii=False).encode('utf-8')).hexdigest(),
    "accepted_rows": 1391,
    "input_file_count": 25
}
with open(batch_commit_path, "x", encoding="utf-8") as f:
    json.dump(commit_data, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())
print("✅ コミットマーカー作成")

# 【10. resume_state更新】
print("\n【10. resume_state更新】")
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs = json.load(f)
if rs["current_session_id"] != session_id or rs["config_hash"] != config_hash:
    raise ValueError("resume_state更新前検証失敗")

rs["status"] = "running"
if TARGET_BATCH not in rs["completed_batch_ids"]:
    rs["completed_batch_ids"].append(TARGET_BATCH)
    # 元のmanifest順序でソート
    rs["completed_batch_ids"] = [b for b in list(batches.keys()) if b in rs["completed_batch_ids"]]
if TARGET_BATCH in rs["pending_batch_ids"]:
    rs["pending_batch_ids"].remove(TARGET_BATCH)

for f_path in batch_files:
    if f_path not in rs["completed_source_files"]:
        rs["completed_source_files"].append(f_path)
rs["completed_source_files"] = [f for b in rs["completed_batch_ids"] for f in batches[b]]

rs["completed_batches"] = len(rs["completed_batch_ids"])
rs["completed_files"] = len(rs["completed_source_files"])
rs["last_completed_batch"] = TARGET_BATCH
rs["last_completed_source_file"] = batch_files[-1]
rs["last_update"] = datetime.now(timezone.utc).isoformat()
rs["completed_output_rows"] = rs.get("completed_output_rows", 0) + 1391

temp_rs = RESUME_STATE_PATH.parent / f"resume_state_{session_id}.tmp.json"
with open(temp_rs, "x", encoding="utf-8") as f:
    json.dump(rs, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())
with open(temp_rs, "r", encoding="utf-8") as f:
    if json.load(f) != rs: raise ValueError("resume_state不一致")
os.replace(temp_rs, RESUME_STATE_PATH)
print("✅ resume_state更新")

# 【12. 成功後クリーンアップ】
if not any(STAGING_BATCH_ROOT.iterdir()):
    STAGING_BATCH_ROOT.rmdir()

atomic_update_lock(f"{TARGET_BATCH}_committed")

# 【13. 最終表示】
print("\n=========================================")
print(f"batch_id: {TARGET_BATCH}")
print(f"入力ファイル数: 25")
print(f"raw行数: {quality_data['raw_rows']}")
print(f"採用行数: 1391")
print(f"Remark 1件数: {quality_data['remark_1_count']}")
print(f"Remark 2件数: {quality_data['remark_2_count']}")
print(f"APE統計: \n{pd.Series(ape_desc).to_string()}")
print(f"Parquet: {batch_parquet_path.name} (Size: {batch_parquet_path.stat().st_size}, SHA256: {pq_sha256})")
print(f"quality JSON: {batch_json_path.name} (Size: {batch_json_path.stat().st_size}, SHA256: {commit_data['quality_sha256']})")
print(f"commit JSON: {batch_commit_path.name} (Size: {batch_commit_path.stat().st_size})")
print(f"completed_batches: {rs['completed_batches']}")
print(f"pending batch数: {len(rs['pending_batch_ids'])}")
print(f"completed_files: {rs['completed_files']}")
print(f"completed_output_rows: {rs['completed_output_rows']}")
print(f"ステージング残存有無: {STAGING_BATCH_ROOT.exists()}")
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    print(f"processing_lock status: {json.load(f)['status']}")
print("resume_state検証結果: 成功")
print("\nbatch_00021本番コミット成功。再開整合性監査へ進める")


--- batch_00021 本番保存・コミット・resume_state更新処理 ---

【2. 実行前確認】
✅ 実行前確認クリア
✅ ロック更新: status = processing_batch_00021
✅ ステージングディレクトリ作成: /content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/temporary/batch_00021_152d121e-2e5f-445f-88e6-e644e5829f2d.staging

【バッチ処理実行】

【5. Parquet出力】

【6. Quality JSON出力】


ValueError: Quality JSONの保存に失敗

In [22]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import traceback
import sys

print("--- batch_00021 失敗後・読み取り中心の回復事前監査 ---")

EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"
TARGET_BATCH = "batch_00021"

# 【1. 同一ランタイム確認】
print("\n【1. 同一ランタイム確認】")
required_vars = ["session_id", "quality_data", "batch_df", "batch_stats", "dry_run_target", "STAGING_BATCH_ROOT"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"【処理停止】メモリ上に必須変数が存在しません: {missing_vars}")

if session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"【処理停止】メモリ上の session_id が {EXPECTED_SESSION_ID} と一致しません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state の current_session_id が一致しません。")

print("✅ 同一ランタイム、完全なメモリ・所有権を確認しました。")

# heartbeatとstatusの更新
now_utc = datetime.now(timezone.utc).isoformat()
current_lock["heartbeat"] = now_utc
current_lock["status"] = "auditing_failed_batch_00021_staging"
temp_path = PROCESSING_LOCK_PATH.parent / f"{PROCESSING_LOCK_PATH.stem}_{EXPECTED_SESSION_ID}_hb.tmp.json"
if temp_path.exists():
    raise FileExistsError(f"一時ファイルが存在します: {temp_path}")
with open(temp_path, "x", encoding="utf-8") as f:
    json.dump(current_lock, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())
with open(temp_path, "r", encoding="utf-8") as f:
    if json.load(f) != current_lock: raise ValueError("ロック一時ファイル不一致")
os.replace(temp_path, PROCESSING_LOCK_PATH)
print("✅ ロックのheartbeatを更新し、statusを auditing_failed_batch_00021_staging としました。")

# 【2. 正式成果物の確認】
print("\n【2. 正式成果物の確認】")
batch_parquet_path = BATCH_DIR / f"{TARGET_BATCH}.parquet"
batch_json_path = BATCH_DIR / f"{TARGET_BATCH}_quality.json"
batch_commit_path = BATCH_DIR / f"{TARGET_BATCH}_commit.json"

paths_to_check = [
    ("Parquet", batch_parquet_path),
    ("Quality JSON", batch_json_path),
    ("Commit JSON", batch_commit_path)
]

for name, p in paths_to_check:
    print(f"- {name}: {'存在する' if p.exists() else '存在しない'}")

if any(p.exists() for _, p in paths_to_check):
    raise RuntimeError("【処理停止】正式成果物が一部存在しています。部分公開の可能性があるため停止します。")

# 【3. resume_state確認】
print("\n【3. resume_state確認】")
pending = current_resume.get("pending_batch_ids", [])
completed = current_resume.get("completed_batch_ids", [])
comp_batches = current_resume.get("completed_batches")
comp_files = current_resume.get("completed_files")
comp_rows = current_resume.get("completed_output_rows", 0)
comp_src = current_resume.get("completed_source_files", [])
status = current_resume.get("status")

print(f"- {TARGET_BATCH} in pending: {TARGET_BATCH in pending}")
print(f"- {TARGET_BATCH} in completed: {TARGET_BATCH in completed}")
print(f"- completed_batches: {comp_batches}")
print(f"- completed_files: {comp_files}")
print(f"- completed_output_rows: {comp_rows}")

# manifestデータ取得
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)
batches = manifest_data.get("batches", {})
batch_files = batches.get(TARGET_BATCH, [])
files_in_src = [f for f in batch_files if f in comp_src]
print(f"- 対象ファイルがcompleted_source_filesに含まれる数: {len(files_in_src)} / {len(batch_files)}")
print(f"- status: {status}")

if TARGET_BATCH not in pending or TARGET_BATCH in completed or comp_batches != 0 or comp_files != 0 or comp_rows != 0 or len(files_in_src) > 0 or status not in ["running", "initialized"]:
    raise RuntimeError("【処理停止】resume_state が予期せぬ状態です。")

# 【4. ステージング内容確認】
print("\n【4. ステージング内容確認】")
print(f"ステージングディレクトリ: {STAGING_BATCH_ROOT}")
if not STAGING_BATCH_ROOT.exists():
    raise RuntimeError("ステージングディレクトリが存在しません。")

staging_parquet = STAGING_BATCH_ROOT / f"{TARGET_BATCH}.parquet"
staging_json = STAGING_BATCH_ROOT / "temp_q.json"

found_files = []
for p in STAGING_BATCH_ROOT.iterdir():
    st = p.stat()
    mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
    sha = "(ディレクトリ)" if p.is_dir() else hashlib.sha256(p.read_bytes()).hexdigest()
    print(f"  - {p.name}: Size={st.st_size}, Mtime={mtime}, SHA256={sha[:12]}...")
    found_files.append(p.name)

expected_files = {f"{TARGET_BATCH}.parquet", "temp_q.json"}
unexpected = set(found_files) - expected_files
missing = expected_files - set(found_files)

if unexpected or missing:
    raise RuntimeError(f"【処理停止】ステージング内のファイル構成が想定外です。予期せぬファイル: {unexpected}, 不足ファイル: {missing}")

# 【5. ステージングParquet検証】
print("\n【5. ステージングParquet検証】")
st_df = pd.read_parquet(staging_parquet)
pq_sha256 = hashlib.sha256(staging_parquet.read_bytes()).hexdigest()

print(f"- 行数: {len(st_df)}")
print(f"- 列数: {len(st_df.columns)}")
print(f"- 列名: {list(st_df.columns)}")
print(f"- batch_id一覧: {st_df['batch_id'].unique()}")
print(f"- APE NaN数: {st_df['APE'].isna().sum()}")
print(f"- APE inf数: {np.isinf(st_df['APE']).sum()}")
print(f"- APE < 1.2: {(st_df['APE'] < 1.2).sum()}")
print(f"- APE > 2.2: {(st_df['APE'] > 2.2).sum()}")
print(f"- 複合キー重複: {st_df.duplicated(subset=['source_file', 'source_line_number']).sum()}")
print(f"- Parquet実ファイルSHA-256: {pq_sha256}")
print("\n[APE統計]")
print(st_df['APE'].describe().to_string())

if len(st_df) != 1391 or len(st_df.columns) != 9 or list(st_df.columns) != ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id'] or st_df['batch_id'].nunique() != 1 or st_df['batch_id'].iloc[0] != TARGET_BATCH or st_df['APE'].isna().sum() > 0 or np.isinf(st_df['APE']).sum() > 0 or (st_df['APE'] < 1.2).sum() > 0 or (st_df['APE'] > 2.2).sum() > 0 or st_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
    raise RuntimeError("【処理停止】Parquetの内容が要件を満たしません。")

print("\n[メモリ上 batch_df との比較]")
if len(st_df) != len(batch_df):
    raise ValueError("行数が一致しません。")
merged = pd.merge(st_df, batch_df, on=['source_file', 'source_line_number'], suffixes=('_st', '_mem'))
if len(merged) != 1391:
    raise ValueError("キー集合が一致しません。")
if not (merged['Datetime_st'] == merged['Datetime_mem']).all():
    raise ValueError("Datetimeが一致しません。")
if not (merged['Remark_st'] == merged['Remark_mem']).all():
    raise ValueError("Remarkが一致しません。")
ape_diff = np.abs(merged['APE_st'] - merged['APE_mem']).max()
print(f"- APE最大絶対差: {ape_diff:.3e}")
if ape_diff > 1e-12:
    raise ValueError("APEの差が許容値を超えました。")

# 【6. temp_q.json検証】
print("\n【6. temp_q.json検証】")
with open(staging_json, "r", encoding="utf-8") as f:
    st_q = json.load(f)

checks_q = {
    "schema_version": st_q.get("quality_schema_version") == "1.0",
    "config_hash": st_q.get("config_hash") == config_hash,
    "session_id": st_q.get("session_id") == session_id,
    "batch_id": st_q.get("batch_id") == TARGET_BATCH,
    "input_file_count": st_q.get("input_file_count") == 25,
    "raw_rows": st_q.get("raw_rows") == 2425,
    "accepted_rows": st_q.get("accepted_rows") == 1391,
    "processing_error_count": st_q.get("processing_error_count") == 0,
    "parquet_sha256": st_q.get("parquet_sha256") == pq_sha256,
    "file_stats_count": len(st_q.get("file_statistics", {})) == 25,
    "has_ape_stats": bool(st_q.get("ape_statistics")),
    "mismatches": st_q.get("date_or_site_mismatches") == 0,
    "key_dups": st_q.get("source_key_duplicate_rows") == 0
}

for k, v in checks_q.items():
    print(f"- {k}: {v}")
if not all(checks_q.values()):
    raise RuntimeError("【処理停止】temp_q.jsonの検証に失敗しました。")

# 【7. JSON正規化による原因確認】
print("\n【7. JSON正規化による原因確認】")
normalized_quality_data = json.loads(json.dumps(quality_data, ensure_ascii=False, allow_nan=False))

def deep_diff(d1, d2, path=""):
    if isinstance(d1, dict) and isinstance(d2, dict):
        keys = set(list(d1.keys()) + list(d2.keys()))
        for k in keys:
            if k not in d1:
                return f"{path}[{repr(k)}]", None, type(None), d2[k], type(d2[k])
            if k not in d2:
                return f"{path}[{repr(k)}]", d1[k], type(d1[k]), None, type(None)
            diff = deep_diff(d1[k], d2[k], path + f"[{repr(k)}]")
            if diff: return diff
    elif isinstance(d1, list) and isinstance(d2, list):
        if len(d1) != len(d2):
            return f"{path}[len]", len(d1), type(len(d1)), len(d2), type(len(d2))
        for i, (v1, v2) in enumerate(zip(d1, d2)):
            diff = deep_diff(v1, v2, path + f"[{i}]")
            if diff: return diff
    else:
        if d1 != d2:
            return path, d1, type(d1), d2, type(d2)
    return None

diff_result = deep_diff(st_q, normalized_quality_data)
if not diff_result:
    print("✅ 失敗原因はJSONオブジェクトキーの文字列化であり、データ内容の欠損ではない")
    audit_passed = True
else:
    print("❌ 一致しませんでした。")
    p, v1, t1, v2, t2 = diff_result
    print(f"  - 辞書パス: {p}")
    print(f"  - temp_q.json再読込値: {v1} (型: {t1})")
    print(f"  - Python上の正規化値: {v2} (型: {t2})")
    audit_passed = False

# 【8, 9, 10. メッセージ】
print("\n【8. quality JSON SHAの注意】")
print("次工程では必ず正式なquality JSONファイルの実バイト列からSHA-256を計算します。")

print("\n【9. commitマーカーの注意】")
print("次工程のcommitマーカーは最終パスへ直接open(\"x\")せず、同一ディレクトリ内の一時JSON → flush → fsync → 再読込 → os.replace の順でアトミックに作成します。")

print("\n【10. 書き込み禁止】")
print("このセルでは processing_lock の更新のみを行いました。ステージングの変更、ファイルの公開、resume_stateの更新は一切行っていません。")

# 【11. 最終判定】
print("\n=========================================")
if audit_passed:
    print("batch_00021ステージングは回復可能。JSON正規化後の正式公開セルへ進める")
else:
    print("ステージングの回復可否は未確定")
    raise RuntimeError("JSONの内容に予期せぬ相違があります。")


--- batch_00021 失敗後・読み取り中心の回復事前監査 ---

【1. 同一ランタイム確認】
✅ 同一ランタイム、完全なメモリ・所有権を確認しました。
✅ ロックのheartbeatを更新し、statusを auditing_failed_batch_00021_staging としました。

【2. 正式成果物の確認】
- Parquet: 存在しない
- Quality JSON: 存在しない
- Commit JSON: 存在しない

【3. resume_state確認】
- batch_00021 in pending: True
- batch_00021 in completed: False
- completed_batches: 0
- completed_files: 0
- completed_output_rows: 0
- 対象ファイルがcompleted_source_filesに含まれる数: 0 / 25
- status: running

【4. ステージング内容確認】
ステージングディレクトリ: /content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/temporary/batch_00021_152d121e-2e5f-445f-88e6-e644e5829f2d.staging
  - batch_00021.parquet: Size=32621, Mtime=2026-07-30T19:17:08+00:00, SHA256=0804e077ced3...
  - temp_q.json: Size=32361, Mtime=2026-07-30T19:17:08+00:00, SHA256=aa059d82b0ab...

【5. ステージングParquet検証】
- 行数: 1391
- 列数: 9
- 列名: ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']
- batch_id一覧: ['batch_00021']
- APE 

In [25]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import sys

print("--- batch_00021 回復・正式公開・commit・resume_state更新 ---")

EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"
TARGET_BATCH = "batch_00021"

# 【1. 同一ランタイム・監査結果確認】
print("\n【1. 同一ランタイム・監査結果確認】")
required_vars = ["session_id", "audit_passed", "STAGING_BATCH_ROOT", "st_df", "st_q",
                 "pq_sha256", "manifest_data", "batches", "batch_files"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"【処理停止】必須変数がメモリ上に存在しません: {missing_vars}")

if session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"【処理停止】session_id 不一致: {session_id}")
if not audit_passed:
    raise RuntimeError("【処理停止】audit_passed が True ではありません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    current_lock = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

if current_lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】processing_lock の session_id が一致しません。")
if current_resume.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】resume_state の current_session_id が一致しません。")

EXPECTED_PQ_SHA256 = "0804e077ced353dff0e4077e08d2684b5d52ceb4ae9f994244fc41a8dd679cdc"
if pq_sha256 != EXPECTED_PQ_SHA256:
    raise RuntimeError(f"【処理停止】pq_sha256 が一致しません: {pq_sha256}")

if len(st_df) != 1391:
    raise RuntimeError(f"【処理停止】st_df の行数が 1391 ではありません: {len(st_df)}")
if st_q.get("accepted_rows") != 1391:
    raise RuntimeError("【処理停止】st_q['accepted_rows'] が 1391 ではありません。")
if st_q.get("processing_error_count") != 0:
    raise RuntimeError("【処理停止】st_q['processing_error_count'] が 0 ではありません。")
if st_q.get("parquet_sha256") != EXPECTED_PQ_SHA256:
    raise RuntimeError("【処理停止】st_q['parquet_sha256'] が一致しません。")

print("✅ メモリ上変数の検証完了")

def atomic_update_lock(status_str):
    now_utc = datetime.now(timezone.utc).isoformat()
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        lock = json.load(f)
    if lock.get("session_id") != EXPECTED_SESSION_ID:
        raise RuntimeError("所有権が失われました。")
    lock["heartbeat"] = now_utc
    lock["status"] = status_str
    temp_path = PROCESSING_LOCK_PATH.parent / f"{PROCESSING_LOCK_PATH.stem}_{EXPECTED_SESSION_ID}_hb.tmp.json"
    if temp_path.exists():
        raise FileExistsError(f"一時ロックファイルが存在します: {temp_path}")
    with open(temp_path, "x", encoding="utf-8") as f:
        json.dump(lock, f, indent=4, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    with open(temp_path, "r", encoding="utf-8") as f:
        if json.load(f) != lock:
            raise ValueError("ロック一時ファイル不一致")
    os.replace(temp_path, PROCESSING_LOCK_PATH)

atomic_update_lock("recovering_batch_00021_publish")
print("✅ ロック status = recovering_batch_00021_publish に更新")

# 【2. 現在状態の再確認】
print("\n【2. 現在状態の再確認】")
batch_parquet_path = BATCH_DIR / f"{TARGET_BATCH}.parquet"
batch_json_path = BATCH_DIR / f"{TARGET_BATCH}_quality.json"
batch_commit_path = BATCH_DIR / f"{TARGET_BATCH}_commit.json"

if batch_parquet_path.exists() or batch_json_path.exists() or batch_commit_path.exists():
    raise FileExistsError("【処理停止】正式パスにファイルが既に存在します。")

pending = current_resume.get("pending_batch_ids", [])
completed = current_resume.get("completed_batch_ids", [])
if TARGET_BATCH not in pending:
    raise RuntimeError(f"【処理停止】{TARGET_BATCH} が pending にありません。")
if TARGET_BATCH in completed:
    raise RuntimeError(f"【処理停止】{TARGET_BATCH} が completed にあります。")

completed_batches_before = current_resume.get("completed_batches", 0)
completed_files_before = current_resume.get("completed_files", 0)
completed_output_rows_before = current_resume.get("completed_output_rows", 0)

print(f"公開前 completed_batches: {completed_batches_before}")
print(f"公開前 completed_files: {completed_files_before}")
print(f"公開前 completed_output_rows: {completed_output_rows_before}")

if (
    completed_batches_before != 0
    or completed_files_before != 0
    or completed_output_rows_before != 0
):
    raise RuntimeError(
        "【処理停止】公開前の完了数が0ではありません。"
        f" completed_batches={completed_batches_before},"
        f" completed_files={completed_files_before},"
        f" completed_output_rows={completed_output_rows_before}"
    )

comp_src = current_resume.get("completed_source_files", [])
if any(f in comp_src for f in batch_files):
    raise RuntimeError("【処理停止】対象ファイルの一部が completed_source_files に含まれています。")

staging_files = [p.name for p in STAGING_BATCH_ROOT.iterdir()]
expected_staging_files = {f"{TARGET_BATCH}.parquet", "temp_q.json"}
if set(staging_files) != expected_staging_files:
    raise RuntimeError(f"【処理停止】ステージング内のファイル構成が不一致: {staging_files}")
print("✅ 現在状態の再確認完了")

# 【3. Quality JSONの正式ステージング名】
print("\n【3. Quality JSONの正式ステージング名】")
staging_parquet = STAGING_BATCH_ROOT / f"{TARGET_BATCH}.parquet"
staging_temp_quality = STAGING_BATCH_ROOT / "temp_q.json"
staging_quality = STAGING_BATCH_ROOT / f"{TARGET_BATCH}_quality.json"

if staging_quality.exists():
    raise FileExistsError("【処理停止】staging_quality が既に存在します。")

os.replace(staging_temp_quality, staging_quality)

with open(staging_quality, "r", encoding="utf-8") as f:
    st_q_reloaded = json.load(f)
if st_q_reloaded != st_q:
    raise ValueError("【処理停止】名称変更後の Quality JSON の内容が st_q と一致しません。")
print("✅ temp_q.json を正式なステージング名に変更し内容を確認")

# 【4. 実ファイルSHA-256】
print("\n【4. 実ファイルSHA-256】")
def get_file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

real_pq_sha = get_file_sha256(staging_parquet)
real_q_sha = get_file_sha256(staging_quality)

if real_pq_sha != EXPECTED_PQ_SHA256:
    raise ValueError(f"【処理停止】Parquetの実SHA-256が不一致: {real_pq_sha}")
print(f"✅ 実Parquet SHA-256確認: {real_pq_sha}")
print(f"✅ 実Quality JSON SHA-256計算: {real_q_sha}")

# 【5. 公開直前の最終検証】
print("\n【5. 公開直前の最終検証】")
# Parquet
v_df = pd.read_parquet(staging_parquet)
if len(v_df) != 1391 or len(v_df.columns) != 9:
    raise ValueError("Parquet形状不一致")
expected_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']
if list(v_df.columns) != expected_cols:
    raise ValueError("Parquet列名不一致")
if list(v_df['batch_id'].unique()) != [TARGET_BATCH]:
    raise ValueError("batch_id不一致")
if v_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
    raise ValueError("複合キー重複あり")
if v_df['APE'].isna().sum() > 0 or np.isinf(v_df['APE']).sum() > 0:
    raise ValueError("APE NaN/infあり")
if (v_df['APE'] < 1.2).sum() > 0 or (v_df['APE'] > 2.2).sum() > 0:
    raise ValueError("APE範囲外あり")
if st_q_reloaded['parquet_sha256'] != real_pq_sha:
    raise ValueError("Quality JSON内のparquet_sha256不一致")

# Quality JSON
checks_q = {
    "config_hash": st_q_reloaded["config_hash"] == config_hash,
    "session_id": st_q_reloaded["session_id"] == session_id,
    "batch_id": st_q_reloaded["batch_id"] == TARGET_BATCH,
    "input_file_count": st_q_reloaded["input_file_count"] == 25,
    "raw_rows": st_q_reloaded["raw_rows"] == 2425,
    "accepted_rows": st_q_reloaded["accepted_rows"] == 1391,
    "processing_error_count": st_q_reloaded["processing_error_count"] == 0,
    "file_statistics_count": len(st_q_reloaded["file_statistics"]) == 25,
    "mismatches": st_q_reloaded["date_or_site_mismatches"] == 0,
    "source_key_duplicate_rows": st_q_reloaded["source_key_duplicate_rows"] == 0
}
if not all(checks_q.values()):
    raise ValueError(f"Quality JSON再検証失敗: {checks_q}")

staging_files_now = [p.name for p in STAGING_BATCH_ROOT.iterdir()]
expected_now = {f"{TARGET_BATCH}.parquet", f"{TARGET_BATCH}_quality.json"}
if set(staging_files_now) != expected_now:
    raise RuntimeError("予定外ファイル存在")
print("✅ 公開直前検証通過")

# 【6. 正式公開】
print("\n【6. 正式公開】")
if batch_parquet_path.exists() or batch_json_path.exists():
    raise FileExistsError("公開直前に正式ファイル出現")

try:
    os.replace(staging_parquet, batch_parquet_path)
    os.replace(staging_quality, batch_json_path)
except Exception as e:
    raise RuntimeError(f"【処理停止】公開移動エラー。Parquet公開済み: {batch_parquet_path.exists()}, JSON公開済み: {batch_json_path.exists()} / {e}")

final_df = pd.read_parquet(batch_parquet_path)
with open(batch_json_path, "r", encoding="utf-8") as f:
    final_q = json.load(f)
final_pq_sha = get_file_sha256(batch_parquet_path)
final_q_sha = get_file_sha256(batch_json_path)

if len(final_df) != 1391:
    raise ValueError("公開後Parquet行数不一致")
if final_pq_sha != real_pq_sha:
    raise ValueError("公開後Parquet SHA不一致")
if final_q_sha != real_q_sha:
    raise ValueError("公開後Quality JSON SHA不一致")
if final_q["parquet_sha256"] != final_pq_sha:
    raise ValueError("公開後JSON内のParquet SHA不一致")
print("✅ 正式公開および公開後検証通過")

# 【7. commitマーカーのアトミック作成】
print("\n【7. commitマーカーのアトミック作成】")
commit_data = {
    "commit_schema_version": "1.0",
    "batch_id": TARGET_BATCH,
    "config_hash": config_hash,
    "session_id": session_id,
    "committed_at": datetime.now(timezone.utc).isoformat(),
    "parquet_filename": batch_parquet_path.name,
    "parquet_sha256": final_pq_sha,
    "quality_filename": batch_json_path.name,
    "quality_sha256": final_q_sha,
    "accepted_rows": 1391,
    "input_file_count": 25
}

temp_commit = BATCH_DIR / f"{TARGET_BATCH}_commit_{session_id}.tmp.json"
if temp_commit.exists():
    raise FileExistsError(f"一時コミットファイルが存在します: {temp_commit}")

with open(temp_commit, "x", encoding="utf-8") as f:
    json.dump(commit_data, f, indent=4, ensure_ascii=False, allow_nan=False)
    f.flush()
    os.fsync(f.fileno())
with open(temp_commit, "r", encoding="utf-8") as f:
    if json.load(f) != commit_data:
        raise ValueError("一時コミットファイル内容不一致")
os.replace(temp_commit, batch_commit_path)

with open(batch_commit_path, "r", encoding="utf-8") as f:
    final_commit = json.load(f)
if final_commit["parquet_sha256"] != final_pq_sha or final_commit["quality_sha256"] != final_q_sha:
    raise ValueError("コミットファイル内のSHA不一致")
print("✅ commitマーカーのアトミック作成成功")

# 【8. resume_stateのアトミック更新】
print("\n【8. resume_stateのアトミック更新】")
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs = json.load(f)
if rs["current_session_id"] != session_id or rs["config_hash"] != config_hash:
    raise ValueError("更新直前のresume_state検証失敗")
if TARGET_BATCH not in rs["pending_batch_ids"] or TARGET_BATCH in rs["completed_batch_ids"]:
    raise ValueError("更新直前のpending/completed状態が不正")

rs["status"] = "running"
rs["completed_batch_ids"].append(TARGET_BATCH)
# 元のmanifestの順序でソート
rs["completed_batch_ids"] = [b for b in list(batches.keys()) if b in rs["completed_batch_ids"]]
rs["pending_batch_ids"].remove(TARGET_BATCH)
rs["completed_source_files"].extend(batch_files)
# 同様にソート
rs["completed_source_files"] = [f for b in rs["completed_batch_ids"] for f in batches[b]]

rs["completed_batches"] = 1
rs["completed_files"] = 25
rs["last_completed_batch"] = TARGET_BATCH
rs["last_completed_source_file"] = batch_files[-1]
rs["last_update"] = datetime.now(timezone.utc).isoformat()
rs["completed_output_rows"] = 1391  # 現在は1件だけなので直接代入または品質JSONから計算

temp_rs = RESUME_STATE_PATH.parent / f"resume_state_{session_id}.tmp.json"
if temp_rs.exists():
    raise FileExistsError("一時resumeファイルが存在します")

with open(temp_rs, "x", encoding="utf-8") as f:
    json.dump(rs, f, indent=4, ensure_ascii=False, allow_nan=False)
    f.flush()
    os.fsync(f.fileno())
with open(temp_rs, "r", encoding="utf-8") as f:
    if json.load(f) != rs:
        raise ValueError("一時resumeファイル内容不一致")
os.replace(temp_rs, RESUME_STATE_PATH)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    final_rs = json.load(f)
if final_rs["completed_batches"] != 1 or final_rs["completed_output_rows"] != 1391 or TARGET_BATCH not in final_rs["completed_batch_ids"]:
    raise ValueError("resume_state更新後検証失敗")
print("✅ resume_stateアトミック更新成功")

# 【9. ステージングの後処理】
print("\n【9. ステージングの後処理】")
remaining_files = list(STAGING_BATCH_ROOT.iterdir())
if not remaining_files:
    STAGING_BATCH_ROOT.rmdir()
    print("✅ 空のステージングディレクトリを削除")
else:
    print(f"⚠️ ステージングにファイルが残っています: {[p.name for p in remaining_files]}")

# 【10. 最終lock更新】
print("\n【10. 最終lock更新】")
atomic_update_lock("batch_00021_committed")
print("✅ processing_lock を batch_00021_committed に更新")

# 【11. 最終監査】
print("\n【11. 最終監査】")
audit_checks = {
    "正式Parquet存在": batch_parquet_path.exists(),
    "正式Quality JSON存在": batch_json_path.exists(),
    "正式Commit JSON存在": batch_commit_path.exists(),
    "batch_00021 completed": TARGET_BATCH in final_rs["completed_batch_ids"],
    "batch_00021 not pending": TARGET_BATCH not in final_rs["pending_batch_ids"],
    "completed_batches == 1": final_rs["completed_batches"] == 1,
    "pending_batch_count == 72": len(final_rs["pending_batch_ids"]) == 72,
    "completed_files == 25": final_rs["completed_files"] == 25,
    "completed_output_rows == 1391": final_rs["completed_output_rows"] == 1391,
    "ステージング消失": not STAGING_BATCH_ROOT.exists(),
    "最終pickleなし": not FINAL_PICKLE_PATH.exists(),
    "最終JSONなし": not FINAL_JSON_PATH.exists()
}
for k, v in audit_checks.items():
    if not v: raise RuntimeError(f"最終監査失敗: {k}")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    if json.load(f)["status"] != "batch_00021_committed":
        raise RuntimeError("最終lock status不一致")
print("✅ 最終監査通過")

# 【12. 最終表示】
print("\n=========================================")
print(f"Parquetパス: {batch_parquet_path.name} (Size: {batch_parquet_path.stat().st_size}, SHA-256: {final_pq_sha})")
print(f"Quality JSONパス: {batch_json_path.name} (Size: {batch_json_path.stat().st_size}, SHA-256: {final_q_sha})")
print(f"Commit JSONパス: {batch_commit_path.name} (Size: {batch_commit_path.stat().st_size})")
print(f"raw_rows: {final_q['raw_rows']}")
print(f"accepted_rows: {final_q['accepted_rows']}")
print(f"Remark 1件数: {final_q['remark_1_count']}")
print(f"Remark 2件数: {final_q['remark_2_count']}")
print("APE統計:\n", pd.Series(final_q['ape_statistics']).to_string())
print(f"completed_batches: {final_rs['completed_batches']}")
print(f"pending batch数: {len(final_rs['pending_batch_ids'])}")
print(f"completed_files: {final_rs['completed_files']}")
print(f"completed_output_rows: {final_rs['completed_output_rows']}")
print(f"staging残存有無: {STAGING_BATCH_ROOT.exists()}")
print(f"lock status: batch_00021_committed")
print("resume_state検証結果: 成功")

print("\nbatch_00021回復コミット成功。")
print("全バッチ再開処理の実装前監査へ進める")


--- batch_00021 回復・正式公開・commit・resume_state更新 ---

【1. 同一ランタイム・監査結果確認】
✅ メモリ上変数の検証完了
✅ ロック status = recovering_batch_00021_publish に更新

【2. 現在状態の再確認】
公開前 completed_batches: 0
公開前 completed_files: 0
公開前 completed_output_rows: 0
✅ 現在状態の再確認完了

【3. Quality JSONの正式ステージング名】
✅ temp_q.json を正式なステージング名に変更し内容を確認

【4. 実ファイルSHA-256】
✅ 実Parquet SHA-256確認: 0804e077ced353dff0e4077e08d2684b5d52ceb4ae9f994244fc41a8dd679cdc
✅ 実Quality JSON SHA-256計算: aa059d82b0ab5aa6e645db972b24a9a0fffa00d75ae3bd5526518365d062dcfc

【5. 公開直前の最終検証】
✅ 公開直前検証通過

【6. 正式公開】
✅ 正式公開および公開後検証通過

【7. commitマーカーのアトミック作成】
✅ commitマーカーのアトミック作成成功

【8. resume_stateのアトミック更新】
✅ resume_stateアトミック更新成功

【9. ステージングの後処理】
✅ 空のステージングディレクトリを削除

【10. 最終lock更新】
✅ processing_lock を batch_00021_committed に更新

【11. 最終監査】
✅ 最終監査通過

Parquetパス: batch_00021.parquet (Size: 32621, SHA-256: 0804e077ced353dff0e4077e08d2684b5d52ceb4ae9f994244fc41a8dd679cdc)
Quality JSONパス: batch_00021_quality.json (Size: 32361, SHA-256: aa059d82b0ab5aa6e645db972b24a9a0fffa00d

In [26]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import traceback
import sys
import re
from pathlib import Path
import collections

print("--- 全バッチ再開処理の実装前監査・汎用関数定義 ---")

# 【14. 実行禁止】
EXECUTE_PENDING_BATCHES = False
print(f"EXECUTE_PENDING_BATCHES = {EXECUTE_PENDING_BATCHES}")

EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"

# 実行前の件数確認
parquet_files_before = list(BATCH_DIR.glob("*.parquet"))
quality_json_files_before = list(BATCH_DIR.glob("*_quality.json"))
commit_json_files_before = list(BATCH_DIR.glob("*_commit.json"))

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs_before = json.load(f)

print("\n【1. 現在セッション確認】")
if 'session_id' not in globals():
    raise RuntimeError("メモリ上に session_id がありません。")
if session_id != EXPECTED_SESSION_ID:
    raise RuntimeError(f"session_id不一致: {session_id}")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    lock = json.load(f)
if lock.get("session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("lockのsession_id不一致")
if rs_before.get("current_session_id") != EXPECTED_SESSION_ID:
    raise RuntimeError("resume_stateのcurrent_session_id不一致")

# update_lock_v2の定義を前倒し
def update_lock_v2(status_str, sess_id):
    now_utc = datetime.now(timezone.utc).isoformat()
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        l = json.load(f)
    if l.get("session_id") != sess_id:
        raise RuntimeError("所有権喪失")
    l["heartbeat"] = now_utc
    l["status"] = status_str
    tmp = PROCESSING_LOCK_PATH.parent / f"{PROCESSING_LOCK_PATH.stem}_{sess_id}_hb.tmp.json"
    if tmp.exists(): raise FileExistsError("tmp lock exists")
    with open(tmp, "x", encoding="utf-8") as f:
        json.dump(l, f, indent=4, ensure_ascii=False)
        f.flush(); os.fsync(f.fileno())
    with open(tmp, "r", encoding="utf-8") as f:
        if json.load(f) != l: raise ValueError("tmp lock mismatch")
    os.replace(tmp, PROCESSING_LOCK_PATH)

update_lock_v2("auditing_all_batch_runner", session_id)
print("✅ セッション確認・lock更新完了")

print("\n【2. committed成果物の完全監査】")
committed_batches = []
partial_batches = []
orange_stagings = []
resume_mismatches = []

for p in BATCH_DIR.iterdir():
    if p.suffix == ".parquet":
        b_id = p.stem
        pq_path = p
        q_path = BATCH_DIR / f"{b_id}_quality.json"
        c_path = BATCH_DIR / f"{b_id}_commit.json"

        if pq_path.exists() and q_path.exists() and c_path.exists():
            committed_batches.append(b_id)
        else:
            partial_batches.append(b_id)

print(f"完全コミットバッチ: {committed_batches}")
print(f"部分公開バッチ: {partial_batches}")

if partial_batches:
    raise RuntimeError(f"部分公開バッチが存在します: {partial_batches}")

if set(committed_batches) != {"batch_00021"}:
    raise RuntimeError(f"期待するバッチ(batch_00021)以外が存在します: {committed_batches}")

b_id = "batch_00021"
pq_path = BATCH_DIR / f"{b_id}.parquet"
q_path = BATCH_DIR / f"{b_id}_quality.json"
c_path = BATCH_DIR / f"{b_id}_commit.json"

with open(q_path, "r", encoding="utf-8") as f: q_data = json.load(f)
with open(c_path, "r", encoding="utf-8") as f: c_data = json.load(f)
pq_df = pd.read_parquet(pq_path)
pq_sha = hashlib.sha256(pq_path.read_bytes()).hexdigest()
q_sha = hashlib.sha256(q_path.read_bytes()).hexdigest()

checks_00021 = {
    "pq_sha_match": pq_sha == c_data["parquet_sha256"],
    "q_sha_match": q_sha == c_data["quality_sha256"],
    "q_pq_sha_match": q_data["parquet_sha256"] == pq_sha,
    "config_hash": q_data["config_hash"] == config_hash,
    "input_files": q_data["input_file_count"] == 25,
    "accepted": q_data["accepted_rows"] == 1391,
    "pq_rows": len(pq_df) == 1391,
    "err_count": q_data["processing_error_count"] == 0,
    "key_dup": q_data.get("source_key_duplicate_rows", pq_df.duplicated(subset=['source_file', 'source_line_number']).sum()) == 0,
    "ape_nan_inf": pq_df['APE'].isna().sum() == 0 and np.isinf(pq_df['APE']).sum() == 0,
    "ape_range": (pq_df['APE'] < 1.2).sum() == 0 and (pq_df['APE'] > 2.2).sum() == 0
}
if not all(checks_00021.values()):
    raise RuntimeError(f"batch_00021検証失敗: {checks_00021}")
print("✅ batch_00021成果物検証完了")

print("\n【3. resume_state再構築照合】")
# re-build
expected_completed_ids = sorted(committed_batches)
expected_completed_files = sum(c_data["input_file_count"] for _ in expected_completed_ids) # 25
expected_output_rows = sum(q_data["accepted_rows"] for _ in expected_completed_ids) # 1391
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f: manifest = json.load(f)
all_batches = list(manifest["batches"].keys())
expected_pending = [b for b in all_batches if b not in expected_completed_ids]
expected_src_files = []
for b in expected_completed_ids:
    expected_src_files.extend(manifest["batches"][b])

checks_rs = {
    "comp_ids": rs_before["completed_batch_ids"] == expected_completed_ids,
    "comp_batches": rs_before["completed_batches"] == len(expected_completed_ids),
    "comp_files": rs_before["completed_files"] == len(expected_src_files),
    "comp_rows": rs_before.get("completed_output_rows", 0) == expected_output_rows,
    "pending_count": len(rs_before["pending_batch_ids"]) == 72,
    "comp_src": rs_before["completed_source_files"] == expected_src_files,
    "failed_empty": rs_before.get("failed_batch_ids", []) == []
}
if not all(checks_rs.values()):
    resume_mismatches.append(checks_rs)
    raise RuntimeError(f"resume_state不一致: {checks_rs}")
print("✅ resume_state再構築照合完了")

print("\n【4. ディレクトリ状態】")
if list(TEMP_DIR.iterdir()): raise RuntimeError("TEMP_DIR not empty")
if list(ERROR_DIR.iterdir()): raise RuntimeError("ERROR_DIR not empty")
stagings = list(OUTPUT_DIR.glob(".ape_HSR_r12_350_1050_*"))
if stagings: raise RuntimeError(f"孤立ステージングあり: {stagings}")
tmps = list(CHECKPOINT_ROOT.rglob("*.tmp*"))
if tmps: raise RuntimeError(f"tmpファイルあり: {tmps}")
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists(): raise RuntimeError("最終成果物あり")
print("✅ ディレクトリ状態確認完了")

print("\n【5, 6, 7, 8, 9, 13. 汎用処理関数定義】")

def make_json_safe_v2(obj, path="root"):
    if isinstance(obj, dict):
        if any(not isinstance(k, str) for k in obj.keys()):
            obj = {str(k): v for k, v in obj.items()}
        return {k: make_json_safe_v2(v, f"{path}.{k}") for k, v in obj.items()}
    elif isinstance(obj, list) or isinstance(obj, tuple):
        return [make_json_safe_v2(item, f"{path}[{i}]") for i, item in enumerate(obj)]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            raise ValueError(f"NaN/inf found at {path}")
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, (datetime, pd.Timestamp)):
        return obj.isoformat()
    elif pd.isna(obj):
        return None
    else:
        return obj

def atomic_write_json_v2(data_dict, target_path, sess_id):
    safe_data = make_json_safe_v2(data_dict)
    tmp = target_path.parent / f"{target_path.stem}_{sess_id}.tmp.json"
    if tmp.exists(): raise FileExistsError(f"{tmp} exists")
    with open(tmp, "x", encoding="utf-8") as f:
        json.dump(safe_data, f, indent=4, ensure_ascii=False, allow_nan=False)
        f.flush(); os.fsync(f.fileno())
    with open(tmp, "r", encoding="utf-8") as f:
        if json.load(f) != safe_data: raise ValueError("json write mismatch")
    os.replace(tmp, target_path)

# update_lock_v2 already defined

def parse_hsr_header_v2(filepath):
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, nrows=0)
    cols = list(df.columns)
    if cols[:5] != ["地点番号", "地点名", "年月日", "時分", "リマーク"]:
        raise ValueError("header error")
    w_cols = []
    for c in cols[5:]:
        m = re.match(r"^日射強度\((\d+)nm\)", c)
        if m: w_cols.append({'name': c, 'wave': int(m.group(1))})
    target = [x['name'] for x in w_cols if 350 <= x['wave'] <= 1050]
    full = [x['name'] for x in w_cols if 350 <= x['wave'] <= 1700]
    return target, full

def parse_datetime_columns_v2(df):
    pts = []
    for _, r in df.iterrows():
        try:
            bd = datetime.strptime(str(r['年月日']).strip(), "%Y/%m/%d")
            hm = str(r['時分']).strip()
            if hm == "24:00":
                pts.append(bd + timedelta(days=1))
            else:
                h, m = hm.split(':')
                pts.append(bd.replace(hour=int(h), minute=int(m)))
        except: pts.append(pd.NaT)
    df['Datetime'] = pts
    return df

def calculate_ape_for_file_v2(filepath, rel_path, config):
    target_w, full_w = parse_hsr_header_v2(filepath)
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False)
    df['source_file'] = rel_path
    df['source_row_index'] = df.index
    df['source_line_number'] = df.index + 2
    df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')
    df = parse_datetime_columns_v2(df)

    # Filtering
    m_site = df['地点番号'] == 301
    m_rem = (df['remark_num'] == 1) | (df['remark_num'] == 2)
    m_dt = df['Datetime'].notna()
    arr_t = df[target_w].to_numpy(dtype=np.float64)
    m_val = np.isfinite(arr_t).all(axis=1) & (arr_t != -9999).all(axis=1)
    m_adopt = m_site & m_rem & m_dt & m_val

    adopted = df[m_adopt].copy()
    if not adopted.empty:
        G = adopted[target_w].to_numpy(dtype=np.float64)
        waves = np.arange(350, 1051, dtype=np.float64)
        num = np.sum(G, axis=1)
        den = np.sum(G * waves[None, :], axis=1)
        m_nd = np.isfinite(num) & np.isfinite(den) & (num > 0) & (den > 0)
        adopted = adopted[m_nd].copy()
        if not adopted.empty:
            num = num[m_nd]
            den = den[m_nd]
            adopted['APE'] = 1239.84193 * num / den
            adopted = adopted[(adopted['APE'] >= config['ape_valid_range_eV'][0]) & (adopted['APE'] <= config['ape_valid_range_eV'][1])]

    cols = ['Datetime', '地点番号', '地点名', 'remark_num', 'APE', 'source_file', 'source_row_index', 'source_line_number']
    if adopted.empty:
        out_df = pd.DataFrame(columns=cols).rename(columns={'地点番号':'SiteNum', '地点名':'SiteName', 'remark_num':'Remark'})
    else:
        out_df = adopted[cols].rename(columns={'地点番号':'SiteNum', '地点名':'SiteName', 'remark_num':'Remark'})
    return out_df, {"raw_rows": len(df), "accepted_rows": len(out_df)}

def process_batch_v2(batch_id, rel_paths, config, sess_id):
    # dummy implementation for audit
    return pd.DataFrame(), {}, 0

def stage_and_validate_batch_v2(): pass
def publish_and_commit_batch_v2(): pass
def rebuild_expected_resume_v2(): pass
def update_resume_from_commits_v2(): pass
def audit_committed_batches_v2(): pass
def run_pending_batches_v2(): pass

print("✅ 汎用関数(v2)定義完了")

print("\n【11. 処理順序】")
pending_ids = manifest["batches"].keys()
pending_actual = [b for b in pending_ids if b not in committed_batches]
print(f"先頭5件: {pending_actual[:5]}")
print(f"末尾5件: {pending_actual[-5:]}")
if len(manifest["batches"]["batch_00072"]) != 24:
    raise RuntimeError("batch_00072 != 24 files")

print("\n【14. 実行前後の成果物件数確認】")
parquet_files_after = list(BATCH_DIR.glob("*.parquet"))
quality_json_files_after = list(BATCH_DIR.glob("*_quality.json"))
commit_json_files_after = list(BATCH_DIR.glob("*_commit.json"))

if len(parquet_files_after) != 1 or len(parquet_files_before) != 1:
    raise RuntimeError("Parquet count changed or not 1")
if len(quality_json_files_after) != 1 or len(quality_json_files_before) != 1:
    raise RuntimeError("Q-JSON count changed or not 1")
if len(commit_json_files_after) != 1 or len(commit_json_files_before) != 1:
    raise RuntimeError("C-JSON count changed or not 1")

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs_after = json.load(f)

if rs_after["completed_batches"] != 1: raise RuntimeError()
if len(rs_after["pending_batch_ids"]) != 72: raise RuntimeError()
if rs_after["completed_files"] != 25: raise RuntimeError()
if rs_after.get("completed_output_rows", 0) != 1391: raise RuntimeError()

print("\n【15. 最終表示】")
print(f"committed完全バッチ一覧: {committed_batches}")
print(f"部分公開バッチ一覧: {partial_batches}")
print(f"孤立ステージング一覧: {orange_stagings}")
print(f"resume不一致一覧: {resume_mismatches}")
print(f"pending処理予定数: {len(pending_actual)}")
print(f"処理予定の先頭5バッチ: {pending_actual[:5]}")
print(f"処理予定の末尾5バッチ: {pending_actual[-5:]}")
print(f"最後のバッチの入力数: {len(manifest['batches']['batch_00072'])}")
print("定義したv2関数一覧: [make_json_safe_v2, atomic_write_json_v2, update_lock_v2, parse_hsr_header_v2, parse_datetime_columns_v2, calculate_ape_for_file_v2, process_batch_v2, ...]")
print(f"EXECUTE_PENDING_BATCHES: {EXECUTE_PENDING_BATCHES}")
print(f"実行前成果物件数: Parquet={len(parquet_files_before)}, Q-JSON={len(quality_json_files_before)}, C-JSON={len(commit_json_files_before)}")
print(f"実行後成果物件数: Parquet={len(parquet_files_after)}, Q-JSON={len(quality_json_files_after)}, C-JSON={len(commit_json_files_after)}")
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    print(f"lock status: {json.load(f)['status']}")

print("\n全バッチ再開処理の実装前監査成功。")
print("実行セルの提示前にコードレビュー可能")


--- 全バッチ再開処理の実装前監査・汎用関数定義 ---
EXECUTE_PENDING_BATCHES = False

【1. 現在セッション確認】
✅ セッション確認・lock更新完了

【2. committed成果物の完全監査】
完全コミットバッチ: ['batch_00021']
部分公開バッチ: []
✅ batch_00021成果物検証完了

【3. resume_state再構築照合】
✅ resume_state再構築照合完了

【4. ディレクトリ状態】
✅ ディレクトリ状態確認完了

【5, 6, 7, 8, 9, 13. 汎用処理関数定義】
✅ 汎用関数(v2)定義完了

【11. 処理順序】
先頭5件: ['batch_00000', 'batch_00001', 'batch_00002', 'batch_00003', 'batch_00004']
末尾5件: ['batch_00068', 'batch_00069', 'batch_00070', 'batch_00071', 'batch_00072']

【14. 実行前後の成果物件数確認】

【15. 最終表示】
committed完全バッチ一覧: ['batch_00021']
部分公開バッチ一覧: []
孤立ステージング一覧: []
resume不一致一覧: []
pending処理予定数: 72
処理予定の先頭5バッチ: ['batch_00000', 'batch_00001', 'batch_00002', 'batch_00003', 'batch_00004']
処理予定の末尾5バッチ: ['batch_00068', 'batch_00069', 'batch_00070', 'batch_00071', 'batch_00072']
最後のバッチの入力数: 24
定義したv2関数一覧: [make_json_safe_v2, atomic_write_json_v2, update_lock_v2, parse_hsr_header_v2, parse_datetime_columns_v2, calculate_ape_for_file_v2, process_batch_v2, ...]
EXECUTE_PENDING_BATCHES: False
実

In [33]:
print("--- 第3段階: run_pending_batches_v2 関数の完全実装 ---")

def run_pending_batches_v2(manifest_data, config, sess_id, max_batches=None):
    # 【1. 実行ガード】
    if not EXECUTE_PENDING_BATCHES:
        raise RuntimeError("EXECUTE_PENDING_BATCHES が True ではありません。安全のため処理を停止します。")

    if max_batches is not None:
        if isinstance(max_batches, bool) or not isinstance(max_batches, int) or max_batches < 1:
            raise ValueError(f"max_batches は None または boolではない1以上の整数である必要があります: {max_batches}")

    if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
        raise FileExistsError("最終成果物（FINAL_PICKLE_PATH または FINAL_JSON_PATH）が既に存在します。処理を停止します。")

    required_paths = {
        "RUN_MANIFEST_PATH": RUN_MANIFEST_PATH,
        "RESUME_STATE_PATH": RESUME_STATE_PATH,
        "PROCESSING_LOCK_PATH": PROCESSING_LOCK_PATH,
        "BATCH_DIR": BATCH_DIR,
        "ERROR_DIR": ERROR_DIR,
        "TEMP_DIR": TEMP_DIR
    }
    for name, path in required_paths.items():
        if not path.exists():
            raise FileNotFoundError(f"必須パスが存在しません: {name} ({path})")

    # 【1.5 マニフェストとConfigの一致確認】
    with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
        disk_manifest = json.load(f)

    safe_manifest_data = make_json_safe_v2(manifest_data)
    if safe_manifest_data != disk_manifest:
        raise ValueError("メモリ上の manifest_data が disk上の manifest と完全一致しません。")

    config_keys_to_check = [
        "pipeline_version", "dataset_name", "plane", "w_min", "w_max",
        "remarks_used", "duplicate_policy", "ape_valid_range_eV",
        "source_timezone", "datetime_storage", "source_file_pattern",
        "formula", "batch_size", "expected_site_numbers"
    ]
    safe_config = make_json_safe_v2(config)
    for k in config_keys_to_check:
        if safe_config.get(k) != safe_manifest_data.get(k):
            raise ValueError(f"config の項目 {k} が manifest と一致しません。")

    # 【2. 所有権確認】
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        lock_data = json.load(f)
    if lock_data.get("session_id") != sess_id:
        raise RuntimeError(f"所有権不一致: processing_lock の session_id が {sess_id} と一致しません。")

    with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
        resume_data = json.load(f)
    if resume_data.get("current_session_id") != sess_id:
        raise RuntimeError(f"所有権不一致: resume_state の current_session_id が {sess_id} と一致しません。")

    manifest_config_hash = manifest_data.get("config_hash")
    if resume_data.get("config_hash") != manifest_config_hash:
        raise ValueError("resume_state の config_hash が manifest の config_hash と一致しません。")

    # 【3. 開始前監査】
    temp_files = list(TEMP_DIR.iterdir())
    if temp_files:
        print("【処理停止】TEMP_DIR にファイルまたはディレクトリが残存しています:")
        for tmp_f in temp_files:
            print(f"  - {tmp_f.name}")
        raise RuntimeError("TEMP_DIR が空ではないため、処理を開始できません。")

    audit_committed_batches_v2(manifest_data)
    update_resume_from_commits_v2(manifest_data, sess_id)

    with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
        resume_data_updated = json.load(f)

    pending_ids_from_resume = set(resume_data_updated.get("pending_batch_ids", []))
    manifest_batches_order = list(manifest_data.get("batches", {}).keys())

    pending_batches_to_process = [b for b in manifest_batches_order if b in pending_ids_from_resume]

    if not pending_batches_to_process:
        print("処理待ちのバッチはありません。")
        # 既に完了している場合の検証ステップへ進む

    # 【4. バッチ処理】
    processed_count = 0
    for batch_id in pending_batches_to_process:
        if max_batches is not None and processed_count >= max_batches:
            print(f"指定された max_batches ({max_batches}) に達したため、バッチ処理ループを終了します。")
            break

        expected_staging_dir = TEMP_DIR / f"{batch_id}_{sess_id}.staging"
        staging_dir = None

        try:
            update_lock_v2(f"starting_{batch_id}", sess_id)

            staging_dir, staging_parquet, staging_json, quality_data, pq_sha256 = stage_and_validate_batch_v2(
                batch_id, manifest_data, config, sess_id
            )

            update_lock_v2(f"staged_{batch_id}", sess_id)

            publish_and_commit_batch_v2(
                batch_id, staging_dir, staging_parquet, staging_json, quality_data, pq_sha256, manifest_config_hash, sess_id
            )

            update_resume_from_commits_v2(manifest_data, sess_id)

            update_lock_v2(f"committed_{batch_id}", sess_id)

            with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
                current_rs = json.load(f)

            comp_cnt = current_rs.get("completed_batches", 0)
            pend_cnt = len(current_rs.get("pending_batch_ids", []))
            accepted = quality_data.get("accepted_rows", 0)
            print(f"✅ {batch_id} 完了: 採用行数={accepted} / 進捗: 完了={comp_cnt}, 残り={pend_cnt}")

            processed_count += 1

        except Exception as e:
            # 【5. 例外時】
            error_staging_path = None
            if staging_dir is not None:
                error_staging_path = str(staging_dir.resolve())
            elif expected_staging_dir.exists():
                error_staging_path = str(expected_staging_dir.resolve())

            error_info = {
                "exception_type": type(e).__name__,
                "message": str(e),
                "traceback": traceback.format_exc(),
                "batch_id": batch_id,
                "staging_dir": error_staging_path
            }

            try:
                write_error_record_v2(batch_id, error_info, sess_id)
            except Exception as nested_e:
                print(f"[警告] エラーレコードの保存に失敗しました: {nested_e}")

            try:
                update_lock_v2(f"failed_{batch_id}", sess_id)
            except Exception as nested_e:
                print(f"[警告] lock更新(failed)に失敗しました: {nested_e}")

            raise

    # 【6. 全バッチ終了時】
    audit_committed_batches_v2(manifest_data)
    update_resume_from_commits_v2(manifest_data, sess_id)

    with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
        final_rs = json.load(f)

    if not final_rs.get("pending_batch_ids"):
        total_manifest_batches = len(manifest_data.get("batches", {}))
        total_manifest_files = manifest_data.get("total_hsr_files", 0)
        all_manifest_files = []
        for files in manifest_data.get("batches", {}).values():
            all_manifest_files.extend(files)

        completed_batches_count = final_rs.get("completed_batches", 0)
        completed_files_count = final_rs.get("completed_files", 0)
        completed_source_files = final_rs.get("completed_source_files", [])

        checks = {
            "pending_empty": len(final_rs.get("pending_batch_ids", [])) == 0,
            "completed_batches_match": completed_batches_count == total_manifest_batches,
            "completed_files_match": completed_files_count == total_manifest_files,
            "completed_source_files_no_dups": len(completed_source_files) == len(set(completed_source_files)),
            "completed_source_files_set_match": set(completed_source_files) == set(all_manifest_files),
            "temp_dir_empty": len(list(TEMP_DIR.iterdir())) == 0
        }

        if not all(checks.values()):
            raise RuntimeError(f"全バッチ完了時の最終検証に失敗しました: {checks}")

        # アトミック更新
        final_rs["status"] = "batches_complete"
        atomic_write_json_v2(final_rs, RESUME_STATE_PATH, sess_id)
        update_lock_v2("all_batches_committed", sess_id)
        print("\n🎉 すべてのバッチ処理が完了し、最終検証を通過しました。")

EXECUTE_PENDING_BATCHES = False
print("✅ run_pending_batches_v2 関数が定義されました (ダミー・passなし)")
print(f"EXECUTE_PENDING_BATCHES = {EXECUTE_PENDING_BATCHES}")

--- 第3段階: run_pending_batches_v2 関数の完全実装 ---
✅ run_pending_batches_v2 関数が定義されました (ダミー・passなし)
EXECUTE_PENDING_BATCHES = False


In [36]:
import os
import json
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import uuid
import joblib

print("--- 第4段階: 最終結合・Pickle/JSON生成・監査 ---")

EXPECTED_SESSION_ID = "152d121e-2e5f-445f-88e6-e644e5829f2d"

# 【1. 最終化前監査】
print("\n【1. 最終化前監査】")
if 'session_id' not in globals() or session_id != EXPECTED_SESSION_ID:
    raise RuntimeError("【処理停止】メモリ上の session_id が一致しません。")

if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。上書きを防止します。")

if list(TEMP_DIR.iterdir()):
    raise RuntimeError("【処理停止】TEMP_DIR が空ではありません。")

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs = json.load(f)
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    lock = json.load(f)

if rs["status"] != "batches_complete":
    raise RuntimeError(f"【処理停止】resume_state.statusが不正: {rs['status']}")
if lock["status"] != "all_batches_committed":
    raise RuntimeError(f"【処理停止】processing_lock.statusが不正: {lock['status']}")
if len(rs.get("pending_batch_ids", [])) > 0:
    raise RuntimeError("【処理停止】pending_batch_idsが空ではありません。")

initial_errors = set(p.name for p in ERROR_DIR.iterdir())
valid_batches = audit_committed_batches_v2(manifest_data)

manifest_batches = list(manifest_data["batches"].keys())
if set(valid_batches) != set(manifest_batches):
    raise RuntimeError("【処理停止】コミット済みbatch集合がmanifestの全batch集合と一致しません。")

current_errors = set(p.name for p in ERROR_DIR.iterdir())
if current_errors != initial_errors:
    raise RuntimeError("【処理停止】error JSONが新規作成されています。")

print("✅ 最終化前監査クリア")

# 【2. 全Parquet結合】
print("\n【2. 全Parquet結合】")
dfs = []
for b_id in manifest_batches: # manifestのbatch順で読み込み
    pq_path = BATCH_DIR / f"{b_id}.parquet"
    df = pd.read_parquet(pq_path)
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)
expected_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']
final_df = final_df[expected_cols]

canonical_dtypes = {
    "Datetime": "datetime64[ns]",
    "SiteNum": "int64",
    "SiteName": "string",
    "Remark": "int64",
    "APE": "float64",
    "source_file": "string",
    "source_row_index": "int64",
    "source_line_number": "int64",
    "batch_id": "string"
}

final_df["Datetime"] = pd.to_datetime(
    final_df["Datetime"],
    errors="raise"
)

for col in [
    "SiteNum",
    "SiteName",
    "Remark",
    "APE",
    "source_file",
    "source_row_index",
    "source_line_number",
    "batch_id"
]:
    final_df[col] = final_df[col].astype(canonical_dtypes[col])

for col, expected_dtype in canonical_dtypes.items():
    if str(final_df[col].dtype) != expected_dtype:
        raise ValueError(
            f"dtype不一致: {col}: "
            f"actual={final_df[col].dtype}, "
            f"expected={expected_dtype}"
        )

# 指定された3つのキーでstable sort
final_df = final_df.sort_values(by=['Datetime', 'source_file', 'source_line_number'], kind='stable', ignore_index=True)

# 【3. 結合DataFrameの検証】
print("\n【3. 結合DataFrameの検証】")
if len(final_df) != rs["completed_output_rows"]:
    raise ValueError(f"【検証失敗】行数が resume_state ({rs['completed_output_rows']}) と一致しません ({len(final_df)})。")
if list(final_df.columns) != expected_cols:
    raise ValueError("【検証失敗】列名・列順が一致しません。")

if final_df.isna().sum().sum() > 0:
    raise ValueError("【検証失敗】主要9列に欠損があります。")

if final_df['Datetime'].dt.tz is not None:
    raise ValueError("【検証失敗】Datetimeがtimezone-naiveではありません。")

if not np.isfinite(final_df['APE']).all():
    raise ValueError("【検証失敗】APEに非有限値が含まれています。")

if (final_df['APE'] < CONFIG['ape_valid_range_eV'][0]).any() or (final_df['APE'] > CONFIG['ape_valid_range_eV'][1]).any():
    raise ValueError("【検証失敗】APEがconfigの有効範囲外です。")

if not final_df['Remark'].isin(CONFIG["remarks_used"]).all():
    raise ValueError("【検証失敗】Remarkがconfig['remarks_used']内にありません。")

if final_df.duplicated(subset=['source_file', 'source_line_number']).any():
    raise ValueError("【検証失敗】source_file/source_line_numberの複合キー重複があります。")

all_manifest_files = []
for b in manifest_batches:
    all_manifest_files.extend(manifest_data["batches"][b])

if not final_df['source_file'].isin(all_manifest_files).all():
    raise ValueError("【検証失敗】source_fileがmanifestの入力ファイル集合に含まれません。")

if not final_df['batch_id'].isin(manifest_batches).all():
    raise ValueError("【検証失敗】batch_idがmanifestのbatch集合に含まれません。")

if not final_df['SiteNum'].isin(CONFIG["expected_site_numbers"]).all():
    raise ValueError("【検証失敗】SiteNumがexpected_site_numbers内にありません。")

dups_time_mask = final_df.duplicated(subset=['Datetime'], keep=False)
dups_time_count = dups_time_mask.sum()
dups_time_groups = final_df[dups_time_mask]['Datetime'].nunique()

print(f"✅ 結合DataFrame検証クリア (行数: {len(final_df)})")

# 【4, 5. Pickle形式作成・アトミック保存】
print("\n【4, 5. Pickle形式作成・アトミック保存】")
target_df_dict = {
    "target_dataframe": final_df,
    "processor_config": {
        "plane": CONFIG["plane"],
        "w_min": CONFIG["w_min"],
        "w_max": CONFIG["w_max"],
        "remarks_used": CONFIG["remarks_used"],
        "duplicate_policy": "同一時刻平均なし",
        "source_timezone": CONFIG["source_timezone"],
        "datetime_storage": "timezone-naive"
    },
    "generation_metadata": {
        "config_hash": manifest_data["config_hash"],
        "batch_count": len(manifest_batches),
        "input_file_count": manifest_data["total_hsr_files"],
        "output_row_count": len(final_df),
        "created_at": datetime.now(timezone.utc).isoformat()
    }
}

temp_pickle_path = OUTPUT_DIR / f"final_pickle_{session_id}.tmp.pkl"
if temp_pickle_path.exists():
    raise FileExistsError("【処理停止】一時pickleが既に存在します。")

joblib.dump(target_df_dict, temp_pickle_path, compress=3)

# rb+で開いてflushとfsyncを確実に行う
with open(temp_pickle_path, "rb+") as f:
    f.flush()
    os.fsync(f.fileno())

# 再読込と検証
loaded_dict = joblib.load(temp_pickle_path)
if set(loaded_dict.keys()) != {"target_dataframe", "processor_config", "generation_metadata"}:
    raise ValueError("【検証失敗】再読込: トップレベルキーが不正です。")
if loaded_dict["processor_config"] != target_df_dict["processor_config"]:
    raise ValueError("【検証失敗】再読込: processor_configが不一致。")
if loaded_dict["generation_metadata"] != target_df_dict["generation_metadata"]:
    raise ValueError("【検証失敗】再読込: generation_metadataが不一致。")
pd.testing.assert_frame_equal(loaded_dict["target_dataframe"], target_df_dict["target_dataframe"])

tmp_pkl_sha = get_file_sha256_v2(temp_pickle_path)
tmp_pkl_size = temp_pickle_path.stat().st_size

if FINAL_PICKLE_PATH.exists():
    raise FileExistsError("【処理停止】公開直前にFINAL_PICKLE_PATHが作成されています。")

os.replace(temp_pickle_path, FINAL_PICKLE_PATH)
final_pkl_sha = get_file_sha256_v2(FINAL_PICKLE_PATH)

if final_pkl_sha != tmp_pkl_sha:
    raise ValueError("【検証失敗】公開前後でpickleのSHA-256が一致しません。")

print(f"✅ Pickleのアトミック保存・検証クリア (SHA-256: {final_pkl_sha})")

# 【6. 最終品質JSON】
print("\n【6. 最終品質JSONの作成】")
total_raw_rows = 0
raw_rem_1 = 0
raw_rem_2 = 0
m9999_cells = 0
nan_cells = 0
other_neg_cells = 0
rows_2400 = 0
parse_fails = 0
date_site_mismatches = 0
num_den_invalid = 0
ape_below = 0
ape_above = 0
batch_stats_list = []

for b_id in manifest_batches:
    q_path = BATCH_DIR / f"{b_id}_quality.json"
    with open(q_path, "r", encoding="utf-8") as f:
        q = json.load(f)
    total_raw_rows += q["raw_rows"]

    raw_rem_1 += int(q["remark_counts"].get("1", 0))
    raw_rem_2 += int(q["remark_counts"].get("2", 0))

    m9999_cells += q["minus_9999_cells_target"]
    nan_cells += q["NaN_cells_target"]
    other_neg_cells += q["other_negative_cells_target"]
    rows_2400 += q["24_00_rows"]
    parse_fails += q["datetime_parse_failures"]
    date_site_mismatches += q["date_or_site_mismatches"]
    num_den_invalid += q["numerator_or_denominator_invalid_rows"]
    ape_below += q["ape_below_1_2"]
    ape_above += q["ape_above_2_2"]

    batch_stats_list.append({
        "batch_id": b_id,
        "input_file_count": q["input_file_count"],
        "accepted_rows": q["accepted_rows"],
        "parquet_sha256": q["parquet_sha256"],
        "quality_sha256": get_file_sha256_v2(q_path)
    })

ape_stats = final_df['APE'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict()
accepted_rem_1 = (final_df['Remark'] == 1).sum()
accepted_rem_2 = (final_df['Remark'] == 2).sum()
run_manifest_sha = get_file_sha256_v2(RUN_MANIFEST_PATH)

final_q = {
    "final_quality_schema_version": "1.0",
    "pipeline_version": CONFIG["pipeline_version"],
    "config_hash": manifest_data["config_hash"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "plane": CONFIG["plane"],
    "wavelength_range_nm": [CONFIG["w_min"], CONFIG["w_max"]],
    "remarks_used": CONFIG["remarks_used"],
    "duplicate_policy": "同一時刻平均なし",
    "total_batches": len(manifest_batches),
    "total_input_files": manifest_data["total_hsr_files"],
    "total_raw_rows": total_raw_rows,
    "total_output_rows": len(final_df),
    "accepted_remark_1_rows": int(accepted_rem_1),
    "accepted_remark_2_rows": int(accepted_rem_2),
    "raw_remark_1_rows": raw_rem_1,
    "raw_remark_2_rows": raw_rem_2,
    "ape_statistics": ape_stats,
    "datetime_min": final_df['Datetime'].min().isoformat(),
    "datetime_max": final_df['Datetime'].max().isoformat(),
    "duplicate_datetime_rows": int(dups_time_count),
    "duplicate_datetime_groups": int(dups_time_groups),
    "source_key_duplicate_rows": 0,
    "minus_9999_cells_target": m9999_cells,
    "NaN_cells_target": nan_cells,
    "other_negative_cells_target": other_neg_cells,
    "24_00_rows": rows_2400,
    "datetime_parse_failures": parse_fails,
    "date_or_site_mismatches": date_site_mismatches,
    "numerator_or_denominator_invalid_rows": num_den_invalid,
    "ape_below_range_count": ape_below,
    "ape_above_range_count": ape_above,
    "final_pickle_filename": FINAL_PICKLE_PATH.name,
    "final_pickle_sha256": final_pkl_sha,
    "final_pickle_size_bytes": tmp_pkl_size,
    "run_manifest_filename": RUN_MANIFEST_PATH.name,
    "run_manifest_sha256": run_manifest_sha,
    "batch_details": batch_stats_list
}

if FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】FINAL_JSON_PATHが既に存在します。")

atomic_write_json_v2(final_q, FINAL_JSON_PATH, session_id)
print("✅ 最終品質JSON出力クリア")

# 【7. 最終再監査】
print("\n【7. 最終再監査】")
if not FINAL_PICKLE_PATH.exists() or not FINAL_JSON_PATH.exists():
    raise FileNotFoundError("【処理停止】最終成果物が存在しません。")

with open(FINAL_JSON_PATH, "r", encoding="utf-8") as f:
    re_q = json.load(f)

if get_file_sha256_v2(FINAL_PICKLE_PATH) != re_q["final_pickle_sha256"]:
    raise ValueError("【検証失敗】再監査: pickle SHA-256不一致")

re_pkl = joblib.load(FINAL_PICKLE_PATH)
if not isinstance(re_pkl, dict) or "target_dataframe" not in re_pkl:
    raise ValueError("【検証失敗】再監査: pickle形式が joblib mapping['target_dataframe'] ではありません。")

pd.testing.assert_frame_equal(re_pkl["target_dataframe"], final_df)

if re_q["total_output_rows"] != len(final_df):
    raise ValueError("【検証失敗】再監査: total_output_rowsが実行数と不一致")
if re_q["accepted_remark_1_rows"] + re_q["accepted_remark_2_rows"] != len(final_df):
    raise ValueError("【検証失敗】再監査: Remark合計と行数が不一致")
if re_q["source_key_duplicate_rows"] != 0:
    raise ValueError("【検証失敗】再監査: source_key_duplicate_rows != 0")

tmp_pickles = list(OUTPUT_DIR.glob("*.tmp.pkl"))
if tmp_pickles:
    raise RuntimeError("【処理停止】一時pickleが残存しています。")

if list(TEMP_DIR.iterdir()):
    raise RuntimeError("【処理停止】TEMP_DIRが空ではありません。")

print("✅ 最終再監査クリア。状態を更新します。")
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    final_rs = json.load(f)
final_rs["status"] = "finalized"
final_rs["last_update"] = datetime.now(timezone.utc).isoformat()
atomic_write_json_v2(final_rs, RESUME_STATE_PATH, session_id)

update_lock_v2("final_outputs_committed", session_id)

# 【最終結果表示】
print("\n=========================================")
print(f"FINAL_PICKLE_PATH: {FINAL_PICKLE_PATH}")
print(f"FINAL_JSON_PATH: {FINAL_JSON_PATH}")
print(f"pickle SHA-256: {final_pkl_sha}")
print(f"総行数: {len(final_df)}")
print(f"Remark 1行数: {accepted_rem_1}")
print(f"Remark 2行数: {accepted_rem_2}")
print(f"APE平均: {ape_stats['mean']:.4f}, 標準偏差: {ape_stats['std']:.4f}, 最小: {ape_stats['min']:.4f}, 最大: {ape_stats['max']:.4f}")
print(f"Datetime範囲: {final_q['datetime_min']} 〜 {final_q['datetime_max']}")
print(f"同一時刻重複対象行数: {dups_time_count}")
print(f"source由来キー重複数: {final_q['source_key_duplicate_rows']}")
print("最終監査結果: すべて成功。パイプラインが完全に終了しました。")


--- 第4段階: 最終結合・Pickle/JSON生成・監査 ---

【1. 最終化前監査】
✅ 最終化前監査クリア

【2. 全Parquet結合】

【3. 結合DataFrameの検証】
✅ 結合DataFrame検証クリア (行数: 76742)

【4, 5. Pickle形式作成・アトミック保存】
✅ Pickleのアトミック保存・検証クリア (SHA-256: f54c7ebc517610e1b0eda8e4fb4589a363e8f033fc14eadd087775bf24e10ff8)

【6. 最終品質JSONの作成】
✅ 最終品質JSON出力クリア

【7. 最終再監査】
✅ 最終再監査クリア。状態を更新します。

FINAL_PICKLE_PATH: /content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050.pkl
FINAL_JSON_PATH: /content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050_quality_report.json
pickle SHA-256: f54c7ebc517610e1b0eda8e4fb4589a363e8f033fc14eadd087775bf24e10ff8
総行数: 76742
Remark 1行数: 76742
Remark 2行数: 0
APE平均: 1.9118, 標準偏差: 0.0290, 最小: 1.7917, 最大: 2.1777
Datetime範囲: 2011-05-26T15:30:00 〜 2015-12-31T14:50:00
同一時刻重複対象行数: 0
source由来キー重複数: 0
最終監査結果: すべて成功。パイプラインが完全に終了しました。


In [34]:
import json
import os
import glob

print("--- 自動1バッチ試験＆残り全バッチ連続実行 ---")

# 【1. 実行前状態の保存・確認】
print("\n【1. 実行前状態の保存・確認】")
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

if list(TEMP_DIR.iterdir()):
    raise RuntimeError("【処理停止】TEMP_DIR が空ではありません。")

audit_committed_batches_v2(manifest_data)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs_initial = json.load(f)

initial_completed = set(rs_initial.get("completed_batch_ids", []))
initial_pending = rs_initial.get("pending_batch_ids", [])
initial_errors = set(p.name for p in ERROR_DIR.iterdir())

manifest_batches_order = list(manifest_data.get("batches", {}).keys())
expected_first_batch = None
for b in manifest_batches_order:
    if b in initial_pending:
        expected_first_batch = b
        break

print(f"開始前 completed_batches: {len(initial_completed)}")
print(f"開始前 pending_batch_ids: {len(initial_pending)}")
print(f"開始前 error files: {len(initial_errors)}")
print(f"1バッチ試験の対象(expected_first_batch): {expected_first_batch}")

if not expected_first_batch:
    print("処理すべきpendingバッチが存在しません。")
else:
    # 【2. 実行フラグと連続実行】
    try:
        EXECUTE_PENDING_BATCHES = True
        print("\n【3. 自動1バッチ試験】")
        run_pending_batches_v2(manifest_data, CONFIG, session_id, max_batches=1)

        # 1バッチ試験後の検証
        print("\n--- 1バッチ試験後の検証 ---")
        with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
            rs_test = json.load(f)

        test_completed = set(rs_test.get("completed_batch_ids", []))
        test_pending = rs_test.get("pending_batch_ids", [])
        test_errors = set(p.name for p in ERROR_DIR.iterdir())

        new_completed = test_completed - initial_completed

        checks_test = {
            "completed_increased_by_1": len(test_completed) == len(initial_completed) + 1,
            "pending_decreased_by_1": len(test_pending) == len(initial_pending) - 1,
            "new_completed_match": list(new_completed)[0] == expected_first_batch if len(new_completed) == 1 else False,
            "temp_dir_empty": len(list(TEMP_DIR.iterdir())) == 0,
            "no_new_errors": len(test_errors - initial_errors) == 0,
            "parquet_exists": (BATCH_DIR / f"{expected_first_batch}.parquet").exists(),
            "quality_json_exists": (BATCH_DIR / f"{expected_first_batch}_quality.json").exists(),
            "commit_json_exists": (BATCH_DIR / f"{expected_first_batch}_commit.json").exists(),
            "finals_not_exist": not FINAL_PICKLE_PATH.exists() and not FINAL_JSON_PATH.exists()
        }

        if not all(checks_test.values()):
            print(f"[失敗詳細] {checks_test}")
            raise RuntimeError("1バッチ試験後の検証に失敗しました。処理を停止します。")

        audit_committed_batches_v2(manifest_data)
        print("✅ 1バッチ試験と検証が成功しました。")

        if test_pending:
            print("\n【4. 残り全バッチ実行】")
            run_pending_batches_v2(manifest_data, CONFIG, session_id, max_batches=None)
        else:
            print("\n【4. 残り全バッチ実行】スキップ (全バッチ完了済み)")

    finally:
        EXECUTE_PENDING_BATCHES = False
        print(f"\n[finally] EXECUTE_PENDING_BATCHES = {EXECUTE_PENDING_BATCHES}")

# 【5. 全件終了後の確認】
print("\n【5. 全件終了後の確認】")
audit_committed_batches_v2(manifest_data)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    rs_final = json.load(f)
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    lock_final = json.load(f)

total_manifest_files = manifest_data.get("total_hsr_files", sum(len(v) for v in manifest_data.get("batches", {}).values()))
all_manifest_files = []
for files in manifest_data.get("batches", {}).values():
    all_manifest_files.extend(files)

final_errors = set(p.name for p in ERROR_DIR.iterdir())
comp_src = rs_final.get("completed_source_files", [])

checks_final = {
    "resume_status": rs_final.get("status") == "batches_complete",
    "lock_status": lock_final.get("status") == "all_batches_committed",
    "pending_empty": len(rs_final.get("pending_batch_ids", [])) == 0,
    "completed_batches_match": rs_final.get("completed_batches") == len(manifest_data.get("batches", {})),
    "completed_files_match": rs_final.get("completed_files") == total_manifest_files,
    "comp_src_no_dups": len(comp_src) == len(set(comp_src)),
    "comp_src_set_match": set(comp_src) == set(all_manifest_files),
    "temp_dir_empty": len(list(TEMP_DIR.iterdir())) == 0,
    "no_new_errors": len(final_errors - initial_errors) == 0,
    "finals_not_exist": not FINAL_PICKLE_PATH.exists() and not FINAL_JSON_PATH.exists()
}

if not all(checks_final.values()):
    print(f"[最終検証失敗詳細] {checks_final}")
    raise RuntimeError("全件終了後の検証に失敗しました。")

print("✅ 全件終了後の検証を通過しました。")

# 【最終表示】
print("\n=========================================")
print(f"完了バッチ数: {rs_final.get('completed_batches')}")
print(f"完了入力ファイル数: {rs_final.get('completed_files')}")
print(f"総APE行数 completed_output_rows: {rs_final.get('completed_output_rows', 'N/A')}")
print("Remark 1・2の集計は「最終結合時に算出」")
print(f"error JSON増加数: {len(final_errors - initial_errors)}")
print(f"TEMP_DIR残存数: {len(list(TEMP_DIR.iterdir()))}")
print(f"最終pickle未生成: {not FINAL_PICKLE_PATH.exists()}")
print("次工程: 最終結合")


--- 自動1バッチ試験＆残り全バッチ連続実行 ---

【1. 実行前状態の保存・確認】
開始前 completed_batches: 1
開始前 pending_batch_ids: 72
開始前 error files: 0
1バッチ試験の対象(expected_first_batch): batch_00000

【3. 自動1バッチ試験】
✅ batch_00000 完了: 採用行数=0 / 進捗: 完了=2, 残り=71
指定された max_batches (1) に達したため、バッチ処理ループを終了します。

--- 1バッチ試験後の検証 ---
✅ 1バッチ試験と検証が成功しました。

【4. 残り全バッチ実行】
✅ batch_00001 完了: 採用行数=0 / 進捗: 完了=3, 残り=70
✅ batch_00002 完了: 採用行数=0 / 進捗: 完了=4, 残り=69
✅ batch_00003 完了: 採用行数=0 / 進捗: 完了=5, 残り=68
✅ batch_00004 完了: 採用行数=0 / 進捗: 完了=6, 残り=67
✅ batch_00005 完了: 採用行数=238 / 進捗: 完了=7, 残り=66
✅ batch_00006 完了: 採用行数=1287 / 進捗: 完了=8, 残り=65
✅ batch_00007 完了: 採用行数=1428 / 進捗: 完了=9, 残り=64
✅ batch_00008 完了: 採用行数=1420 / 進捗: 完了=10, 残り=63
✅ batch_00009 完了: 採用行数=1213 / 進捗: 完了=11, 残り=62
✅ batch_00010 完了: 採用行数=1158 / 進捗: 完了=12, 残り=61
✅ batch_00011 完了: 採用行数=958 / 進捗: 完了=13, 残り=60
✅ batch_00012 完了: 採用行数=835 / 進捗: 完了=14, 残り=59
✅ batch_00013 完了: 採用行数=697 / 進捗: 完了=15, 残り=58
✅ batch_00014 完了: 採用行数=876 / 進捗: 完了=16, 残り=57
✅ batch_00015 完了: 採用行数=835 / 進捗: 完了=17, 残り=56
✅

In [31]:
print("--- 第2段階: stage_and_validate_batch_v2 の修正版 (rb+ 対応) ---")

def stage_and_validate_batch_v2(batch_id, manifest_data, config, sess_id):
    batch_files = manifest_data["batches"].get(batch_id)
    if not batch_files:
        raise ValueError(f"manifestにバッチが存在しません: {batch_id}")

    file_metadata = manifest_data.get("file_metadata", {})
    for rel_p in batch_files:
        if rel_p not in file_metadata:
            raise ValueError(f"メタデータが見つかりません: {rel_p}")
        full_path = INPUT_ROOT / rel_p
        st = full_path.stat()
        meta = file_metadata[rel_p]
        if st.st_size != meta["size"] or st.st_mtime_ns != meta["mtime_ns"]:
            raise ValueError(f"ファイルのサイズまたはmtimeが変更されています: {rel_p}")

    staging_dir = TEMP_DIR / f"{batch_id}_{sess_id}.staging"
    staging_dir.mkdir(parents=True, exist_ok=False)

    staging_parquet = staging_dir / f"{batch_id}.parquet"
    staging_json = staging_dir / f"{batch_id}_quality.json"

    start_time = datetime.now(timezone.utc)
    df, stats = process_batch_v2(batch_id, batch_files, config, sess_id)
    end_time = datetime.now(timezone.utc)

    parquet_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']

    if not df.empty:
        df = df[parquet_cols]
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        df['SiteNum'] = df['SiteNum'].astype('int64')
        df['SiteName'] = df['SiteName'].astype('string')
        df['Remark'] = df['Remark'].astype('int64')
        df['APE'] = df['APE'].astype('float64')
        df['source_file'] = df['source_file'].astype('string')
        df['source_row_index'] = df['source_row_index'].astype('int64')
        df['source_line_number'] = df['source_line_number'].astype('int64')
        df['batch_id'] = df['batch_id'].astype('string')
        df.to_parquet(staging_parquet, index=False)
    else:
        pd.DataFrame(columns=parquet_cols).astype({
            'Datetime': 'datetime64[ns]', 'SiteNum': 'int64', 'SiteName': 'string',
            'Remark': 'int64', 'APE': 'float64', 'source_file': 'string',
            'source_row_index': 'int64', 'source_line_number': 'int64', 'batch_id': 'string'
        }).to_parquet(staging_parquet, index=False)

    # 【修正】読み取り専用時の OSError 回避のため rb+ を使用
    with open(staging_parquet, "rb+") as f:
        f.flush()
        os.fsync(f.fileno())

    loaded_df = pd.read_parquet(staging_parquet)

    if list(loaded_df.columns) != parquet_cols:
        raise ValueError("Parquet列構造不一致")

    expected_dtypes = {
        'Datetime': 'datetime64[ns]', 'SiteNum': 'int64', 'SiteName': 'string',
        'Remark': 'int64', 'APE': 'float64', 'source_file': 'string',
        'source_row_index': 'int64', 'source_line_number': 'int64', 'batch_id': 'string'
    }
    for col, exp_type in expected_dtypes.items():
        if str(loaded_df[col].dtype) != exp_type:
            raise ValueError(f"dtype不一致: {col} は {loaded_df[col].dtype} です。期待値は {exp_type}")

    if not loaded_df.empty and loaded_df['Datetime'].dt.tz is not None:
        raise ValueError("Datetime列がtimezone-naiveではありません")

    if len(loaded_df) != len(df):
        raise ValueError("Parquet行数不一致")

    if loaded_df[parquet_cols].isna().sum().sum() > 0:
         raise ValueError("主要9列に欠損値が存在します")

    if not loaded_df['Remark'].isin(config["remarks_used"]).all():
        raise ValueError("Remark列にconfig['remarks_used']以外の値が含まれています")

    if not loaded_df['source_file'].isin(batch_files).all():
        raise ValueError("source_file列にbatch_files以外の値が含まれています")

    if not loaded_df.empty:
        if not (loaded_df['batch_id'] == batch_id).all():
            raise ValueError("batch_id不一致")
        if (loaded_df['APE'] < config['ape_valid_range_eV'][0]).sum() > 0 or (loaded_df['APE'] > config['ape_valid_range_eV'][1]).sum() > 0:
            raise ValueError("APE範囲外が存在")

    pq_sha256 = get_file_sha256_v2(staging_parquet)

    def sum_stats(key):
        return sum(s.get(key, 0) for s in stats.values())

    rem_counts_all = collections.Counter()
    for s in stats.values():
        rem_counts_all.update(s.get('remark_counts', {}))

    quality_data = {
        "quality_schema_version": "1.0",
        "pipeline_version": config["pipeline_version"],
        "config_hash": manifest_data["config_hash"],
        "session_id": sess_id,
        "batch_id": batch_id,
        "started_at": start_time.isoformat(),
        "completed_at": end_time.isoformat(),
        "input_file_count": len(batch_files),
        "input_files": batch_files,
        "file_metadata": {p: {"size": file_metadata[p]["size"], "mtime_ns": file_metadata[p]["mtime_ns"]} for p in batch_files},
        "raw_rows": sum_stats('raw_rows'),
        "remark_counts": {str(k): v for k, v in rem_counts_all.items()},
        "remark_1_count": sum_stats('remark_1_count'),
        "remark_2_count": sum_stats('remark_2_count'),
        "target_350_1050_valid_rows": sum_stats('target_350_1050_valid_rows'),
        "full_350_1700_valid_rows": sum_stats('full_350_1700_valid_rows'),
        "target_valid_but_full_invalid_rows": sum_stats('target_valid_but_full_invalid_rows'),
        "remark_valid_but_target_invalid_rows": sum_stats('remark_valid_but_target_invalid_rows'),
        "numerator_or_denominator_invalid_rows": sum_stats('numerator_or_denominator_invalid_rows'),
        "ape_below_1_2": sum_stats('ape_below_1_2'),
        "ape_in_range": sum_stats('ape_in_range'),
        "ape_above_2_2": sum_stats('ape_above_2_2'),
        "accepted_rows": len(loaded_df),
        "minus_9999_cells_target": sum_stats('minus_9999_cells_target'),
        "minus_9999_cells_full": sum_stats('minus_9999_cells_full'),
        "other_negative_cells_target": sum_stats('other_negative_cells_target'),
        "NaN_cells_target": sum_stats('NaN_cells_target'),
        "24_00_rows": sum_stats('24_00_rows'),
        "datetime_parse_failures": sum_stats('datetime_parse_failures'),
        "timestamp_duplicate_rows": sum_stats('timestamp_duplicate_rows'),
        "source_key_duplicate_rows": int(loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum()),
        "date_or_site_mismatches": sum_stats('date_or_site_mismatches'),
        "processing_error_count": 0,
        "file_statistics": stats,
        "ape_statistics": loaded_df['APE'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict() if not loaded_df.empty else {},
        "parquet_schema": {col: str(dtype) for col, dtype in loaded_df.dtypes.items()},
        "parquet_sha256": pq_sha256
    }

    atomic_write_json_v2(quality_data, staging_json, sess_id)
    return staging_dir, staging_parquet, staging_json, quality_data, pq_sha256

print("✅ stage_and_validate_batch_v2 が修正され、再定義されました (ダミー・passなし)")


--- 第2段階: stage_and_validate_batch_v2 の修正版 (rb+ 対応) ---
✅ stage_and_validate_batch_v2 が修正され、再定義されました (ダミー・passなし)


In [28]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import traceback
import sys
import re
import math
from pathlib import Path
import collections

print("--- 第1段階: 汎用関数(v2)の完全実装 (修正反映版) ---")

def make_json_safe_v2(obj, path="root"):
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            str_k = str(k)
            if str_k in new_dict:
                raise ValueError(f"辞書キーの文字列化衝突が発生しました: {path}.{str_k}")
            new_dict[str_k] = make_json_safe_v2(v, f"{path}.{k}")
        return new_dict
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe_v2(item, f"{path}[{i}]") for i, item in enumerate(obj)]
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif isinstance(obj, (np.integer, int)):
        return int(obj)
    elif isinstance(obj, (np.floating, float)):
        if math.isnan(obj) or math.isinf(obj):
            raise ValueError(f"NaN/inf は JSON に保存できません: {path}")
        return float(obj)
    elif isinstance(obj, (datetime, pd.Timestamp)):
        return obj.isoformat()
    elif pd.isna(obj):
        return None
    else:
        return obj

def parse_hsr_header_v2(filepath):
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, nrows=0)
    cols = list(df.columns)
    if len(cols) != 1356:
        raise ValueError(f"全列数が1356ではありません: {len(cols)}")
    if cols[:5] != ["地点番号", "地点名", "年月日", "時分", "リマーク"]:
        raise ValueError(f"先頭5列構造不一致: {cols[:5]}")

    w_cols = []
    for c in cols[5:]:
        m = re.match(r"^日射強度\((\d+)nm\)\[W/m2/μm\]$", c)
        if not m:
            raise ValueError(f"余分な列または予期せぬ列名: {c}")
        w_cols.append({'name': c, 'wave': int(m.group(1))})

    if len(w_cols) != 1351:
        raise ValueError(f"波長列数が1351ではありません: {len(w_cols)}")
    waves = [x['wave'] for x in w_cols]
    expected_waves = list(range(350, 1701))
    if waves != expected_waves:
        raise ValueError("波長順序または重複・欠落エラー")

    target_w = [x['name'] for x in w_cols if 350 <= x['wave'] <= 1050]
    full_w = [x['name'] for x in w_cols if 350 <= x['wave'] <= 1700]

    if len(target_w) != 701:
        raise ValueError(f"APE対象列数が701ではありません: {len(target_w)}")

    return target_w, full_w

def parse_datetime_columns_v2(df, file_yymmdd):
    pts = []
    failures = 0
    c_2400 = 0
    date_mismatches = 0
    for idx, r in df.iterrows():
        try:
            ymd = str(r['年月日']).strip()
            hm = str(r['時分']).strip()
            bd = datetime.strptime(ymd, "%Y/%m/%d")
            if bd.strftime("%y%m%d") != file_yymmdd:
                date_mismatches += 1
            if hm == "24:00":
                pts.append(bd + timedelta(days=1))
                c_2400 += 1
            else:
                h, m_str = hm.split(':')
                pts.append(bd.replace(hour=int(h), minute=int(m_str)))
        except Exception:
            failures += 1
            pts.append(pd.NaT)
    df['Datetime'] = pts
    return df, failures, c_2400, date_mismatches

def calculate_ape_for_file_v2(filepath, rel_path, config):
    target_w, full_w = parse_hsr_header_v2(filepath)
    df = pd.read_csv(filepath, encoding="shift_jis", header=0, low_memory=False)

    m_file = re.fullmatch(r"10HSR(\d{6})_(\d{3})\.csv", Path(filepath).name)
    if not m_file:
        raise ValueError(f"ファイル名形式エラー: {Path(filepath).name}")
    f_yymmdd, f_site = m_file.group(1), m_file.group(2)

    df['source_file'] = rel_path
    df['source_row_index'] = df.index
    df['source_line_number'] = df.index + 2
    df['remark_num'] = pd.to_numeric(df['リマーク'], errors='coerce')

    site_mismatches = (df['地点番号'].astype(str) != f_site).sum()
    if f_site not in [str(x) for x in config["expected_site_numbers"]]:
        raise ValueError(f"期待されない地点番号: {f_site}")

    df, parse_fails, c_2400, date_mismatches = parse_datetime_columns_v2(df, f_yymmdd)

    key_dups = df.duplicated(subset=['source_file', 'source_line_number']).sum()
    valid_dt_mask = df['Datetime'].notna()
    time_dups = df[valid_dt_mask].duplicated(subset=['Datetime'], keep=False).sum()

    if parse_fails > 0 or date_mismatches > 0 or site_mismatches > 0 or key_dups > 0:
        raise ValueError(f"ファイル不整合 ({rel_path}): parse_fails={parse_fails}, date_mismatches={date_mismatches}, site_mismatches={site_mismatches}, key_dups={key_dups}")

    rem_counts = df['remark_num'].value_counts().to_dict()
    m_rem = df['remark_num'].isin(config["remarks_used"])
    m_site = df['地点番号'] == int(f_site)

    arr_t = df[target_w].to_numpy(dtype=np.float64)
    m_val_t = np.isfinite(arr_t).all(axis=1) & (arr_t != -9999).all(axis=1)
    m9999_t = (arr_t == -9999).sum()
    nan_t = np.isnan(arr_t).sum()

    neg_mask = (arr_t < 0) & (arr_t != -9999)
    neg_t = neg_mask.sum()
    other_negative_records = []
    if neg_mask.any():
        neg_rows = df[neg_mask.any(axis=1)].head(20)
        for idx, row in neg_rows.iterrows():
            row_arr = arr_t[idx]
            neg_vals = row_arr[(row_arr < 0) & (row_arr != -9999)]
            if len(neg_vals) > 0:
                other_negative_records.append({
                    'source_file': str(row['source_file']),
                    'source_line_number': int(row['source_line_number']),
                    '最小負値': float(neg_vals.min()),
                    '負値セル数': int(len(neg_vals))
                })

    arr_f = df[full_w].to_numpy(dtype=np.float64)
    m_val_f = np.isfinite(arr_f).all(axis=1) & (arr_f != -9999).all(axis=1)
    m9999_f = (arr_f == -9999).sum()

    m_adopt = m_site & m_rem & valid_dt_mask & m_val_t
    adopted = df[m_adopt].copy()

    ape_below, ape_in, ape_above = 0, 0, 0
    num_den_invalid = 0
    bad_recs = []

    if not adopted.empty:
        waves = np.arange(350, 1051, dtype=np.float64)
        G = adopted[target_w].to_numpy(dtype=np.float64)
        num = np.sum(G, axis=1)
        den = np.sum(G * waves[None, :], axis=1)

        m_nd = np.isfinite(num) & np.isfinite(den) & (num > 0) & (den > 0)
        num_den_invalid = (~m_nd).sum()

        if num_den_invalid > 0:
            bad_df = adopted[~m_nd].copy()
            bad_df['num'] = num[~m_nd]
            bad_df['den'] = den[~m_nd]
            for _, r in bad_df.iterrows():
                bad_recs.append({
                    'source_file': r['source_file'],
                    'source_line_number': int(r['source_line_number']),
                    'Remark': int(r['remark_num']),
                    'numerator': float(r['num']),
                    'denominator': float(r['den']),
                    '異常理由': 'num/den invalid'
                })
            adopted = adopted[m_nd].copy()
            num = num[m_nd]
            den = den[m_nd]

        if not adopted.empty:
            APE = 1239.84193 * num / den
            adopted['APE'] = APE

            APE_chk = 1239.84193 / (den / num)
            max_d = np.abs(APE - APE_chk).max()
            if max_d > 1e-12:
                raise ValueError(f"APE差分大: {max_d}")

            ape_below = (APE < config['ape_valid_range_eV'][0]).sum()
            ape_in = ((APE >= config['ape_valid_range_eV'][0]) & (APE <= config['ape_valid_range_eV'][1])).sum()
            ape_above = (APE > config['ape_valid_range_eV'][1]).sum()
            adopted = adopted[(adopted['APE'] >= config['ape_valid_range_eV'][0]) & (adopted['APE'] <= config['ape_valid_range_eV'][1])]

    cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number']

    if adopted.empty:
        out_df = pd.DataFrame({
            'Datetime': pd.Series(dtype='datetime64[ns]'),
            'SiteNum': pd.Series(dtype='int64'),
            'SiteName': pd.Series(dtype='string'),
            'Remark': pd.Series(dtype='int64'),
            'APE': pd.Series(dtype='float64'),
            'source_file': pd.Series(dtype='string'),
            'source_row_index': pd.Series(dtype='int64'),
            'source_line_number': pd.Series(dtype='int64'),
        })
    else:
        out_df = adopted[['Datetime', '地点番号', '地点名', 'remark_num', 'APE', 'source_file', 'source_row_index', 'source_line_number']].rename(columns={'地点番号':'SiteNum', '地点名':'SiteName', 'remark_num':'Remark'})
        out_df['SiteNum'] = out_df['SiteNum'].astype('int64')
        out_df['SiteName'] = out_df['SiteName'].astype('string')
        out_df['Remark'] = out_df['Remark'].astype('int64')
        out_df['APE'] = out_df['APE'].astype('float64')
        out_df['source_file'] = out_df['source_file'].astype('string')
        out_df['source_row_index'] = out_df['source_row_index'].astype('int64')
        out_df['source_line_number'] = out_df['source_line_number'].astype('int64')

    stats = {
        'raw_rows': len(df),
        'remark_counts': rem_counts,
        'remark_1_count': int(rem_counts.get(1, 0)),
        'remark_2_count': int(rem_counts.get(2, 0)),
        'target_350_1050_valid_rows': int(m_val_t.sum()),
        'full_350_1700_valid_rows': int(m_val_f.sum()),
        'target_valid_but_full_invalid_rows': int((m_val_t & ~m_val_f).sum()),
        'remark_valid_but_target_invalid_rows': int((m_rem & ~m_val_t).sum()),
        'minus_9999_cells_target': int(m9999_t),
        'minus_9999_cells_full': int(m9999_f),
        'other_negative_cells_target': int(neg_t),
        'other_negative_records': other_negative_records,
        'NaN_cells_target': int(nan_t),
        'numerator_or_denominator_invalid_rows': int(num_den_invalid),
        'invalid_num_den_records': bad_recs,
        'ape_below_1_2': int(ape_below),
        'ape_in_range': int(ape_in),
        'ape_above_2_2': int(ape_above),
        'accepted_rows': len(out_df),
        '24_00_rows': c_2400,
        'datetime_parse_failures': parse_fails,
        'date_or_site_mismatches': date_mismatches + site_mismatches,
        'timestamp_duplicate_rows': int(time_dups),
        'source_key_duplicate_rows': int(key_dups)
    }
    return out_df, stats

def process_batch_v2(batch_id, rel_paths, config, sess_id):
    all_dfs = []
    all_stats = {}
    error_records = []
    for i, rel_p in enumerate(rel_paths):
        if i > 0 and i % 5 == 0:
            update_lock_v2(f"processing_{batch_id}_file_{i}", sess_id)
        try:
            df, stats = calculate_ape_for_file_v2(INPUT_ROOT / rel_p, rel_p, config)
            df['batch_id'] = str(batch_id)
            df['batch_id'] = df['batch_id'].astype('string')
            all_dfs.append(df)
            all_stats[rel_p] = stats
        except Exception as e:
            err_info = {
                "source_file": rel_p,
                "exception_type": type(e).__name__,
                "message": str(e),
                "traceback": traceback.format_exc()
            }
            error_records.append(err_info)

    if error_records:
        err_msgs = [f"File: {e['source_file']}, Error: {e['exception_type']} - {e['message']}\n{e['traceback']}" for e in error_records]
        raise RuntimeError(f"バッチ内エラー発生:\n" + "\n---\n".join(err_msgs))

    valid_dfs = [d for d in all_dfs if not d.empty]
    if valid_dfs:
        final_df = pd.concat(valid_dfs, ignore_index=True)
        final_df = final_df.sort_values(by=['Datetime', 'source_file', 'source_line_number'], kind='stable', ignore_index=True)
    else:
        final_df = pd.DataFrame({
            'Datetime': pd.Series(dtype='datetime64[ns]'),
            'SiteNum': pd.Series(dtype='int64'),
            'SiteName': pd.Series(dtype='string'),
            'Remark': pd.Series(dtype='int64'),
            'APE': pd.Series(dtype='float64'),
            'source_file': pd.Series(dtype='string'),
            'source_row_index': pd.Series(dtype='int64'),
            'source_line_number': pd.Series(dtype='int64'),
            'batch_id': pd.Series(dtype='string')
        })

    if final_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
        raise RuntimeError("バッチ内由来キー重複")

    return final_df, all_stats

print("✅ make_json_safe_v2 が定義されました (ダミー・passなし)")
print("✅ parse_hsr_header_v2 が定義されました (ダミー・passなし)")
print("✅ parse_datetime_columns_v2 が定義されました (ダミー・passなし)")
print("✅ calculate_ape_for_file_v2 が定義されました (ダミー・passなし)")
print("✅ process_batch_v2 が定義されました (ダミー・passなし)")
print("5関数が完全に実装されました。次段階へ進めます。")


--- 第1段階: 汎用関数(v2)の完全実装 (修正反映版) ---
✅ make_json_safe_v2 が定義されました (ダミー・passなし)
✅ parse_hsr_header_v2 が定義されました (ダミー・passなし)
✅ parse_datetime_columns_v2 が定義されました (ダミー・passなし)
✅ calculate_ape_for_file_v2 が定義されました (ダミー・passなし)
✅ process_batch_v2 が定義されました (ダミー・passなし)
5関数が完全に実装されました。次段階へ進めます。


In [30]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import collections
import uuid

print("--- 第2段階: 保存・commit・再開状態関数の完全実装 (監査指摘反映版) ---")

def write_error_record_v2(batch_id, error_info, sess_id):
    now_utc_str = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f")
    unique_id = str(uuid.uuid4())[:8]
    error_filename = f"{batch_id}_{sess_id}_{now_utc_str}_{unique_id}_error.json"
    error_file = ERROR_DIR / error_filename

    if error_file.exists():
        raise FileExistsError(f"エラーファイルが既に存在します: {error_file}")

    error_info_copy = dict(error_info)
    error_info_copy["batch_id"] = batch_id
    error_info_copy["session_id"] = sess_id
    error_info_copy["created_at"] = datetime.now(timezone.utc).isoformat()

    atomic_write_json_v2(error_info_copy, error_file, sess_id)

def stage_and_validate_batch_v2(batch_id, manifest_data, config, sess_id):
    batch_files = manifest_data["batches"].get(batch_id)
    if not batch_files:
        raise ValueError(f"manifestにバッチが存在しません: {batch_id}")

    file_metadata = manifest_data.get("file_metadata", {})
    for rel_p in batch_files:
        if rel_p not in file_metadata:
            raise ValueError(f"メタデータが見つかりません: {rel_p}")
        full_path = INPUT_ROOT / rel_p
        st = full_path.stat()
        meta = file_metadata[rel_p]
        if st.st_size != meta["size"] or st.st_mtime_ns != meta["mtime_ns"]:
            raise ValueError(f"ファイルのサイズまたはmtimeが変更されています: {rel_p}")

    staging_dir = TEMP_DIR / f"{batch_id}_{sess_id}.staging"
    staging_dir.mkdir(parents=True, exist_ok=False)

    staging_parquet = staging_dir / f"{batch_id}.parquet"
    staging_json = staging_dir / f"{batch_id}_quality.json"

    start_time = datetime.now(timezone.utc)
    df, stats = process_batch_v2(batch_id, batch_files, config, sess_id)
    end_time = datetime.now(timezone.utc)

    parquet_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']

    if not df.empty:
        df = df[parquet_cols]
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        df['SiteNum'] = df['SiteNum'].astype('int64')
        df['SiteName'] = df['SiteName'].astype('string')
        df['Remark'] = df['Remark'].astype('int64')
        df['APE'] = df['APE'].astype('float64')
        df['source_file'] = df['source_file'].astype('string')
        df['source_row_index'] = df['source_row_index'].astype('int64')
        df['source_line_number'] = df['source_line_number'].astype('int64')
        df['batch_id'] = df['batch_id'].astype('string')
        df.to_parquet(staging_parquet, index=False)
    else:
        pd.DataFrame(columns=parquet_cols).astype({
            'Datetime': 'datetime64[ns]', 'SiteNum': 'int64', 'SiteName': 'string',
            'Remark': 'int64', 'APE': 'float64', 'source_file': 'string',
            'source_row_index': 'int64', 'source_line_number': 'int64', 'batch_id': 'string'
        }).to_parquet(staging_parquet, index=False)

    with open(staging_parquet, "rb") as f:
        f.flush()
        os.fsync(f.fileno())

    loaded_df = pd.read_parquet(staging_parquet)

    if list(loaded_df.columns) != parquet_cols:
        raise ValueError("Parquet列構造不一致")

    expected_dtypes = {
        'Datetime': 'datetime64[ns]', 'SiteNum': 'int64', 'SiteName': 'string',
        'Remark': 'int64', 'APE': 'float64', 'source_file': 'string',
        'source_row_index': 'int64', 'source_line_number': 'int64', 'batch_id': 'string'
    }
    for col, exp_type in expected_dtypes.items():
        if str(loaded_df[col].dtype) != exp_type:
            raise ValueError(f"dtype不一致: {col} は {loaded_df[col].dtype} です。期待値は {exp_type}")

    if not loaded_df.empty and loaded_df['Datetime'].dt.tz is not None:
        raise ValueError("Datetime列がtimezone-naiveではありません")

    if len(loaded_df) != len(df):
        raise ValueError("Parquet行数不一致")

    if loaded_df[parquet_cols].isna().sum().sum() > 0:
         raise ValueError("主要9列に欠損値が存在します")

    if not loaded_df['Remark'].isin(config["remarks_used"]).all():
        raise ValueError("Remark列にconfig['remarks_used']以外の値が含まれています")

    if not loaded_df['source_file'].isin(batch_files).all():
        raise ValueError("source_file列にbatch_files以外の値が含まれています")

    if not loaded_df.empty:
        if not (loaded_df['batch_id'] == batch_id).all():
            raise ValueError("batch_id不一致")
        if (loaded_df['APE'] < config['ape_valid_range_eV'][0]).sum() > 0 or (loaded_df['APE'] > config['ape_valid_range_eV'][1]).sum() > 0:
            raise ValueError("APE範囲外が存在")

    pq_sha256 = get_file_sha256_v2(staging_parquet)

    def sum_stats(key):
        return sum(s.get(key, 0) for s in stats.values())

    rem_counts_all = collections.Counter()
    for s in stats.values():
        rem_counts_all.update(s.get('remark_counts', {}))

    quality_data = {
        "quality_schema_version": "1.0",
        "pipeline_version": config["pipeline_version"],
        "config_hash": manifest_data["config_hash"],
        "session_id": sess_id,
        "batch_id": batch_id,
        "started_at": start_time.isoformat(),
        "completed_at": end_time.isoformat(),
        "input_file_count": len(batch_files),
        "input_files": batch_files,
        "file_metadata": {p: {"size": file_metadata[p]["size"], "mtime_ns": file_metadata[p]["mtime_ns"]} for p in batch_files},
        "raw_rows": sum_stats('raw_rows'),
        "remark_counts": {str(k): v for k, v in rem_counts_all.items()},
        "remark_1_count": sum_stats('remark_1_count'),
        "remark_2_count": sum_stats('remark_2_count'),
        "target_350_1050_valid_rows": sum_stats('target_350_1050_valid_rows'),
        "full_350_1700_valid_rows": sum_stats('full_350_1700_valid_rows'),
        "target_valid_but_full_invalid_rows": sum_stats('target_valid_but_full_invalid_rows'),
        "remark_valid_but_target_invalid_rows": sum_stats('remark_valid_but_target_invalid_rows'),
        "numerator_or_denominator_invalid_rows": sum_stats('numerator_or_denominator_invalid_rows'),
        "ape_below_1_2": sum_stats('ape_below_1_2'),
        "ape_in_range": sum_stats('ape_in_range'),
        "ape_above_2_2": sum_stats('ape_above_2_2'),
        "accepted_rows": len(loaded_df),
        "minus_9999_cells_target": sum_stats('minus_9999_cells_target'),
        "minus_9999_cells_full": sum_stats('minus_9999_cells_full'),
        "other_negative_cells_target": sum_stats('other_negative_cells_target'),
        "NaN_cells_target": sum_stats('NaN_cells_target'),
        "24_00_rows": sum_stats('24_00_rows'),
        "datetime_parse_failures": sum_stats('datetime_parse_failures'),
        "timestamp_duplicate_rows": sum_stats('timestamp_duplicate_rows'),
        "source_key_duplicate_rows": int(loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum()),
        "date_or_site_mismatches": sum_stats('date_or_site_mismatches'),
        "processing_error_count": 0,
        "file_statistics": stats,
        "ape_statistics": loaded_df['APE'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict() if not loaded_df.empty else {},
        "parquet_schema": {col: str(dtype) for col, dtype in loaded_df.dtypes.items()},
        "parquet_sha256": pq_sha256
    }

    atomic_write_json_v2(quality_data, staging_json, sess_id)
    return staging_dir, staging_parquet, staging_json, quality_data, pq_sha256

def publish_and_commit_batch_v2(batch_id, staging_dir, staging_parquet, staging_json, quality_data, pq_sha256, config_hash, sess_id):
    batch_parquet_path = BATCH_DIR / f"{batch_id}.parquet"
    batch_json_path = BATCH_DIR / f"{batch_id}_quality.json"
    batch_commit_path = BATCH_DIR / f"{batch_id}_commit.json"

    if batch_parquet_path.exists() or batch_json_path.exists() or batch_commit_path.exists():
        raise FileExistsError("公開直前に正式ファイルが既に存在します。")

    os.replace(staging_parquet, batch_parquet_path)
    os.replace(staging_json, batch_json_path)

    final_pq_sha = get_file_sha256_v2(batch_parquet_path)
    if final_pq_sha != pq_sha256:
        raise ValueError("公開後Parquet SHA不一致")
    with open(batch_json_path, "r", encoding="utf-8") as f:
        final_q = json.load(f)
    if final_q != make_json_safe_v2(quality_data):
        raise ValueError("公開後Quality JSON内容不一致")

    commit_data = {
        "commit_schema_version": "1.0",
        "batch_id": batch_id,
        "config_hash": config_hash,
        "session_id": sess_id,
        "committed_at": datetime.now(timezone.utc).isoformat(),
        "parquet_filename": batch_parquet_path.name,
        "parquet_sha256": final_pq_sha,
        "quality_filename": batch_json_path.name,
        "quality_sha256": get_file_sha256_v2(batch_json_path),
        "accepted_rows": quality_data["accepted_rows"],
        "input_file_count": quality_data["input_file_count"]
    }

    atomic_write_json_v2(commit_data, batch_commit_path, sess_id)

    remaining_files = list(staging_dir.iterdir())
    if remaining_files:
        raise RuntimeError(f"staging_dir内に想定外のファイルが残っています: {remaining_files}")

    staging_dir.rmdir()

def audit_committed_batches_v2(manifest_data):
    pqs = {p.stem for p in BATCH_DIR.glob("*.parquet")}
    qjs = {p.name.replace("_quality.json", "") for p in BATCH_DIR.glob("*_quality.json")}
    cjs = {p.name.replace("_commit.json", "") for p in BATCH_DIR.glob("*_commit.json")}

    all_ids = pqs | qjs | cjs
    if not all_ids:
        return []

    if pqs != all_ids or qjs != all_ids or cjs != all_ids:
        raise RuntimeError(f"孤立成果物検出: pqs={pqs}, qjs={qjs}, cjs={cjs}")

    valid_batches = []
    for b_id in sorted(all_ids):
        if b_id not in manifest_data["batches"]:
             raise ValueError(f"{b_id} がmanifestのbatchesに存在しません")

        c_path = BATCH_DIR / f"{b_id}_commit.json"
        q_path = BATCH_DIR / f"{b_id}_quality.json"
        pq_path = BATCH_DIR / f"{b_id}.parquet"

        with open(c_path, "r", encoding="utf-8") as f:
            c_data = json.load(f)
        with open(q_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)

        if c_data.get("commit_schema_version") != "1.0": raise ValueError(f"{b_id} commit_schema_version不一致")
        if q_data.get("quality_schema_version") != "1.0": raise ValueError(f"{b_id} quality_schema_version不一致")
        if c_data["batch_id"] != b_id: raise ValueError(f"{b_id} commit内batch_id不一致")
        if q_data["batch_id"] != b_id: raise ValueError(f"{b_id} Quality JSON内batch_id不一致")

        conf_hash = manifest_data["config_hash"]
        if c_data["config_hash"] != conf_hash or q_data["config_hash"] != conf_hash:
            raise ValueError(f"{b_id} config_hash不一致")

        if c_data["parquet_filename"] != pq_path.name: raise ValueError(f"{b_id} parquet_filename異常")
        if c_data["quality_filename"] != q_path.name: raise ValueError(f"{b_id} quality_filename異常")

        pq_sha = get_file_sha256_v2(pq_path)
        if pq_sha != c_data["parquet_sha256"]: raise ValueError(f"{b_id} Parquet SHA不一致(commit)")
        if pq_sha != q_data["parquet_sha256"]: raise ValueError(f"{b_id} Parquet SHA不一致(Quality JSON)")

        q_sha = get_file_sha256_v2(q_path)
        if q_sha != c_data["quality_sha256"]: raise ValueError(f"{b_id} Quality JSON SHA不一致")

        pq_df = pd.read_parquet(pq_path)
        expected_files = manifest_data["batches"][b_id]

        if len(pq_df) != c_data["accepted_rows"] or len(pq_df) != q_data["accepted_rows"]:
             raise ValueError(f"{b_id} 行数不一致")

        if c_data["input_file_count"] != len(expected_files) or q_data["input_file_count"] != len(expected_files):
            raise ValueError(f"{b_id} 入力ファイル数異常")

        if q_data["input_files"] != expected_files:
             raise ValueError(f"{b_id} input_files順序・内容不一致")

        if not pq_df.empty and not (pq_df['batch_id'] == b_id).all():
             raise ValueError(f"{b_id} Parquet内のbatch_id不一致")

        if pq_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
             raise ValueError(f"{b_id} source_file/source_line_number重複あり")

        valid_batches.append(b_id)

    return valid_batches

def rebuild_expected_resume_v2(manifest_data, valid_batches):
    expected_completed_files = 0
    expected_output_rows = 0
    expected_src_files = []

    manifest_batches = list(manifest_data["batches"].keys())
    ordered_valid_batches = [b for b in manifest_batches if b in valid_batches]

    for b_id in ordered_valid_batches:
        q_path = BATCH_DIR / f"{b_id}_quality.json"
        with open(q_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)
        expected_output_rows += q_data["accepted_rows"]
        expected_completed_files += q_data["input_file_count"]
        expected_src_files.extend(manifest_data["batches"][b_id])

    if len(expected_src_files) != len(set(expected_src_files)):
         raise ValueError("completed_source_filesに重複が存在します")
    if len(expected_src_files) != expected_completed_files:
         raise ValueError("completed_source_filesの要素数とcompleted_filesが一致しません")

    expected_pending = [b for b in manifest_batches if b not in ordered_valid_batches]

    return {
        "completed_batch_ids": ordered_valid_batches,
        "pending_batch_ids": expected_pending,
        "completed_source_files": expected_src_files,
        "completed_batches": len(ordered_valid_batches),
        "completed_files": expected_completed_files,
        "completed_output_rows": expected_output_rows
    }

def update_resume_from_commits_v2(manifest_data, sess_id):
    with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
        rs = json.load(f)

    if rs.get("current_session_id") != sess_id:
        raise RuntimeError(f"resume_stateのcurrent_session_id不一致: {rs.get('current_session_id')}")
    if rs.get("config_hash") != manifest_data["config_hash"]:
        raise RuntimeError("resume_stateのconfig_hash不一致")

    valid_batches = audit_committed_batches_v2(manifest_data)
    rebuilt = rebuild_expected_resume_v2(manifest_data, valid_batches)

    rs["total_batches"] = len(manifest_data["batches"])
    rs["total_files"] = manifest_data.get("total_hsr_files", sum(len(v) for v in manifest_data["batches"].values()))
    rs["completed_batch_ids"] = rebuilt["completed_batch_ids"]
    rs["pending_batch_ids"] = rebuilt["pending_batch_ids"]
    rs["completed_source_files"] = rebuilt["completed_source_files"]
    rs["completed_batches"] = rebuilt["completed_batches"]
    rs["completed_files"] = rebuilt["completed_files"]
    rs["completed_output_rows"] = rebuilt["completed_output_rows"]

    if valid_batches:
        rs["last_completed_batch"] = rebuilt["completed_batch_ids"][-1]
        rs["last_completed_source_file"] = rebuilt["completed_source_files"][-1]
    else:
        rs["last_completed_batch"] = None
        rs["last_completed_source_file"] = None

    rs["last_update"] = datetime.now(timezone.utc).isoformat()

    atomic_write_json_v2(rs, RESUME_STATE_PATH, sess_id)

print("✅ 第2段階の6関数が修正され、定義されました (ダミー・passなし)")


--- 第2段階: 保存・commit・再開状態関数の完全実装 (監査指摘反映版) ---
✅ 第2段階の6関数が修正され、定義されました (ダミー・passなし)


In [29]:
import json
import os
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
import collections
import traceback

print("--- 第2段階: 保存・commit・再開状態関数の完全実装 ---")

# 補助関数
def get_file_sha256_v2(filepath):
    with open(filepath, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def atomic_write_json_v2(data_dict, target_path, sess_id):
    safe_data = make_json_safe_v2(data_dict)
    temp_name = f"{target_path.stem}_{sess_id}.tmp.json"
    temp_path = target_path.parent / temp_name

    if temp_path.exists():
        raise FileExistsError(f"一時ファイルが既に存在します。上書きは禁止されています: {temp_path}")

    with open(temp_path, "x", encoding="utf-8") as f:
        json.dump(safe_data, f, indent=4, ensure_ascii=False, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())

    with open(temp_path, "r", encoding="utf-8") as f:
        read_back = json.load(f)
    if read_back != safe_data:
        raise ValueError(f"一時ファイルの内容が元の辞書と一致しません: {temp_path}")

    os.replace(temp_path, target_path)

def update_lock_v2(status_str, sess_id):
    now_utc = datetime.now(timezone.utc).isoformat()
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        lock = json.load(f)
    if lock.get("session_id") != sess_id:
        raise RuntimeError(f"所有権が失われました。現在のsession_id: {lock.get('session_id')}")
    lock["heartbeat"] = now_utc
    lock["status"] = status_str
    atomic_write_json_v2(lock, PROCESSING_LOCK_PATH, sess_id)

def write_error_record_v2(batch_id, error_info, sess_id):
    error_file = ERROR_DIR / f"{batch_id}_{sess_id}_error.json"
    atomic_write_json_v2(error_info, error_file, sess_id)

def stage_and_validate_batch_v2(batch_id, manifest_data, config, sess_id):
    batch_files = manifest_data["batches"].get(batch_id)
    if not batch_files:
        raise ValueError(f"manifestにバッチが存在しません: {batch_id}")

    file_metadata = manifest_data.get("file_metadata", {})
    for rel_p in batch_files:
        if rel_p not in file_metadata:
            raise ValueError(f"メタデータが見つかりません: {rel_p}")
        full_path = INPUT_ROOT / rel_p
        st = full_path.stat()
        meta = file_metadata[rel_p]
        if st.st_size != meta["size"] or st.st_mtime_ns != meta["mtime_ns"]:
            raise ValueError(f"ファイルのサイズまたはmtimeが変更されています: {rel_p}")

    staging_dir = TEMP_DIR / f"{batch_id}_{sess_id}.staging"
    staging_dir.mkdir(parents=True, exist_ok=False)

    staging_parquet = staging_dir / f"{batch_id}.parquet"
    staging_json = staging_dir / f"{batch_id}_quality.json"

    start_time = datetime.now(timezone.utc)
    df, stats = process_batch_v2(batch_id, batch_files, config, sess_id)
    end_time = datetime.now(timezone.utc)

    parquet_cols = ['Datetime', 'SiteNum', 'SiteName', 'Remark', 'APE', 'source_file', 'source_row_index', 'source_line_number', 'batch_id']

    if not df.empty:
        df = df[parquet_cols]
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        df['SiteNum'] = df['SiteNum'].astype('int64')
        df['SiteName'] = df['SiteName'].astype('string')
        df['Remark'] = df['Remark'].astype('int64')
        df['APE'] = df['APE'].astype('float64')
        df['source_file'] = df['source_file'].astype('string')
        df['source_row_index'] = df['source_row_index'].astype('int64')
        df['source_line_number'] = df['source_line_number'].astype('int64')
        df['batch_id'] = df['batch_id'].astype('string')
        df.to_parquet(staging_parquet, index=False)
    else:
        pd.DataFrame(columns=parquet_cols).astype({
            'Datetime': 'datetime64[ns]', 'SiteNum': 'int64', 'SiteName': 'string',
            'Remark': 'int64', 'APE': 'float64', 'source_file': 'string',
            'source_row_index': 'int64', 'source_line_number': 'int64', 'batch_id': 'string'
        }).to_parquet(staging_parquet, index=False)

    with open(staging_parquet, "rb") as f:
        f.flush()
        os.fsync(f.fileno())

    loaded_df = pd.read_parquet(staging_parquet)
    if list(loaded_df.columns) != parquet_cols:
        raise ValueError("Parquet列構造不一致")
    if len(loaded_df) != len(df):
        raise ValueError("Parquet行数不一致")
    if loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum() > 0:
        raise ValueError("Parquet内で複合由来キー重複あり")
    if not loaded_df.empty:
        if not (loaded_df['batch_id'] == batch_id).all():
            raise ValueError("batch_id不一致")
        if loaded_df['APE'].isna().sum() > 0 or np.isinf(loaded_df['APE']).sum() > 0:
            raise ValueError("NaN/inf APEが存在")
        if (loaded_df['APE'] < config['ape_valid_range_eV'][0]).sum() > 0 or (loaded_df['APE'] > config['ape_valid_range_eV'][1]).sum() > 0:
            raise ValueError("APE範囲外が存在")

    pq_sha256 = get_file_sha256_v2(staging_parquet)

    def sum_stats(key):
        return sum(s.get(key, 0) for s in stats.values())

    rem_counts_all = collections.Counter()
    for s in stats.values():
        rem_counts_all.update(s.get('remark_counts', {}))

    quality_data = {
        "quality_schema_version": "1.0",
        "pipeline_version": config["pipeline_version"],
        "config_hash": manifest_data["config_hash"],
        "session_id": sess_id,
        "batch_id": batch_id,
        "started_at": start_time.isoformat(),
        "completed_at": end_time.isoformat(),
        "input_file_count": len(batch_files),
        "input_files": batch_files,
        "file_metadata": {p: {"size": file_metadata[p]["size"], "mtime_ns": file_metadata[p]["mtime_ns"]} for p in batch_files},
        "raw_rows": sum_stats('raw_rows'),
        "remark_counts": {str(k): v for k, v in rem_counts_all.items()},
        "remark_1_count": sum_stats('remark_1_count'),
        "remark_2_count": sum_stats('remark_2_count'),
        "target_350_1050_valid_rows": sum_stats('target_350_1050_valid_rows'),
        "full_350_1700_valid_rows": sum_stats('full_350_1700_valid_rows'),
        "target_valid_but_full_invalid_rows": sum_stats('target_valid_but_full_invalid_rows'),
        "remark_valid_but_target_invalid_rows": sum_stats('remark_valid_but_target_invalid_rows'),
        "numerator_or_denominator_invalid_rows": sum_stats('numerator_or_denominator_invalid_rows'),
        "ape_below_1_2": sum_stats('ape_below_1_2'),
        "ape_in_range": sum_stats('ape_in_range'),
        "ape_above_2_2": sum_stats('ape_above_2_2'),
        "accepted_rows": len(loaded_df),
        "minus_9999_cells_target": sum_stats('minus_9999_cells_target'),
        "minus_9999_cells_full": sum_stats('minus_9999_cells_full'),
        "other_negative_cells_target": sum_stats('other_negative_cells_target'),
        "NaN_cells_target": sum_stats('NaN_cells_target'),
        "24_00_rows": sum_stats('24_00_rows'),
        "datetime_parse_failures": sum_stats('datetime_parse_failures'),
        "timestamp_duplicate_rows": sum_stats('timestamp_duplicate_rows'),
        "source_key_duplicate_rows": int(loaded_df.duplicated(subset=['source_file', 'source_line_number']).sum()),
        "date_or_site_mismatches": sum_stats('date_or_site_mismatches'),
        "processing_error_count": 0,
        "file_statistics": stats,
        "ape_statistics": loaded_df['APE'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict() if not loaded_df.empty else {},
        "parquet_schema": {col: str(dtype) for col, dtype in loaded_df.dtypes.items()},
        "parquet_sha256": pq_sha256
    }

    atomic_write_json_v2(quality_data, staging_json, sess_id)
    return staging_dir, staging_parquet, staging_json, quality_data, pq_sha256

def publish_and_commit_batch_v2(batch_id, staging_dir, staging_parquet, staging_json, quality_data, pq_sha256, config_hash, sess_id):
    batch_parquet_path = BATCH_DIR / f"{batch_id}.parquet"
    batch_json_path = BATCH_DIR / f"{batch_id}_quality.json"
    batch_commit_path = BATCH_DIR / f"{batch_id}_commit.json"

    if batch_parquet_path.exists() or batch_json_path.exists() or batch_commit_path.exists():
        raise FileExistsError("公開直前に正式ファイルが既に存在します。")

    os.replace(staging_parquet, batch_parquet_path)
    os.replace(staging_json, batch_json_path)

    final_pq_sha = get_file_sha256_v2(batch_parquet_path)
    if final_pq_sha != pq_sha256:
        raise ValueError("公開後Parquet SHA不一致")
    with open(batch_json_path, "r", encoding="utf-8") as f:
        final_q = json.load(f)
    if final_q != make_json_safe_v2(quality_data):
        raise ValueError("公開後Quality JSON内容不一致")

    commit_data = {
        "commit_schema_version": "1.0",
        "batch_id": batch_id,
        "config_hash": config_hash,
        "session_id": sess_id,
        "committed_at": datetime.now(timezone.utc).isoformat(),
        "parquet_filename": batch_parquet_path.name,
        "parquet_sha256": final_pq_sha,
        "quality_filename": batch_json_path.name,
        "quality_sha256": get_file_sha256_v2(batch_json_path),
        "accepted_rows": quality_data["accepted_rows"],
        "input_file_count": quality_data["input_file_count"]
    }

    atomic_write_json_v2(commit_data, batch_commit_path, sess_id)

    for f in staging_dir.iterdir():
        os.unlink(f)
    staging_dir.rmdir()

def audit_committed_batches_v2(manifest_data):
    pqs = {p.stem for p in BATCH_DIR.glob("*.parquet")}
    qjs = {p.name.replace("_quality.json", "") for p in BATCH_DIR.glob("*_quality.json")}
    cjs = {p.name.replace("_commit.json", "") for p in BATCH_DIR.glob("*_commit.json")}

    all_ids = pqs | qjs | cjs
    if not all_ids:
        return []

    if pqs != all_ids or qjs != all_ids or cjs != all_ids:
        raise RuntimeError(f"孤立成果物検出: pqs={pqs}, qjs={qjs}, cjs={cjs}")

    valid_batches = []
    for b_id in sorted(all_ids):
        c_path = BATCH_DIR / f"{b_id}_commit.json"
        with open(c_path, "r", encoding="utf-8") as f:
            c_data = json.load(f)

        if c_data["config_hash"] != manifest_data["config_hash"]:
            raise ValueError(f"{b_id} config_hash不一致")
        if c_data["parquet_filename"] != f"{b_id}.parquet":
            raise ValueError(f"{b_id} parquet_filename異常")

        pq_path = BATCH_DIR / c_data["parquet_filename"]
        if get_file_sha256_v2(pq_path) != c_data["parquet_sha256"]:
            raise ValueError(f"{b_id} Parquet SHA不一致")

        expected_files = manifest_data["batches"].get(b_id, [])
        if c_data["input_file_count"] != len(expected_files):
            raise ValueError(f"{b_id} input_file_count異常")

        valid_batches.append(b_id)

    return valid_batches

def rebuild_expected_resume_v2(manifest_data, valid_batches):
    expected_completed_files = 0
    expected_output_rows = 0
    expected_src_files = []

    for b_id in valid_batches:
        q_path = BATCH_DIR / f"{b_id}_quality.json"
        with open(q_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)
        expected_output_rows += q_data["accepted_rows"]
        expected_completed_files += q_data["input_file_count"]
        expected_src_files.extend(manifest_data["batches"][b_id])

    all_batches = list(manifest_data["batches"].keys())
    expected_pending = [b for b in all_batches if b not in valid_batches]

    return {
        "completed_batch_ids": valid_batches,
        "pending_batch_ids": expected_pending,
        "completed_source_files": expected_src_files,
        "completed_batches": len(valid_batches),
        "completed_files": expected_completed_files,
        "completed_output_rows": expected_output_rows
    }

def update_resume_from_commits_v2(manifest_data, sess_id):
    valid_batches = audit_committed_batches_v2(manifest_data)
    rebuilt = rebuild_expected_resume_v2(manifest_data, valid_batches)

    with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
        rs = json.load(f)

    rs["completed_batch_ids"] = rebuilt["completed_batch_ids"]
    rs["pending_batch_ids"] = rebuilt["pending_batch_ids"]
    rs["completed_source_files"] = rebuilt["completed_source_files"]
    rs["completed_batches"] = rebuilt["completed_batches"]
    rs["completed_files"] = rebuilt["completed_files"]
    rs["completed_output_rows"] = rebuilt["completed_output_rows"]
    if valid_batches:
        rs["last_completed_batch"] = valid_batches[-1]
        rs["last_completed_source_file"] = rebuilt["completed_source_files"][-1]
    rs["last_update"] = datetime.now(timezone.utc).isoformat()

    atomic_write_json_v2(rs, RESUME_STATE_PATH, sess_id)

print("✅ 保存・commit・再開状態関数群が定義されました (ダミー・passなし)")


--- 第2段階: 保存・commit・再開状態関数の完全実装 ---
✅ 保存・commit・再開状態関数群が定義されました (ダミー・passなし)


In [37]:
with open(FINAL_JSON_PATH, "r", encoding="utf-8") as f:
    q = json.load(f)

print("HSR生データ内 Remark 2:", q["raw_remark_2_rows"])
print("最終採用 Remark 2:", q["accepted_remark_2_rows"])


HSR生データ内 Remark 2: 0
最終採用 Remark 2: 0
